
# Code-Fixing Agent Harness — Project Index

## Implementation Roadmap

| Step | Component | Description |
|---:|---|---|
| 01 | Project Setup | Define project objectives and architecture |
| 02 | Dependencies | Install required Python libraries |
| 03 | Imports & Configuration | Initialize libraries, API and workspace settings |
| 04 | Exception Handling | Define custom agent exceptions |
| 05 | Data Structures | Implement ToolCall and AssistantTurn |
| 06 | Message Validation | Validate conversation and tool-call protocols |
| 07 | LLM Interface | Define a common LLM client protocol |
| 08 | Scripted LLM | Build a deterministic mock LLM for testing |
| 09 | Gemini Integration | Connect Gemini through the OpenAI-compatible API |
| 10 | Tool Results | Standardize tool execution results |
| 11 | Tool Registry | Register and execute tools |
| 12 | Argument Validation | Validate tool arguments and schemas |
| 13 | Workspace Security | Restrict file access to the project workspace |
| 14 | File Exploration | Implement list_files and read_file |
| 15 | Code Search | Implement search_code |
| 16 | Sample Project | Create a multi-file Python project |
| 17 | Tool Registration | Register exploration tools with Gemini |
| 18 | Agent Loop | Implement the basic ReAct execution loop |
| 19 | Resumable Execution | Support continuing interrupted agent runs |
| 20 | Token Tracking | Track prompt, completion and total tokens |
| 21 | Tool Output Management | Limit and manage large tool outputs |
| 22 | Working Memory | Maintain structured project and debugging information |
| 23 | Automatic Memory Updates | Update memory from tool execution results |
| 24 | Context Compaction | Preserve relevant context while removing older history |
| 25 | Integrated Agent | Combine memory, tools, token tracking and compaction |
| 26 | Session Persistence | Save and restore agent sessions |
| 27 | Introduce Regression Bug | Add a realistic bug to the sample project |
| 28 | Baseline Experiment | Test the agent without memory or compaction |
| 29 | Memory-Only Experiment | Test the agent with Working Memory |
| 30 | Memory + Compaction Experiment | Test the agent with memory and context compaction |
| 31 | Final Evaluation | Compare results and summarize findings |

## Final Experimental Comparison

Three configurations were evaluated on the same debugging task:

- **Baseline:** 20,448 total tokens — 5/5 tests passed.
- **Memory Only:** 19,096 total tokens — 5/5 tests passed.
- **Memory + Compaction:** 18,670 total tokens — 5/5 tests passed; 6 compaction events.

## Project Outcome

Implemented a Gemini-powered ReAct code-fixing agent with controlled file-system tools, automated testing, structured Working Memory, token tracking, context compaction and session persistence.

The first end-to-end debugging experiment demonstrated successful bug fixing across all three configurations.



# Building an Agent Harness from Scratch
## Step 1: Foundation Layer — Protocol, LLM Client & Reliability

### Project Overview

In this project, we build a code-fixing agent from scratch. The agent will eventually be able to inspect a Python repository, understand errors, execute tools, iteratively debug code, and propose patches.

Rather than directly coupling the agent to a particular LLM provider, we will build an **Agent Harness**: a software layer responsible for managing conversations, validating model outputs, executing tools, handling failures, and controlling the agent's execution loop.

### Step 1 Objectives

The foundation layer establishes the infrastructure required by the agent:

1. **Protocol Validation:** Define and validate the structure of conversation messages exchanged between the agent and the LLM.
2. **Normalized Data Types:** Create provider-independent representations of assistant responses and tool calls.
3. **LLM Client Interface:** Define a common interface for communicating with language models.
4. **Fake LLM:** Build a deterministic mock client for testing agent behavior without consuming API quota.
5. **Gemini Client:** Implement a real LLM client using Google's Gemini API through its OpenAI-compatible interface.
6. **Reliability:** Implement request throttling, daily request budgets, retries, exponential backoff, and error handling.
7. **Observability:** Track API requests, latency, token consumption, and retries.

### Technology Stack

- Python 3
- Google Colab
- Gemini API (free tier)
- OpenAI Python SDK
- Dataclasses and Python typing
- Pytest / assertion-based testing

### Design Philosophy

- Keep the agent independent of the underlying LLM provider.
- Prefer deterministic offline tests over repeated real API calls.
- Treat malformed model outputs as recoverable observations rather than infrastructure failures.
- Separate model errors, protocol errors, and infrastructure errors.
- Never expose API keys in source code, notebook outputs, or logs.

### Architecture

The foundation layer will follow this structure:

    Agent Harness
        |
        +-- Protocol Validator
        |
        +-- Normalized Data Types
        |
        +-- LLM Client Interface
        |       |
        |       +-- ScriptedLLM (Offline Testing)
        |       |
        |       +-- GeminiClient (Real API)
        |
        +-- Reliability Layer
        |       |
        |       +-- Rate Limiter
        |       +-- Daily Budget
        |       +-- Retry Mechanism
        |
        +-- Usage Logging & Observability

### Model Configuration

Initial model: `gemini-2.5-flash-lite`

The model name will be configurable so that alternative Gemini models can be tested without changing the agent architecture.


Concept:
- openai: Provides the Python SDK. We'll configure it to communicate with Gemini's OpenAI-compatible endpoint.
- python-dotenv: Helps manage environment variables and API secrets.
- pytest: Used for structured offline testing.

In [2]:

%pip install -q -U openai python-dotenv pytest


Before implementing the agent components, we establish a common configuration and import the Python libraries required throughout the notebook.

An agent harness consists of several independent modules. Rather than repeatedly importing libraries or hardcoding configuration values, we define them centrally.

We will use:
- dataclasses: To represent structured objects such as tool calls and assistant responses.
- typing: To define interfaces and type contracts.
- json: To serialize and deserialize tool arguments.
- time and random: For rate limiting and retry backoff.
- copy: To create independent snapshots of conversation histories.
- logging: To track execution and infrastructure events.
- collections: To implement sliding-window request tracking.
We also define the Gemini configuration centrally. This allows us to switch models without modifying the client implementation.

In [8]:

# ============================================================
# CELL 3: IMPORTS AND GLOBAL CONFIGURATION
# ============================================================

import os
import json
import time
import random
import copy
import logging

from dataclasses import dataclass, field
from typing import (
    Protocol,
    Optional,
    Any,
    Callable,
    List,
    Dict
)

from collections import deque
from datetime import datetime, timezone


# ------------------------------------------------------------
# Gemini API Configuration
# ------------------------------------------------------------

GEMINI_MODEL = "gemini-2.5-flash-lite"

GEMINI_BASE_URL = (
    "https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_API_KEY_ENV = "GEMINI_API_KEY"


# ------------------------------------------------------------
# Agent Configuration
# ------------------------------------------------------------

DEFAULT_TEMPERATURE = 0.2

MAX_OUTPUT_TOKENS = 2048

REQUEST_TIMEOUT = 60

MAX_RETRIES = 3

BASE_BACKOFF = 1.0

MAX_BACKOFF = 30.0


# ------------------------------------------------------------
# Rate Limiting Configuration
# ------------------------------------------------------------

# Conservative initial client-side settings.
# Adjust these after checking your actual AI Studio quota.

MAX_REQUESTS_PER_MINUTE = 5

MAX_REQUESTS_PER_RUN = 30


# ------------------------------------------------------------
# Logging Configuration
# ------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)

logger = logging.getLogger("AgentHarness")


# ------------------------------------------------------------
# Configuration Summary
# ------------------------------------------------------------

print("Agent Harness Configuration")
print("-" * 40)
print("Model:", GEMINI_MODEL)
print("Base URL:", GEMINI_BASE_URL)
print("Temperature:", DEFAULT_TEMPERATURE)
print("Max output tokens:", MAX_OUTPUT_TOKENS)
print("Max retries:", MAX_RETRIES)
print("RPM limit:", MAX_REQUESTS_PER_MINUTE)
print("Request budget:", MAX_REQUESTS_PER_RUN)
print("API key loaded:", bool(os.getenv(GEMINI_API_KEY_ENV)))


Agent Harness Configuration
----------------------------------------
Model: gemini-2.5-flash-lite
Base URL: https://generativelanguage.googleapis.com/v1beta/openai/
Temperature: 0.2
Max output tokens: 2048
Max retries: 3
RPM limit: 5
Request budget: 30
API key loaded: False


In [6]:

# ============================================================
# Load Gemini API Key from Google Colab Secrets
# ============================================================

from google.colab import userdata

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    print("Gemini API key loaded successfully:", bool(GEMINI_API_KEY))

except Exception as e:
    GEMINI_API_KEY = None
    print("Could not load Gemini API key:", type(e).__name__)


Gemini API key loaded successfully: True


In [7]:
# Verify without exposing the actual key

assert GEMINI_API_KEY is not None
assert len(GEMINI_API_KEY) > 0

print("API key verification successful.")

API key verification successful.


Concept: Why do we need custom exceptions?

An agent interacts with an LLM, processes its responses, and executes tools. Failures can occur at different layers.

A major design principle in agent engineering is that not all failures should be handled in the same way.

For example:
1. Protocol errors: The conversation history violates our expected message format.
2. LLM infrastructure errors: The API is unavailable, times out, or repeatedly returns rate-limit errors.
3. Bad request errors: The API rejects our request because of an invalid parameter, authentication issue, or unsupported operation.
4. Budget errors: Our local request budget has been exhausted.
5. Script exhaustion: Our fake LLM has no more predefined responses to return.
We create a custom exception hierarchy so that different components can catch and handle specific failures.
Exception hierarchy
Exception

Python base exception


AgentHarnessError

Base exception for our project


ProtocolError
Invalid conversation structure

LLMError
LLM-related failures

BudgetExceeded
Request budget exhausted

ScriptExhausted
Fake responses exhausted


LLMError subclasses
- LLMInfraError
- LLMBadRequestError


Why distinguish infrastructure errors from bad requests?

Consider two scenarios:
- HTTP 429: We exceeded a rate limit. Waiting and retrying may solve the problem.
- HTTP 400: Our request is invalid. Repeating the same request will generally not solve it.
Our retry mechanism should retry transient infrastructure failures, but not ordinary invalid requests.

In [9]:

# ============================================================
# CELL 4: EXCEPTION HIERARCHY
# ============================================================

class AgentHarnessError(Exception):
    """Base exception for all custom agent harness errors."""
    pass


class ProtocolError(AgentHarnessError):
    """Raised when a conversation violates protocol invariants."""
    pass


class LLMError(AgentHarnessError):
    """Base exception for language model client errors."""
    pass


class LLMInfraError(LLMError):
    """Raised when an LLM request fails due to infrastructure issues."""
    pass


class LLMBadRequestError(LLMError):
    """Raised when the LLM provider rejects an invalid request."""
    pass


class BudgetExceeded(AgentHarnessError):
    """Raised when the configured API request budget is exhausted."""
    pass


class ScriptExhausted(AgentHarnessError):
    """Raised when the fake LLM has no more scripted responses."""
    pass


print("Exception hierarchy initialized successfully.")


Exception hierarchy initialized successfully.


In [10]:

# ============================================================
# CELL 4.1: EXCEPTION TESTS
# ============================================================

assert issubclass(ProtocolError, AgentHarnessError)

assert issubclass(LLMError, AgentHarnessError)
assert issubclass(LLMInfraError, LLMError)
assert issubclass(LLMBadRequestError, LLMError)

assert issubclass(BudgetExceeded, AgentHarnessError)
assert issubclass(ScriptExhausted, AgentHarnessError)

# Verify exceptions can be raised and caught

try:
    raise LLMInfraError("Gemini API temporarily unavailable")

except LLMError as e:
    print("Caught LLM error:", e)

print("\nAll exception tests passed.")


Caught LLM error: Gemini API temporarily unavailable

All exception tests passed.



# 5. Normalized Data Types: ToolCall and AssistantTurn

## 5.1 Why Do We Need Normalized Data Types?

Our agent communicates with an LLM through an API. The API returns a provider-specific response containing generated text, tool calls, token usage, and other metadata.

Different LLM providers may represent these responses differently.

We do not want the rest of our agent to depend directly on Gemini's response format.

Instead, we introduce an intermediate representation called a **Normalized Response**.

The architecture looks like this:

    Gemini API Response
            |
            v
       GeminiClient
            |
            v
       AssistantTurn
            |
            v
       Agent Harness

The agent only interacts with `AssistantTurn`, regardless of which LLM provider generated the response.

This makes our architecture modular and allows us to replace Gemini with another provider without rewriting the agent's core logic.

---

## 5.2 ToolCall: Representing Tool Execution Requests

An LLM does not directly execute Python functions or interact with our system.

Instead, it generates a structured request asking the agent to execute a particular tool.

For example, suppose our code-fixing agent encounters a failing test and wants to execute `pytest`.

The model might generate the following tool call:

### Example: Raw Tool Call Response

```json
{
    "id": "call_1",
    "type": "function",
    "function": {
        "name": "execute",
        "arguments": "{\"command\": \"pytest\"}"
    }
}
```

We convert this provider response into a simpler internal Python object:

### Example: Normalized ToolCall

```python
ToolCall(
    id="call_1",
    name="execute",
    arguments='{"command": "pytest"}'
)
```

Our `ToolCall` dataclass contains three attributes:

| Attribute | Description |
|---|---|
| `id` | Identifier of the tool call |
| `name` | Name of the requested tool |
| `arguments` | Raw JSON argument string generated by the model |

### Important Design Decision: Why Keep Arguments as a String?

Notice that `arguments` is stored as a string rather than a Python dictionary.

There is an important reason for this.

LLMs can generate malformed JSON.

For example:

```python
'{"command": "pytest"'
```

This is an invalid JSON string because the closing brace is missing.

Our LLM client should preserve the original model output rather than attempting to parse or repair it.

Parsing and validating tool arguments will happen later, inside the tool-execution component.

This separation ensures that:

1. The LLM client is responsible for communicating with the model.
2. The agent loop is responsible for interpreting model responses.
3. The tool executor is responsible for validating and executing tool arguments.

We also use `@dataclass(frozen=True)` to make `ToolCall` immutable.

Once a tool call has been created, its ID, name, and arguments should not accidentally change during execution.

---

## 5.3 AssistantTurn: Representing a Complete LLM Response

A model response may contain ordinary text, tool calls, or both.

We need a single object capable of representing all these possibilities.

This is the purpose of `AssistantTurn`.

It contains the following attributes:

| Attribute | Description |
|---|---|
| `content` | Text generated by the assistant |
| `tool_calls` | List of requested tool executions |
| `finish_reason` | Reason why generation stopped |
| `usage` | Token consumption information, if available |
| `raw` | Original provider response for debugging |

### Example 1: Text-Only Response

Suppose the model responds:

```text
The bug is caused by an incorrect array index.
```

We represent it as:

```python
AssistantTurn(
    content="The bug is caused by an incorrect array index.",
    tool_calls=[],
    finish_reason="stop"
)
```

### Example 2: Tool-Calling Response

Suppose the model wants to execute a test:

```python
AssistantTurn(
    content=None,
    tool_calls=[
        ToolCall(
            id="call_1",
            name="execute",
            arguments='{"command": "pytest"}'
        )
    ],
    finish_reason="tool_calls"
)
```

Here, `content=None` indicates that the response contains no ordinary text and consists of a tool execution request.

### Why Use `Optional` Fields?

Some providers may not return every piece of metadata.

For example:
- A response may not contain token usage.
- A tool-call response may have no text content.
- A provider may not supply a finish reason.

Therefore, fields such as `content`, `usage`, and `finish_reason` must support missing values.

We use Python's `Optional` type to represent this.

---

## 5.4 The to_message() Method

Our agent maintains a conversation history as a list of dictionaries.

For example:

```python
messages = [
    {
        "role": "system",
        "content": "You are a code-fixing agent."
    },
    {
        "role": "user",
        "content": "Find the bug in this function."
    }
]
```

However, our internal response is represented by an `AssistantTurn` object.

We need a conversion method to transform this object into the standard conversation message format.

That is the purpose of `to_message()`.

### Example 1: Text-Only Conversion

Input:

```python
turn = AssistantTurn(
    content="I found the bug.",
    tool_calls=[]
)
```

Output:

```python
{
    "role": "assistant",
    "content": "I found the bug."
}
```

### Example 2: Tool-Call Conversion

Input:

```python
turn = AssistantTurn(
    content=None,
    tool_calls=[
        ToolCall(
            id="call_1",
            name="execute",
            arguments='{"command": "pytest"}'
        )
    ]
)
```

Output:

```python
{
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": "call_1",
            "type": "function",
            "function": {
                "name": "execute",
                "arguments": "{\"command\": \"pytest\"}"
            }
        }
    ]
}
```

Notice that internal debugging information such as `usage` and `raw` is not included in the conversation message.

Only the information required by the conversation protocol is serialized.

---

## 5.5 What We Are Implementing

We will create two Python dataclasses:

**1. ToolCall**
- Immutable representation of a tool execution request.
- Stores the call ID, function name, and raw JSON arguments.

**2. AssistantTurn**
- Represents one complete normalized LLM response.
- Stores generated text, tool calls, finish reason, usage, and raw provider response.
- Implements `to_message()` to convert the response into the standard conversation format.

These two classes form the foundation of the provider-independent communication layer of our agent harness.


In [11]:

# ============================================================
# CELL 5: NORMALIZED DATA TYPES
# ============================================================

@dataclass(frozen=True)
class ToolCall:
    """
    Provider-independent representation of a tool call.

    Attributes:
        id: Unique identifier within an assistant turn.
        name: Name of the requested tool.
        arguments: Raw JSON argument string generated by the LLM.
    """

    id: str
    name: str
    arguments: str


@dataclass
class AssistantTurn:
    """
    Normalized representation of one LLM response.

    Attributes:
        content: Text generated by the assistant.
        tool_calls: List of requested tool executions.
        finish_reason: Reason generation stopped.
        usage: Token usage information, if available.
        raw: Original provider response for debugging.
    """

    content: Optional[str]
    tool_calls: List[ToolCall] = field(default_factory=list)
    finish_reason: Optional[str] = None
    usage: Optional[Dict[str, Any]] = None
    raw: Optional[Any] = None

    def to_message(self) -> Dict[str, Any]:
        """
        Convert AssistantTurn into the standard conversation
        protocol message format.
        """

        message = {
            "role": "assistant",
            "content": self.content
        }

        if self.tool_calls:
            message["tool_calls"] = [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.name,
                        "arguments": tool_call.arguments
                    }
                }
                for tool_call in self.tool_calls
            ]

        return message


print("ToolCall and AssistantTurn initialized successfully.")


ToolCall and AssistantTurn initialized successfully.



# 5.1 Unit Testing: ToolCall and AssistantTurn

Before moving to the message protocol validator, we need to verify that our normalized data structures behave correctly.

We will use Python's built-in `assert` statements to test the expected behavior.

Testing is particularly important in an agent harness because these objects are passed between multiple components. An error in their representation can propagate into the agent loop, tool executor, and conversation history.

## Test 1: ToolCall Construction

Create a ToolCall instance and verify that its attributes are stored correctly.

## Test 2: ToolCall Immutability

ToolCall is declared using `@dataclass(frozen=True)`.

This means its fields cannot be modified after initialization.

Attempting to modify an attribute should raise a `FrozenInstanceError`.

## Test 3: Text-Only Assistant Response

Create an AssistantTurn containing ordinary text and no tool calls.

Verify that `to_message()` produces:

```python
{
    "role": "assistant",
    "content": "Hello"
}
```

The `tool_calls` field should be omitted when no tool calls exist.

## Test 4: Tool-Calling Assistant Response

Create an AssistantTurn containing one ToolCall.

Verify that the resulting message contains the correct tool-call structure, including its ID, function name, and raw arguments.

## Test 5: Multiple Tool Calls

An assistant can request multiple tool executions in a single response.

For example, it may request reading a file and running a test independently.

Verify that `to_message()` preserves every tool call in the original order.

## Test 6: Raw Argument Preservation

The client must preserve the exact argument string generated by the model.

Even malformed JSON must remain unchanged.

For example:

```python
raw_arguments = '{"command": "pytest"'
```

We should not parse, repair, or modify this string at this stage.

## Test 7: No Mutation of Original Object

Calling `to_message()` should not modify the original AssistantTurn or its ToolCall objects.

## Test 8: Independent Default Lists

Each AssistantTurn must receive its own independent tool_calls list.

This is why we use `field(default_factory=list)` instead of a shared mutable default.

If all these tests pass, our normalized data types are ready to be used by the next component.


In [12]:

# ============================================================
# CELL 5.1: UNIT TESTS FOR NORMALIZED DATA TYPES
# ============================================================

from dataclasses import FrozenInstanceError


# ------------------------------------------------------------
# Test 1: ToolCall Construction
# ------------------------------------------------------------

tc = ToolCall(
    id="call_1",
    name="execute",
    arguments='{"command": "pytest"}'
)

assert tc.id == "call_1"
assert tc.name == "execute"
assert tc.arguments == '{"command": "pytest"}'

print("Test 1 passed: ToolCall construction")


# ------------------------------------------------------------
# Test 2: ToolCall Immutability
# ------------------------------------------------------------

try:
    tc.name = "read_file"
    assert False, "Expected FrozenInstanceError"

except FrozenInstanceError:
    pass

print("Test 2 passed: ToolCall immutability")


# ------------------------------------------------------------
# Test 3: Text-Only Assistant Response
# ------------------------------------------------------------

turn = AssistantTurn(
    content="Hello",
    finish_reason="stop"
)

message = turn.to_message()

assert message == {
    "role": "assistant",
    "content": "Hello"
}

assert "tool_calls" not in message

print("Test 3 passed: Text-only response")


# ------------------------------------------------------------
# Test 4: Single Tool Call
# ------------------------------------------------------------

turn = AssistantTurn(
    content=None,
    tool_calls=[tc],
    finish_reason="tool_calls"
)

message = turn.to_message()

assert message["role"] == "assistant"
assert message["content"] is None
assert len(message["tool_calls"]) == 1

assert message["tool_calls"][0]["id"] == "call_1"
assert message["tool_calls"][0]["function"]["name"] == "execute"

print("Test 4 passed: Single tool call")


# ------------------------------------------------------------
# Test 5: Multiple Tool Calls
# ------------------------------------------------------------

tc1 = ToolCall(
    id="call_1",
    name="read_file",
    arguments='{"path": "main.py"}'
)

tc2 = ToolCall(
    id="call_2",
    name="execute",
    arguments='{"command": "pytest"}'
)

turn = AssistantTurn(
    content=None,
    tool_calls=[tc1, tc2]
)

message = turn.to_message()

assert len(message["tool_calls"]) == 2

assert message["tool_calls"][0]["id"] == "call_1"
assert message["tool_calls"][1]["id"] == "call_2"

print("Test 5 passed: Multiple tool calls")


# ------------------------------------------------------------
# Test 6: Raw Argument Preservation
# ------------------------------------------------------------

malformed_arguments = '{"command": "pytest"'

tc = ToolCall(
    id="call_3",
    name="execute",
    arguments=malformed_arguments
)

turn = AssistantTurn(
    content=None,
    tool_calls=[tc]
)

message = turn.to_message()

assert message["tool_calls"][0]["function"]["arguments"] == malformed_arguments

print("Test 6 passed: Raw arguments preserved")


# ------------------------------------------------------------
# Test 7: No Mutation of Original Object
# ------------------------------------------------------------

original_tc = ToolCall(
    id="call_4",
    name="execute",
    arguments='{"command": "ls"}'
)

turn = AssistantTurn(
    content=None,
    tool_calls=[original_tc]
)

original_tool_calls = list(turn.tool_calls)

message = turn.to_message()

assert turn.tool_calls == original_tool_calls
assert turn.tool_calls[0] is original_tc

print("Test 7 passed: Original object unchanged")


# ------------------------------------------------------------
# Test 8: Independent Default Lists
# ------------------------------------------------------------

turn1 = AssistantTurn(content="Response 1")
turn2 = AssistantTurn(content="Response 2")

turn1.tool_calls.append(tc1)

assert len(turn1.tool_calls) == 1
assert len(turn2.tool_calls) == 0

print("Test 8 passed: Independent default lists")


print("\nAll 8 normalized data type tests passed.")


Test 1 passed: ToolCall construction
Test 2 passed: ToolCall immutability
Test 3 passed: Text-only response
Test 4 passed: Single tool call
Test 5 passed: Multiple tool calls
Test 6 passed: Raw arguments preserved
Test 7 passed: Original object unchanged
Test 8 passed: Independent default lists

All 8 normalized data type tests passed.



# 6. Conversation Message Protocol

## 6.1 What Is a Conversation Protocol?

An LLM API does not simply receive a paragraph of text and return another paragraph.

Instead, conversations are represented as structured sequences of messages.

Each message has a role that describes who generated it and, in some cases, additional metadata.

Our agent maintains this sequence as a Python list of dictionaries called `messages`.

For example:

```python
messages = [
    {
        "role": "system",
        "content": "You are a code-fixing agent."
    },
    {
        "role": "user",
        "content": "Find the bug in this Python function."
    },
    {
        "role": "assistant",
        "content": "I will inspect the code."
    }
]
```

The conversation history is sent to the LLM whenever the agent needs another response.

The structure and ordering of these messages matter because the model needs to understand the context of the conversation.

---

## 6.2 The Four Message Roles

Our protocol uses four message roles.

### 1. System

The system message defines the agent's overall instructions and behavior.

Example:

```python
{
    "role": "system",
    "content": "You are a helpful code-fixing agent."
}
```

It establishes the agent's operating instructions.

### 2. User

The user message contains the task or information provided to the agent.

Example:

```python
{
    "role": "user",
    "content": "Fix the bug in main.py."
}
```

### 3. Assistant

The assistant message represents the LLM's response.

It may contain ordinary text:

```python
{
    "role": "assistant",
    "content": "I will inspect main.py."
}
```

Alternatively, it may request tool execution:

```python
{
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": "call_1",
            "type": "function",
            "function": {
                "name": "read_file",
                "arguments": "{\"path\": \"main.py\"}"
            }
        }
    ]
}
```

The assistant does not execute the tool itself. It only requests execution.

### 4. Tool

A tool message contains the result returned by our tool executor.

For example:

```python
{
    "role": "tool",
    "tool_call_id": "call_1",
    "content": "File contents: def add(a, b): return a - b"
}
```

Notice that the tool message contains the ID of the assistant's original tool call.

This identifier establishes the relationship between a requested operation and its result.

---

## 6.3 Understanding a Complete Tool-Calling Cycle

Consider a code-fixing agent that needs to inspect a Python file.

The interaction happens in the following order.

### Step 1: User Provides a Task

```python
{
    "role": "user",
    "content": "Inspect main.py and identify the bug."
}
```

### Step 2: Assistant Requests a Tool

The model generates a request to read the file.

```python
{
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": "call_1",
            "type": "function",
            "function": {
                "name": "read_file",
                "arguments": "{\"path\": \"main.py\"}"
            }
        }
    ]
}
```

### Step 3: Our Agent Executes the Tool

The harness receives the tool request, validates its arguments, and executes the corresponding Python function.

The tool executor might return:

```text
def add(a, b):
    return a - b
```

### Step 4: Tool Result Is Appended

The result is added to the conversation history:

```python
{
    "role": "tool",
    "tool_call_id": "call_1",
    "content": "def add(a, b): return a - b"
}
```

### Step 5: Assistant Receives the Tool Result

The updated conversation is sent back to the model.

The model can now analyze the file contents and respond with an explanation or request another tool.

### Complete Message Sequence

```text
SYSTEM
  |
USER: Inspect main.py
  |
ASSISTANT: Request read_file (call_1)
  |
TOOL: Result for call_1
  |
ASSISTANT: Analyze file contents
  |
USER: [Next user message, if any]
```

The important point is that a tool call and its result are two separate messages.

The assistant requests an operation, while the tool message records what actually happened.

---

## 6.4 Why Do We Need Protocol Validation?

LLM-generated outputs are not guaranteed to be structurally valid.

For example, the model might generate:

- A tool call without an ID.
- An assistant message with an invalid role.
- A tool result referencing a nonexistent call.
- Two assistant messages consecutively.
- An assistant message containing multiple tool calls but only one tool result.
- Malformed JSON inside a tool call's arguments.

Some of these are structural protocol violations, while others are model-output errors that should be handled later.

Our validator is responsible for checking the conversation structure before it is sent to the LLM.

It does not execute tools or parse their arguments.

---

## 6.5 Protocol Invariants

An invariant is a condition that must always hold for a valid conversation.

We will enforce the following rules.

### Rule 1: Non-empty conversation

The messages object must be a non-empty list.

Every element must be a dictionary with a recognized role:

- system
- user
- assistant
- tool

### Rule 2: Exactly one system message

There must be exactly one system message, and it must be the first message.

### Rule 3: User message must appear before the assistant

A conversation must contain a user message before its first assistant response.

### Rule 4: System and user content

Both system and user messages must have non-empty string content.

### Rule 5: Assistant message validity

An assistant message must contain non-empty text unless it contains at least one tool call.

If tool_calls is present:

- It must be a non-empty list.
- Every call must have a non-empty string ID.
- Its type must be "function".
- The function name must be a non-empty string.
- Arguments must be a string.

**Important:** We do not require arguments to be valid JSON here. Malformed JSON is a possible model output and must be preserved for later handling.

Call IDs must be unique within a single assistant message. They may be reused in different assistant turns.

### Rule 6: Tool-call/result linking

Every assistant message containing N tool calls must be followed immediately by exactly N tool messages.

Every tool message must reference one of the assistant's call IDs.

There must be no missing results, duplicate results, or extra results.

The order of tool results does not have to match the order of tool calls.

### Rule 7: Tool message validity

A tool message must contain:

- A string tool_call_id.
- String content.

A tool message can only appear immediately after an assistant message containing tool calls.

### Rule 8: Conversation sequence

Two assistant messages cannot appear consecutively.

An assistant message must be followed by its required tool results, a user message, or the end of the conversation.

### Rule 9: Pending tool calls

Sometimes the agent needs to validate its conversation history immediately after receiving a tool-call response but before executing the tools.

For this situation, we will support:

```python
validate_messages(
    messages,
    allow_trailing_pending_calls=True
)
```

When enabled, the final assistant message may contain tool calls without their results yet.

This exception applies only to the final assistant message.

---

## 6.6 What the Validator Will and Will Not Do

The validator is a structural integrity checker.

It will:

- Check message roles and ordering.
- Validate required fields.
- Check tool-call identifiers.
- Verify tool-call/result correspondence.
- Detect missing and duplicate tool results.
- Support an explicitly pending final tool-call block.

It will not:

- Parse JSON arguments.
- Execute tools.
- Determine whether a requested tool exists.
- Determine whether a command is safe.
- Decide whether the model's reasoning is correct.

These responsibilities belong to later components of our agent harness.

---

## 6.7 Next Implementation

We will now implement a strict function:

```python
validate_messages(
    messages: list[dict],
    allow_trailing_pending_calls: bool = False
) -> None
```

The function will return normally when the conversation is valid and raise our custom `ProtocolError` when an invariant is violated.

We will implement the validator in the next code cell and then develop a comprehensive offline test suite.



# 6.1 Constructing a Sample Conversation

We now translate the message protocol discussed above into an actual Python data structure.

Our example represents a code-fixing agent that receives a task, requests a file-reading operation, receives the result, and then generates a response.

We will construct the conversation history manually.

The example contains:

1. A system message defining the agent's behavior.
2. A user message describing the task.
3. An assistant message requesting a tool call.
4. A tool message containing the execution result.
5. A final assistant response.

### Understanding Tool-Call Linking

The assistant generates a tool call with ID `call_1`.

When the tool executor finishes, its result is appended as a tool message containing:

```python
"tool_call_id": "call_1"
```

This establishes a one-to-one relationship between the requested operation and its result.

The conversation history is therefore not just a collection of independent messages. It is a structured sequence with dependencies between messages.

### Objective

Construct the conversation and inspect:

- The number of messages.
- The role of each message.
- The assistant's requested tool.
- The correspondence between the tool-call ID and the tool-result ID.

This is a demonstration only. We will implement the actual structural checks in the next cell.


In [13]:

# ============================================================
# CELL 6.1: SAMPLE TOOL-CALLING CONVERSATION
# ============================================================

sample_messages = [

    # 1. System message
    {
        "role": "system",
        "content": "You are a code-fixing agent."
    },

    # 2. User message
    {
        "role": "user",
        "content": "Inspect main.py and identify the bug."
    },

    # 3. Assistant requests a tool
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "id": "call_1",
                "type": "function",
                "function": {
                    "name": "read_file",
                    "arguments": '{"path": "main.py"}'
                }
            }
        ]
    },

    # 4. Tool returns the result
    {
        "role": "tool",
        "tool_call_id": "call_1",
        "content": "def add(a, b):\n    return a - b"
    },

    # 5. Assistant responds
    {
        "role": "assistant",
        "content": "The bug is that the function subtracts instead of adding."
    }
]


# ------------------------------------------------------------
# Inspect the conversation
# ------------------------------------------------------------

print("Total messages:", len(sample_messages))
print("\nConversation roles:")

for i, message in enumerate(sample_messages):
    print(f"Message {i}: {message['role']}")


# ------------------------------------------------------------
# Inspect tool-call/result linking
# ------------------------------------------------------------

assistant_message = sample_messages[2]
tool_message = sample_messages[3]

call_id = assistant_message["tool_calls"][0]["id"]
result_id = tool_message["tool_call_id"]

print("\nTool requested:", assistant_message["tool_calls"][0]["function"]["name"])
print("Tool call ID:", call_id)
print("Tool result ID:", result_id)

assert call_id == result_id

print("\nTool-call/result linking verified.")


Total messages: 5

Conversation roles:
Message 0: system
Message 1: user
Message 2: assistant
Message 3: tool
Message 4: assistant

Tool requested: read_file
Tool call ID: call_1
Tool result ID: call_1

Tool-call/result linking verified.



# 7. Implementing the Conversation Protocol Validator

## 7.1 Why Do We Need a Validator?

Our agent maintains a conversation history and repeatedly sends it to the LLM.

As the conversation grows, new messages are appended by different components:

- The user provides instructions.
- The LLM generates assistant responses.
- The assistant requests tool execution.
- The tool executor appends execution results.

Every component must preserve the conversation protocol.

A malformed conversation can cause API errors, incorrect tool execution, or inconsistencies in the agent's state.

Therefore, before sending a conversation to Gemini, we validate its structural integrity.

We will implement:

```python
validate_messages(
    messages,
    allow_trailing_pending_calls=False
)
```

The function returns `None` when the conversation is valid and raises `ProtocolError` when an invariant is violated.

---

## 7.2 Validation Strategy

We will use a sequential validation approach.

Rather than checking every property independently without considering message order, we traverse the conversation from beginning to end.

During traversal, we maintain information about the current conversation state.

In particular, we need to know whether the previous assistant message requested tool execution.

This matters because tool results must immediately follow their corresponding assistant message.

### Internal State

We maintain:

- `seen_system`: Whether the system message has appeared.
- `seen_user`: Whether a user message has appeared.
- `seen_assistant`: Whether an assistant message has appeared.
- `pending_tool_ids`: IDs of tool calls whose results are expected.
- `pending_tool_names`: Optional bookkeeping for debugging.
- `last_role`: Role of the previously processed message.

These variables allow us to validate both individual messages and relationships between consecutive messages.

---

## 7.3 Detailed Validation Logic

### A. Validate the Conversation Container

First, check that:

1. `messages` is a list.
2. It is not empty.
3. Every element is a dictionary.
4. Every message has a recognized role.

### B. Validate the System Message

The system message must appear exactly once and at index 0.

This establishes the initial instructions for the agent.

### C. Validate User Messages

A user message must appear before the first assistant message.

Its content must be a non-empty string.

### D. Validate Assistant Messages

An assistant response can contain text, tool calls, or both.

If it contains no tool calls, its content must be a non-empty string.

If it contains tool calls:

- `tool_calls` must be a non-empty list.
- Every call must be a dictionary.
- Every call must have a valid ID.
- Its type must be `"function"`.
- Its function name must be non-empty.
- Its arguments must be a string.

Call IDs must be unique within the same assistant message.

We deliberately do not parse the arguments as JSON.

### E. Validate Tool Messages

A tool message must contain:

- A valid `tool_call_id`.
- String content.

It must also reference a call requested by the immediately preceding assistant message.

We use sets to compare requested call IDs with returned result IDs.

For example:

```python
requested_ids = {"call_1", "call_2"}

returned_ids = {"call_1", "call_2"}
```

These sets must match exactly.

This detects:

- Missing results.
- Duplicate results.
- Extra results.
- Incorrect IDs.

The order of tool results is not important.

### F. Validate Message Ordering

An assistant message containing tool calls must be followed immediately by its tool results.

After those results, the conversation can continue with another assistant response.

Two assistant messages cannot appear consecutively.

A tool message cannot appear independently.

### G. Support Pending Tool Calls

During an agent execution loop, we sometimes need to validate the conversation immediately after receiving a tool-call response.

At that moment, the tool results have not yet been appended.

We therefore support:

```python
allow_trailing_pending_calls=True
```

This allows the final assistant message to contain unresolved tool calls.

It does not allow pending calls in the middle of a conversation.

---

## 7.4 Error Reporting

Every structural violation should raise our custom `ProtocolError`.

The error message should identify the index of the offending message whenever applicable.

For example:

```text
ProtocolError: Message 3: tool_call_id does not match a pending tool call.
```

This makes debugging easier because the agent can identify exactly where the conversation became invalid.

---

## 7.5 Implementation Notes

We will use small internal validation checks to keep the implementation readable.

The validator will be strict about the protocol but will not attempt to fix malformed conversations automatically.

This is important: silently repairing conversation history can hide bugs in the agent harness.

The next code cell implements all nine protocol invariants.


In [14]:

# ============================================================
# CELL 7: CONVERSATION PROTOCOL VALIDATOR
# ============================================================

def validate_messages(
    messages: List[Dict[str, Any]],
    allow_trailing_pending_calls: bool = False
) -> None:
    """
    Validate the structure and ordering of conversation messages.

    Raises:
        ProtocolError: If any protocol invariant is violated.

    Returns:
        None if the conversation is valid.
    """

    # --------------------------------------------------------
    # Rule 1: Validate conversation container
    # --------------------------------------------------------

    if not isinstance(messages, list):
        raise ProtocolError("Conversation must be a list.")

    if not messages:
        raise ProtocolError("Conversation cannot be empty.")

    valid_roles = {"system", "user", "assistant", "tool"}

    # --------------------------------------------------------
    # Conversation state
    # --------------------------------------------------------

    system_count = 0
    seen_user = False
    seen_assistant = False

    last_role = None

    # Tracks tool calls that must receive results.
    pending_tool_ids = set()

    # Index of the assistant message that generated pending calls.
    pending_assistant_index = None

    # Number of tool results already received.
    received_tool_count = 0

    # --------------------------------------------------------
    # Traverse the conversation
    # --------------------------------------------------------

    for i, message in enumerate(messages):

        # ----------------------------------------------------
        # Validate message dictionary and role
        # ----------------------------------------------------

        if not isinstance(message, dict):
            raise ProtocolError(
                f"Message {i}: Every message must be a dictionary."
            )

        role = message.get("role")

        if not isinstance(role, str) or role not in valid_roles:
            raise ProtocolError(
                f"Message {i}: Invalid or missing role: {role!r}."
            )

        # ----------------------------------------------------
        # If previous assistant has pending tool calls,
        # the next message must be a matching tool result.
        # ----------------------------------------------------

        if pending_tool_ids and role != "tool":

            is_final_pending_assistant = (
                allow_trailing_pending_calls
                and i == len(messages) - 1
                and i == pending_assistant_index
                and received_tool_count == 0
            )

            if is_final_pending_assistant:
                return

            raise ProtocolError(
                f"Message {i}: Expected a tool result for "
                f"pending tool calls from message "
                f"{pending_assistant_index}."
            )

        # ----------------------------------------------------
        # Rule 2: System message
        # ----------------------------------------------------

        if role == "system":

            system_count += 1

            if i != 0:
                raise ProtocolError(
                    f"Message {i}: System message must be first."
                )

            if system_count > 1:
                raise ProtocolError(
                    f"Message {i}: Multiple system messages found."
                )

            content = message.get("content")

            if not isinstance(content, str) or not content.strip():
                raise ProtocolError(
                    f"Message {i}: System content must be a "
                    f"non-empty string."
                )

        # ----------------------------------------------------
        # Rule 3 and 4: User message
        # ----------------------------------------------------

        elif role == "user":

            if seen_assistant:
                # User messages after an assistant are allowed.
                pass

            content = message.get("content")

            if not isinstance(content, str) or not content.strip():
                raise ProtocolError(
                    f"Message {i}: User content must be a "
                    f"non-empty string."
                )

            seen_user = True

        # ----------------------------------------------------
        # Rule 5: Assistant message
        # ----------------------------------------------------

        elif role == "assistant":

            if not seen_user:
                raise ProtocolError(
                    f"Message {i}: A user message must appear "
                    f"before the first assistant message."
                )

            if last_role == "assistant":
                raise ProtocolError(
                    f"Message {i}: Two assistant messages "
                    f"cannot appear consecutively."
                )

            seen_assistant = True

            content = message.get("content")

            tool_calls_present = "tool_calls" in message
            tool_calls = message.get("tool_calls", [])

            # Validate tool_calls container.
            if tool_calls_present:

                if not isinstance(tool_calls, list) or not tool_calls:
                    raise ProtocolError(
                        f"Message {i}: tool_calls must be "
                        f"a non-empty list."
                    )

            else:
                tool_calls = []

            # Validate assistant content.
            if not tool_calls:

                if not isinstance(content, str) or not content.strip():
                    raise ProtocolError(
                        f"Message {i}: Assistant content must be "
                        f"non-empty when no tool calls are present."
                    )

            else:

                if content is not None and (
                    not isinstance(content, str)
                    or (not content.strip() and content != "")
                ):
                    raise ProtocolError(
                        f"Message {i}: Assistant content must be "
                        f"a string or None."
                    )

                # IDs must be unique within this assistant message.
                current_ids = set()

                for j, tool_call in enumerate(tool_calls):

                    if not isinstance(tool_call, dict):
                        raise ProtocolError(
                            f"Message {i}, tool call {j}: "
                            f"Expected a dictionary."
                        )

                    call_id = tool_call.get("id")

                    if not isinstance(call_id, str) or not call_id.strip():
                        raise ProtocolError(
                            f"Message {i}, tool call {j}: "
                            f"Invalid or missing call ID."
                        )

                    if call_id in current_ids:
                        raise ProtocolError(
                            f"Message {i}, tool call {j}: "
                            f"Duplicate call ID {call_id!r}."
                        )

                    current_ids.add(call_id)

                    if tool_call.get("type") != "function":
                        raise ProtocolError(
                            f"Message {i}, tool call {j}: "
                            f"Type must be 'function'."
                        )

                    function = tool_call.get("function")

                    if not isinstance(function, dict):
                        raise ProtocolError(
                            f"Message {i}, tool call {j}: "
                            f"Function must be a dictionary."
                        )

                    name = function.get("name")
                    arguments = function.get("arguments")

                    if not isinstance(name, str) or not name.strip():
                        raise ProtocolError(
                            f"Message {i}, tool call {j}: "
                            f"Invalid function name."
                        )

                    if not isinstance(arguments, str):
                        raise ProtocolError(
                            f"Message {i}, tool call {j}: "
                            f"Arguments must be a string."
                        )

                # Store expected tool results.
                pending_tool_ids = current_ids
                pending_assistant_index = i
                received_tool_count = 0

        # ----------------------------------------------------
        # Rule 6 and 7: Tool messages
        # ----------------------------------------------------

        elif role == "tool":

            if last_role != "assistant" and not pending_tool_ids:
                raise ProtocolError(
                    f"Message {i}: Tool message must belong "
                    f"to an assistant tool-call block."
                )

            tool_call_id = message.get("tool_call_id")
            content = message.get("content")

            if not isinstance(tool_call_id, str) or not tool_call_id.strip():
                raise ProtocolError(
                    f"Message {i}: Invalid or missing tool_call_id."
                )

            if not isinstance(content, str):
                raise ProtocolError(
                    f"Message {i}: Tool content must be a string."
                )

            if tool_call_id not in pending_tool_ids:
                raise ProtocolError(
                    f"Message {i}: Unexpected, duplicate, or "
                    f"unknown tool_call_id {tool_call_id!r}."
                )

            # Consume the matching pending call.
            pending_tool_ids.remove(tool_call_id)
            received_tool_count += 1

            # All expected results have arrived.
            if not pending_tool_ids:
                pending_assistant_index = None
                received_tool_count = 0

        last_role = role

    # --------------------------------------------------------
    # Rule 2: Exactly one system message
    # --------------------------------------------------------

    if system_count != 1:
        raise ProtocolError(
            "Message 0: Conversation must contain exactly "
            "one system message."
        )

    # --------------------------------------------------------
    # Rule 9: Handle trailing pending tool calls
    # --------------------------------------------------------

    if pending_tool_ids:

        if (
            allow_trailing_pending_calls
            and last_role == "assistant"
            and pending_assistant_index == len(messages) - 1
            and received_tool_count == 0
        ):
            return

        raise ProtocolError(
            f"Message {pending_assistant_index}: "
            f"Missing tool results for calls: "
            f"{sorted(pending_tool_ids)}."
        )


print("Conversation protocol validator initialized.")


Conversation protocol validator initialized.



# 7.1 Basic Validation Demonstration

Now that we have implemented the validator, we will test it against our sample conversation.

The sample conversation contains:

1. One system message.
2. One user message.
3. One assistant tool-call message.
4. One matching tool result.
5. One final assistant response.

Every message satisfies the protocol invariants.

We will first validate the complete conversation.

Next, we will remove the tool result and validate again.

The second validation should fail because the assistant requested a tool execution but the corresponding result is missing.

We will also demonstrate the `allow_trailing_pending_calls` option, which permits the agent to validate an assistant tool-call response before executing the requested tool.

This gives us an initial check that our validator works for both completed and intermediate conversation states.


In [15]:

# ============================================================
# CELL 7.1: BASIC VALIDATOR DEMONSTRATION
# ============================================================

# Test 1: Complete conversation

validate_messages(sample_messages)

print("Test 1 passed: Complete conversation is valid.")


# Test 2: Missing tool result

incomplete_messages = sample_messages[:3]

try:
    validate_messages(incomplete_messages)
    assert False, "Expected ProtocolError"

except ProtocolError as e:
    print("\nTest 2 passed: Missing tool result detected.")
    print("Error:", e)


# Test 3: Allow pending tool calls

validate_messages(
    incomplete_messages,
    allow_trailing_pending_calls=True
)

print("\nTest 3 passed: Trailing pending tool calls allowed.")


# Test 4: Malformed JSON arguments are preserved

malformed_messages = copy.deepcopy(incomplete_messages)

malformed_messages[2]["tool_calls"][0]["function"]["arguments"] = (
    '{"command": "pytest"'
)

validate_messages(
    malformed_messages,
    allow_trailing_pending_calls=True
)

print("\nTest 4 passed: Malformed JSON arguments accepted structurally.")

print("\nBasic validator demonstrations completed.")


Test 1 passed: Complete conversation is valid.

Test 2 passed: Missing tool result detected.
Error: Message 2: Missing tool results for calls: ['call_1'].

Test 3 passed: Trailing pending tool calls allowed.

Test 4 passed: Malformed JSON arguments accepted structurally.

Basic validator demonstrations completed.



# 7.2 Comprehensive Testing of the Conversation Protocol Validator

## 7.2.1 Why Do We Need Comprehensive Tests?

A validator that accepts one valid conversation is not necessarily correct.

It must also reject every structurally invalid conversation while accepting valid edge cases.

In agent engineering, this is particularly important because conversation histories are modified repeatedly as the agent interacts with tools and the LLM.

We will build an offline test suite using Python assertions.

No Gemini API calls are required.

## 7.2.2 Testing Strategy

We divide our tests into two categories.

### A. Valid Conversations

These test cases must be accepted by the validator.

We will test:

1. A simple system-user-assistant conversation.
2. A complete single-tool execution cycle.
3. Multiple tool calls in one assistant response.
4. Tool results arriving in a different order from the requests.
5. Reuse of a tool-call ID across different assistant turns.
6. User messages appearing after an assistant response.
7. Malformed JSON arguments preserved as strings.
8. Pending tool calls when explicitly permitted.

### B. Invalid Conversations

These must raise `ProtocolError`.

We will test:

1. Conversation is not a list.
2. Empty conversation.
3. Message is not a dictionary.
4. Invalid or missing role.
5. Missing system message.
6. System message not at index zero.
7. Multiple system messages.
8. Empty system content.
9. Empty user content.
10. Assistant appears before the first user message.
11. Empty assistant content without tool calls.
12. Empty tool_calls list.
13. Duplicate call IDs within one assistant message.
14. Invalid tool-call type.
15. Missing function dictionary.
16. Empty function name.
17. Non-string arguments.
18. Missing tool result.
19. Extra tool result.
20. Duplicate tool result.
21. Incorrect tool_call_id.
22. Tool message without an assistant tool-call block.
23. Consecutive assistant messages.
24. Non-string tool content.
25. Missing tool_call_id.
26. Unresolved tool calls in the middle of a conversation.

## 7.2.3 Expected Behavior

For every valid case:

```python
validate_messages(messages)
```

must return normally.

For every invalid case:

```python
validate_messages(messages)
```

must raise `ProtocolError`.

We use helper functions to avoid repeating exception-handling code.

This also makes the test suite easier to extend as the validator evolves.

## 7.2.4 Important Edge Case: Multiple Tool Calls

Suppose the assistant requests two operations:

```python
requested_ids = {"call_1", "call_2"}
```

The tool executor may return their results in reverse order:

```python
returned_ids = {"call_2", "call_1"}
```

This is still valid because each result references the correct call ID.

However, if only `call_1` returns, the conversation is invalid because `call_2` remains unresolved.

## 7.2.5 Important Edge Case: Reusing Call IDs

Call IDs only need to be unique within an individual assistant message.

For example, the following is permitted:

- First assistant turn: `call_1`
- Second assistant turn: `call_1`

This is why we do not maintain a global set of all historical tool-call IDs.

## 7.2.6 Acceptance Criteria

The test suite should:

- Accept all valid conversations.
- Reject all invalid conversations.
- Identify protocol violations using `ProtocolError`.
- Execute entirely offline.
- Complete in a few seconds.

Passing these tests gives us greater confidence that the protocol validator is ready for integration into the LLM client.


In [17]:

# ============================================================
# CELL 7.2: COMPREHENSIVE PROTOCOL VALIDATOR TESTS
# ============================================================

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def make_call(call_id="call_1", name="execute", arguments='{"command": "ls"}'):
    return {
        "id": call_id,
        "type": "function",
        "function": {
            "name": name,
            "arguments": arguments
        }
    }


def make_base_conversation():
    return [
        {"role": "system", "content": "You are a code-fixing agent."},
        {"role": "user", "content": "Fix the bug."}
    ]


def expect_valid(messages, name, allow_pending=False):
    validate_messages(
        messages,
        allow_trailing_pending_calls=allow_pending
    )
    print(f"[PASS] {name}")


def expect_invalid(messages, name, allow_pending=False):
    try:
        validate_messages(
            messages,
            allow_trailing_pending_calls=allow_pending
        )
        raise AssertionError(f"Expected ProtocolError: {name}")

    except ProtocolError:
        print(f"[PASS] {name}")


# ============================================================
# PART A: VALID CONVERSATIONS
# ============================================================

print("=" * 55)
print("PART A: VALID CONVERSATIONS")
print("=" * 55)


# 1. Simple conversation
m = make_base_conversation() + [
    {"role": "assistant", "content": "I will inspect the code."}
]
expect_valid(m, "Simple conversation")


# 2. Single tool-call cycle
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [make_call()]
    },
    {
        "role": "tool",
        "tool_call_id": "call_1",
        "content": "Execution successful."
    }
]
expect_valid(m, "Single tool-call cycle")


# 3. Multiple tool calls
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            make_call("call_1"),
            make_call("call_2", name="read_file")
        ]
    },
    {
        "role": "tool",
        "tool_call_id": "call_1",
        "content": "Result 1"
    },
    {
        "role": "tool",
        "tool_call_id": "call_2",
        "content": "Result 2"
    }
]
expect_valid(m, "Multiple tool calls")


# 4. Tool results in reverse order
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            make_call("call_1"),
            make_call("call_2")
        ]
    },
    {"role": "tool", "tool_call_id": "call_2", "content": "Result 2"},
    {"role": "tool", "tool_call_id": "call_1", "content": "Result 1"}
]
expect_valid(m, "Tool results in reverse order")


# 5. Reusing call IDs across different assistant turns
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [make_call("call_1")]
    },
    {"role": "tool", "tool_call_id": "call_1", "content": "First result"},
    {"role": "user", "content": "Now inspect another file."},
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [make_call("call_1", name="read_file")]
    },
    {"role": "tool", "tool_call_id": "call_1", "content": "Second result"}
]
expect_valid(m, "Call ID reuse across different turns")


# 6. User message after assistant response
m = make_base_conversation() + [
    {"role": "assistant", "content": "I found the issue."},
    {"role": "user", "content": "Please explain the fix."}
]
expect_valid(m, "User message after assistant")


# 7. Malformed JSON arguments
# m = make_base_conversation() + [
#     {
#         "role": "assistant",
#         "content": None,
#         "tool_calls": [
#             make_call(arguments='{"command": "pytest"')
#         ]
#     }
# ]
# expect_valid(m, "Malformed JSON arguments accepted structurally")
# 7. Malformed JSON arguments are structurally accepted
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            make_call(arguments='{"command": "pytest"')
        ]
    },
    {
        "role": "tool",
        "tool_call_id": "call_1",
        "content": "Tests completed."
    }
]

expect_valid(m, "Malformed JSON arguments accepted structurally")

# 8. Pending tool calls explicitly allowed
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [make_call()]
    }
]
expect_valid(m, "Trailing pending tool calls", allow_pending=True)


# ============================================================
# PART B: INVALID CONVERSATIONS
# ============================================================

print("\n" + "=" * 55)
print("PART B: INVALID CONVERSATIONS")
print("=" * 55)


# 1. Conversation is not a list
expect_invalid("not a list", "Non-list conversation")


# 2. Empty conversation
expect_invalid([], "Empty conversation")


# 3. Message is not a dictionary
expect_invalid([None], "Non-dictionary message")


# 4. Invalid role
m = make_base_conversation() + [
    {"role": "developer", "content": "Invalid"}
]
expect_invalid(m, "Invalid role")


# 5. Missing system message
m = [
    {"role": "user", "content": "Hello"}
]
expect_invalid(m, "Missing system message")


# 6. System message not first
m = [
    {"role": "user", "content": "Hello"},
    {"role": "system", "content": "Instructions"}
]
expect_invalid(m, "System message not first")


# 7. Multiple system messages
m = make_base_conversation() + [
    {"role": "system", "content": "Another system message"}
]
expect_invalid(m, "Multiple system messages")


# 8. Empty system content
m = [
    {"role": "system", "content": "  "},
    {"role": "user", "content": "Hello"}
]
expect_invalid(m, "Empty system content")


# 9. Empty user content
m = [
    {"role": "system", "content": "Instructions"},
    {"role": "user", "content": ""}
]
expect_invalid(m, "Empty user content")


# 10. Assistant before user
m = [
    {"role": "system", "content": "Instructions"},
    {"role": "assistant", "content": "Hello"}
]
expect_invalid(m, "Assistant before user")


# 11. Empty assistant content without tool calls
m = make_base_conversation() + [
    {"role": "assistant", "content": None}
]
expect_invalid(m, "Empty assistant content")


# 12. Empty tool_calls list
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": []
    }
]
expect_invalid(m, "Empty tool_calls list")


# 13. Duplicate IDs within one assistant message
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            make_call("call_1"),
            make_call("call_1")
        ]
    }
]
expect_invalid(m, "Duplicate IDs within one turn")


# 14. Invalid tool-call type
bad_call = make_call()
bad_call["type"] = "invalid"

m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [bad_call]
    }
]
expect_invalid(m, "Invalid tool-call type")


# 15. Missing function dictionary
bad_call = make_call()
bad_call["function"] = None

m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [bad_call]
    }
]
expect_invalid(m, "Missing function dictionary")


# 16. Empty function name
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [make_call(name="  ")]
    }
]
expect_invalid(m, "Empty function name")


# 17. Non-string arguments
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            make_call(arguments={"command": "ls"})
        ]
    }
]
expect_invalid(m, "Non-string arguments")


# 18. Missing tool result
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [make_call()]
    }
]
expect_invalid(m, "Missing tool result")


# 19. Extra tool result
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [make_call()]
    },
    {"role": "tool", "tool_call_id": "call_1", "content": "Result"},
    {"role": "tool", "tool_call_id": "call_extra", "content": "Extra"}
]
expect_invalid(m, "Extra tool result")


# 20. Duplicate tool result
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [make_call()]
    },
    {"role": "tool", "tool_call_id": "call_1", "content": "Result"},
    {"role": "tool", "tool_call_id": "call_1", "content": "Duplicate"}
]
expect_invalid(m, "Duplicate tool result")


# 21. Incorrect tool_call_id
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [make_call()]
    },
    {"role": "tool", "tool_call_id": "wrong_id", "content": "Result"}
]
expect_invalid(m, "Incorrect tool_call_id")


# 22. Tool message without assistant tool call
m = make_base_conversation() + [
    {"role": "tool", "tool_call_id": "call_1", "content": "Result"}
]
expect_invalid(m, "Tool message without preceding tool call")


# 23. Consecutive assistant messages
m = make_base_conversation() + [
    {"role": "assistant", "content": "First response"},
    {"role": "assistant", "content": "Second response"}
]
expect_invalid(m, "Consecutive assistant messages")


# 24. Non-string tool content
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [make_call()]
    },
    {"role": "tool", "tool_call_id": "call_1", "content": 123}
]
expect_invalid(m, "Non-string tool content")


# 25. Missing tool_call_id
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [make_call()]
    },
    {"role": "tool", "content": "Result"}
]
expect_invalid(m, "Missing tool_call_id")


# 26. Pending calls in the middle of a conversation
m = make_base_conversation() + [
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [make_call()]
    },
    {"role": "user", "content": "Continue"}
]
expect_invalid(m, "Pending calls in the middle",
               allow_pending=True)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 55)
print("ALL VALIDATOR TESTS COMPLETED SUCCESSFULLY")
print("=" * 55)

print("Valid conversation cases: 8")
print("Invalid conversation cases: 26")
print("Total test cases: 34")


PART A: VALID CONVERSATIONS
[PASS] Simple conversation
[PASS] Single tool-call cycle
[PASS] Multiple tool calls
[PASS] Tool results in reverse order
[PASS] Call ID reuse across different turns
[PASS] User message after assistant
[PASS] Malformed JSON arguments accepted structurally
[PASS] Trailing pending tool calls

PART B: INVALID CONVERSATIONS
[PASS] Non-list conversation
[PASS] Empty conversation
[PASS] Non-dictionary message
[PASS] Invalid role
[PASS] Missing system message
[PASS] System message not first
[PASS] Multiple system messages
[PASS] Empty system content
[PASS] Empty user content
[PASS] Assistant before user
[PASS] Empty assistant content
[PASS] Empty tool_calls list
[PASS] Duplicate IDs within one turn
[PASS] Invalid tool-call type
[PASS] Missing function dictionary
[PASS] Empty function name
[PASS] Non-string arguments
[PASS] Missing tool result
[PASS] Extra tool result
[PASS] Duplicate tool result
[PASS] Incorrect tool_call_id
[PASS] Tool message without preceding too


### Interface example

Suppose we have two implementations:

- `ScriptedLLM`
- `GeminiClient`

Both expose:

`chat(messages, tools=None, **params) -> AssistantTurn`

The agent can switch between them without changing its reasoning loop.

This is an example of dependency inversion: the agent depends on an abstraction rather than a specific provider implementation.


In [18]:

class LLMClient(Protocol):
    """
    Common interface for all language model clients.

    Implementations may connect to a real provider (Gemini)
    or return predefined responses (ScriptedLLM).
    """

    def chat(
        self,
        messages: List[Dict[str, Any]],
        tools: Optional[List[Dict[str, Any]]] = None,
        **params: Any
    ) -> AssistantTurn:
        """
        Generate one assistant turn.

        Args:
            messages:
                Conversation history in the standard message format.

            tools:
                Optional list of tool definitions.

            params:
                Optional generation parameters such as temperature
                and max_tokens.

        Returns:
            A normalized AssistantTurn object.
        """
        ...


print("LLMClient interface initialized successfully.")


LLMClient interface initialized successfully.


In [19]:

class ScriptedLLM:
    """
    Deterministic fake LLM for testing the agent harness.

    Responses are returned in the order they were provided.
    No external API calls are made.
    """

    def __init__(self, responses: List[AssistantTurn]):
        if not isinstance(responses, list):
            raise TypeError("responses must be a list.")

        if not responses:
            raise ValueError("At least one scripted response is required.")

        if not all(isinstance(r, AssistantTurn) for r in responses):
            raise TypeError(
                "Every scripted response must be an AssistantTurn."
            )

        self._responses = copy.deepcopy(responses)
        self._index = 0
        self.call_count = 0
        self.request_history = []

    def chat(
        self,
        messages: List[Dict[str, Any]],
        tools: Optional[List[Dict[str, Any]]] = None,
        **params: Any
    ) -> AssistantTurn:

        if self._index >= len(self._responses):
            raise ScriptExhausted(
                "No scripted LLM responses remain."
            )

        # Preserve a snapshot of the request for debugging.
        self.request_history.append({
            "messages": copy.deepcopy(messages),
            "tools": copy.deepcopy(tools),
            "params": copy.deepcopy(params)
        })

        response = copy.deepcopy(self._responses[self._index])

        self._index += 1
        self.call_count += 1

        return response


print("ScriptedLLM initialized successfully.")


ScriptedLLM initialized successfully.


In [20]:

scripted_responses = [
    AssistantTurn(
        content=None,
        tool_calls=[
            ToolCall(
                id="call_1",
                name="read_file",
                arguments='{"path": "main.py"}'
            )
        ],
        finish_reason="tool_calls"
    ),
    AssistantTurn(
        content="The file contains a bug in the addition function.",
        finish_reason="stop"
    )
]

fake_llm = ScriptedLLM(scripted_responses)

test_messages = [
    {"role": "system", "content": "You are a code-fixing agent."},
    {"role": "user", "content": "Inspect main.py."}
]

# First model call
response_1 = fake_llm.chat(test_messages)

print("First response:")
print("Content:", response_1.content)
print("Tool calls:", response_1.tool_calls)

# Second model call
response_2 = fake_llm.chat(test_messages)

print("\nSecond response:")
print("Content:", response_2.content)

print("\nTotal model calls:", fake_llm.call_count)
print("Request history entries:", len(fake_llm.request_history))

assert response_1.tool_calls[0].name == "read_file"
assert response_2.content is not None
assert fake_llm.call_count == 2
assert len(fake_llm.request_history) == 2

print("\nBasic ScriptedLLM tests passed.")


First response:
Content: None
Tool calls: [ToolCall(id='call_1', name='read_file', arguments='{"path": "main.py"}')]

Second response:
Content: The file contains a bug in the addition function.

Total model calls: 2
Request history entries: 2

Basic ScriptedLLM tests passed.


In [21]:

# Test 1: Script exhaustion

exhaustion_llm = ScriptedLLM([
    AssistantTurn(content="Only response")
])

exhaustion_llm.chat(test_messages)

try:
    exhaustion_llm.chat(test_messages)
    raise AssertionError("Expected ScriptExhausted was not raised.")
except ScriptExhausted:
    print("[PASS] Script exhaustion detected.")


# Test 2: Response isolation

original_response = AssistantTurn(content="Original response")
isolation_llm = ScriptedLLM([original_response])

# Modify the original object after initialization.
original_response.content = "Modified externally"

isolated_response = isolation_llm.chat(test_messages)

assert isolated_response.content == "Original response"

# Modify the returned response.
isolated_response.content = "Modified by caller"

# The internally stored response must remain unchanged.
assert isolation_llm._responses[0].content == "Original response"

print("[PASS] Response isolation verified.")


# Test 3: Request-history isolation

history_llm = ScriptedLLM([
    AssistantTurn(content="Done")
])

messages_for_history = [
    {"role": "user", "content": "Inspect main.py"}
]

history_llm.chat(messages_for_history)

# Modify the original input after the call.
messages_for_history[0]["content"] = "Completely different request"
messages_for_history.append(
    {"role": "user", "content": "Another message"}
)

saved_messages = history_llm.request_history[0]["messages"]

assert len(saved_messages) == 1
assert saved_messages[0]["content"] == "Inspect main.py"

print("[PASS] Request-history isolation verified.")


# Test 4: Constructor input validation

try:
    ScriptedLLM([])
    raise AssertionError("Expected ValueError for empty responses.")
except ValueError:
    print("[PASS] Empty response list rejected.")

try:
    ScriptedLLM(["not an AssistantTurn"])
    raise AssertionError("Expected TypeError for invalid response.")
except TypeError:
    print("[PASS] Invalid response type rejected.")


print("\nAll ScriptedLLM reliability tests passed.")


[PASS] Script exhaustion detected.
[PASS] Response isolation verified.
[PASS] Request-history isolation verified.
[PASS] Empty response list rejected.
[PASS] Invalid response type rejected.

All ScriptedLLM reliability tests passed.



# Cell 10: GeminiClient — Real LLM API Integration

## 1. Objective

Until now, we have used ScriptedLLM, which returns manually predefined responses. We now implement GeminiClient to communicate with Google's Gemini API.

We use the OpenAI Python SDK with Google's OpenAI-compatible API endpoint. This allows us to reuse the same SDK interface while communicating with Gemini.

Configuration:

- API endpoint: https://generativelanguage.googleapis.com/v1beta/openai/
- Initial model: gemini-2.5-flash-lite
- Authentication: GEMINI_API_KEY
- SDK: openai

## 2. Architecture

The interaction follows this sequence:

1. Receive conversation messages and optional tool definitions.
2. Validate the conversation structure.
3. Send a chat completion request to Gemini.
4. Extract the assistant response.
5. Convert any returned tool calls into our ToolCall dataclass.
6. Convert the result into an AssistantTurn object.

The output must be provider-independent.

For example, Gemini might return a response containing:

    {
        "role": "assistant",
        "content": null,
        "tool_calls": [
            {
                "id": "call_123",
                "type": "function",
                "function": {
                    "name": "read_file",
                    "arguments": "{\"path\": \"main.py\"}"
                }
            }
        ]
    }

We normalize it into:

    AssistantTurn(
        content=None,
        tool_calls=[
            ToolCall(
                id="call_123",
                name="read_file",
                arguments='{"path": "main.py"}'
            )
        ]
    )

## 3. Important design decisions

### A. Provider independence

The rest of our harness interacts with GeminiClient through the LLMClient interface.

The agent does not need to understand Gemini-specific response objects.

### B. Input validation

We use validate_messages() before sending a request. For normal requests, unresolved tool calls are not permitted.

### C. Error handling

We distinguish two broad categories:

- LLMBadRequestError: The provider rejects the request because of an invalid request, authentication, permissions, or another non-retryable client error.
- LLMInfraError: Temporary failures such as rate limiting, server errors, connection failures, or timeouts.

Temporary errors will be handled by the retry layer we implement later. For now, this client makes one request and translates errors into our custom exception hierarchy.

### D. API key security

The API key is read from the GEMINI_API_KEY variable loaded from Colab Secrets. We never print or log its value.

### E. Usage tracking

When Gemini provides token usage information, we preserve it in AssistantTurn. This will later support request and token monitoring.

## 4. Example usage

    gemini = GeminiClient(api_key=GEMINI_API_KEY)

    response = gemini.chat([
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain what a Python list is."}
    ])

    print(response.content)

This will be our first actual API request.


In [22]:

from openai import OpenAI, APIStatusError, APIConnectionError, APITimeoutError


class GeminiClient:
    """
    Gemini API client using Google's OpenAI-compatible endpoint.

    Returns provider-independent AssistantTurn objects.
    """

    def __init__(
        self,
        api_key: Optional[str] = None,
        model: str = GEMINI_MODEL,
        base_url: str = GEMINI_BASE_URL,
        timeout: int = REQUEST_TIMEOUT
    ):
        self.api_key = api_key or GEMINI_API_KEY

        if not self.api_key:
            raise ValueError(
                "Gemini API key not found. Load GEMINI_API_KEY from Colab Secrets."
            )

        self.model = model
        self.client = OpenAI(
            api_key=self.api_key,
            base_url=base_url,
            timeout=timeout
        )

    def chat(
        self,
        messages: List[Dict[str, Any]],
        tools: Optional[List[Dict[str, Any]]] = None,
        **params: Any
    ) -> AssistantTurn:

        # Validate the conversation before making an API request.
        validate_messages(messages)

        # Preserve caller-owned input.
        request_messages = copy.deepcopy(messages)

        request_params = {
            "model": self.model,
            "messages": request_messages,
            "temperature": params.pop(
                "temperature", DEFAULT_TEMPERATURE
            ),
            "max_tokens": params.pop(
                "max_tokens", MAX_OUTPUT_TOKENS
            ),
            **params
        }

        if tools is not None:
            request_params["tools"] = copy.deepcopy(tools)

        try:
            completion = self.client.chat.completions.create(
                **request_params
            )

        except (APIConnectionError, APITimeoutError) as e:
            raise LLMInfraError(
                f"Gemini connection or timeout error: {type(e).__name__}"
            ) from e

        except APIStatusError as e:
            status_code = e.status_code

            if status_code == 429 or status_code >= 500:
                raise LLMInfraError(
                    f"Gemini temporary API error: HTTP {status_code}"
                ) from e

            raise LLMBadRequestError(
                f"Gemini rejected the request: HTTP {status_code}"
            ) from e

        except Exception as e:
            raise LLMInfraError(
                f"Unexpected Gemini client error: {type(e).__name__}"
            ) from e

        # Extract the first assistant choice.
        if not completion.choices:
            raise LLMInfraError(
                "Gemini returned no completion choices."
            )

        choice = completion.choices[0]
        message = choice.message

        # Normalize tool calls.
        normalized_tool_calls = []

        if message.tool_calls:
            for tool_call in message.tool_calls:
                normalized_tool_calls.append(
                    ToolCall(
                        id=tool_call.id,
                        name=tool_call.function.name,
                        arguments=tool_call.function.arguments
                    )
                )

        # Extract token usage when available.
        usage = None

        if completion.usage is not None:
            usage = {
                "prompt_tokens": completion.usage.prompt_tokens,
                "completion_tokens": completion.usage.completion_tokens,
                "total_tokens": completion.usage.total_tokens
            }

        return AssistantTurn(
            content=message.content,
            tool_calls=normalized_tool_calls,
            finish_reason=choice.finish_reason,
            usage=usage,
            raw=completion
        )


print("GeminiClient initialized successfully.")


GeminiClient initialized successfully.



## 1. Update the model

We change the model to `gemini-3.5-flash-lite`.

The model is listed in Google's current Gemini API model and pricing information. Free-tier availability is subject to project-specific limits and quotas.

## 2. Initialize the real client

We instantiate GeminiClient with the updated model and the API key loaded from Colab Secrets.

This step initializes the SDK client. It does not send a request.

## 3. Send the first real request

We send a small text-only prompt:

"Explain the difference between a Python list and a tuple in three sentences."

We then inspect:

- Response text
- Finish reason
- Token usage, if provided



In [30]:

GEMINI_MODEL = "gemini-3.5-flash-lite"

gemini = GeminiClient(
    api_key=GEMINI_API_KEY,
    model=GEMINI_MODEL
)

print("Gemini client ready.")
print("Model:", gemini.model)


Gemini client ready.
Model: gemini-3.5-flash-lite


In [31]:

live_test_messages = [
    {
        "role": "system",
        "content": "You are a concise and technically accurate assistant."
    },
    {
        "role": "user",
        "content": (
            "Explain the difference between a Python list "
            "and a tuple in exactly three sentences."
        )
    }
]

try:
    live_response = gemini.chat(live_test_messages)

    print("Gemini API request successful!")
    print("=" * 50)

    print("\nModel:", gemini.model)
    print("\nResponse:")
    print(live_response.content)

    print("\nFinish reason:", live_response.finish_reason)
    print("Token usage:", live_response.usage)

    assert isinstance(live_response, AssistantTurn)
    assert isinstance(live_response.content, str)
    assert len(live_response.content.strip()) > 0

    print("\nLive Gemini integration test passed.")

except (LLMInfraError, LLMBadRequestError) as e:
    print("Gemini API request failed.")
    print("Error type:", type(e).__name__)
    print("Details:", e)


Gemini API request successful!

Model: gemini-3.5-flash-lite

Response:
Python lists are mutable, meaning their elements can be modified, added, or removed after creation, whereas tuples are immutable and fixed in size. Lists are defined using square brackets `[]`, while tuples are defined using parentheses `()`. Because of their immutability, tuples are generally more memory-efficient and faster than lists, and they can be used as dictionary keys when containing hashable elements.

Finish reason: stop
Token usage: {'prompt_tokens': 26, 'completion_tokens': 79, 'total_tokens': 105}

Live Gemini integration test passed.



# Cell 11: Testing Gemini Function Calling

## 1. Objective

Our code-fixing agent needs to interact with its environment.

For example, when a user asks:

"Inspect main.py and identify the bug."

Gemini should be able to request a tool such as:

    read_file(path="main.py")

The model does not execute this function itself. It only generates a structured tool-call request. Our agent harness will execute the requested function and return its result to Gemini.

The responsibilities are:

- **LLM:** Decides which tool to request and generates arguments.
- **Agent harness:** Validates the request, executes the requested function, and returns its result.

## 2. How do we describe a tool to Gemini?

We provide a tool definition containing:

1. Function name.
2. Function description.
3. Parameter names and types.
4. Required parameters.

For example, consider a `read_file` tool.

Its Python representation is:

    read_file_tool = {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the contents of a file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Path to the file."
                    }
                },
                "required": ["path"]
            }
        }
    }

Here:

- `name` identifies the function.
- `description` tells Gemini when to use it.
- `parameters` defines the expected arguments.
- `properties` describes each argument.
- `required` specifies mandatory arguments.

This schema is a description of the function, not the implementation itself.

## 3. What does Gemini return?

Suppose we send the following user request:

"Read main.py and identify the bug."

Gemini may return a response containing:

    AssistantTurn(
        content=None,
        tool_calls=[
            ToolCall(
                id="call_123",
                name="read_file",
                arguments='{"path": "main.py"}'
            )
        ]
    )

Let's understand each field.

**id**

A unique identifier for this particular tool call. We need it to associate the tool's result with the correct request.

**name**

The function Gemini wants us to execute. Here, it is `read_file`.

**arguments**

A JSON-encoded string containing the function arguments.

We can parse it in Python:

    arguments = json.loads(call.arguments)

    # Result:
    {"path": "main.py"}

The resulting dictionary can later be passed to the actual Python function.

## 4. What is tool_choice="required"?

Normally, the model can choose between returning text and requesting a tool.

For this experiment, we explicitly set:

    tool_choice="required"

This forces the model to request at least one available tool, rather than returning only ordinary text.

This is useful for testing whether our function-calling integration works.

## 5. Expected interaction

The complete flow is:

    User request
        |
        v
    Gemini receives tool definitions
        |
        v
    Gemini selects read_file
        |
        v
    Gemini generates arguments
        |
        v
    Gemini returns structured tool call
        |
        v
    GeminiClient converts it into ToolCall
        |
        v
    AssistantTurn is returned

Important: At this stage, we are NOT executing the requested function. We are only testing whether Gemini can generate a valid tool-call request and whether our client can normalize it.

## 6. What have we verified?

Our live test successfully returned:

    Tool name: read_file
    Arguments: {"path": "main.py"}

We also parsed the argument string using Python's json module and verified that it produces a dictionary.

Therefore, we have confirmed that:

- Gemini supports our function-calling request.
- The OpenAI-compatible SDK receives the tool-call response.
- GeminiClient normalizes the response into our ToolCall dataclass.
- The arguments can be parsed for subsequent execution.


In [32]:

read_file_tool = {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": (
            "Read the contents of a file and return its text."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "description": "Path to the file to read."
                }
            },
            "required": ["path"],
            "additionalProperties": False
        }
    }
}

tool_definitions = [read_file_tool]

print("Tool schema defined successfully.")
print(json.dumps(tool_definitions, indent=2))


Tool schema defined successfully.
[
  {
    "type": "function",
    "function": {
      "name": "read_file",
      "description": "Read the contents of a file and return its text.",
      "parameters": {
        "type": "object",
        "properties": {
          "path": {
            "type": "string",
            "description": "Path to the file to read."
          }
        },
        "required": [
          "path"
        ],
        "additionalProperties": false
      }
    }
  }
]


In [33]:

tool_test_messages = [
    {
        "role": "system",
        "content": (
            "You are a code-fixing assistant. "
            "Use available tools when necessary."
        )
    },
    {
        "role": "user",
        "content": (
            "Inspect main.py and identify the bug. "
            "Read the file before answering."
        )
    }
]

try:
    tool_response = gemini.chat(
        messages=tool_test_messages,
        tools=tool_definitions,
        tool_choice="required"
    )

    print("Gemini tool-calling request successful!")
    print("=" * 55)

    print("Content:", tool_response.content)
    print("Finish reason:", tool_response.finish_reason)

    print("\nReturned tool calls:")

    for call in tool_response.tool_calls:
        print("ID:", call.id)
        print("Name:", call.name)
        print("Arguments:", call.arguments)

    assert len(tool_response.tool_calls) > 0
    assert any(
        call.name == "read_file"
        for call in tool_response.tool_calls
    )

    print("\nGemini function-calling test passed.")

except (LLMInfraError, LLMBadRequestError) as e:
    print("Function-calling test failed.")
    print("Error type:", type(e).__name__)
    print("Details:", e)


Gemini tool-calling request successful!
Content: None
Finish reason: tool_calls

Returned tool calls:
ID: call_5109623225046025170
Name: read_file
Arguments: {"path":"main.py"}

Gemini function-calling test passed.


In [34]:

import json

assert isinstance(tool_response, AssistantTurn)

for call in tool_response.tool_calls:
    assert isinstance(call, ToolCall)
    assert isinstance(call.id, str)
    assert isinstance(call.name, str)
    assert isinstance(call.arguments, str)

    # Parse arguments for inspection only.
    parsed_arguments = json.loads(call.arguments)

    print("Validated tool:", call.name)
    print("Parsed arguments:", parsed_arguments)

    assert isinstance(parsed_arguments, dict)

print("\nNormalized tool-call structure verified.")


Validated tool: read_file
Parsed arguments: {'path': 'main.py'}

Normalized tool-call structure verified.



# Cell 12: Tool Registry and Execution Engine

## 1. Objective

So far, Gemini can generate structured tool calls, but our harness cannot execute them yet.

We now build the component responsible for:

1. Registering available Python functions.
2. Looking up a function by its name.
3. Parsing the JSON arguments generated by Gemini.
4. Validating the arguments.
5. Executing the corresponding Python function.
6. Returning the execution result in a standardized format.

This component will be called the ToolRegistry.

## 2. Why do we need a ToolRegistry?

Suppose Gemini generates this request:

    ToolCall(
        id="call_123",
        name="read_file",
        arguments='{"path": "main.py"}'
    )

The harness needs to find the Python function named read_file and execute it with the supplied arguments.

We should not use arbitrary dynamic execution such as eval().

Instead, we maintain an explicit registry of functions that the agent is permitted to call.

For example:

    registry.register(
        name="read_file",
        function=read_file
    )

The registry maintains a mapping:

    {
        "read_file": read_file,
        "execute_python": execute_python
    }

Only registered functions can be invoked.

## 3. Architecture

The execution pipeline is:

    ToolCall
       |
       v
    Extract function name
       |
       v
    Look up function in registry
       |
       v
    Parse arguments using json.loads()
       |
       v
    Validate argument structure
       |
       v
    Execute registered function
       |
       v
    Capture result or execution error
       |
       v
    Return standardized ToolResult

## 4. ToolResult data structure

We need a normalized representation of tool execution results.

We define:

    ToolResult(
        tool_call_id="call_123",
        name="read_file",
        content="File contents...",
        success=True
    )

The fields mean:

- tool_call_id: Links the result to the original Gemini tool call.
- name: Name of the executed function.
- content: String representation of the result or error.
- success: Indicates whether execution succeeded.

Why do we need tool_call_id?

Suppose Gemini requests three tools in one assistant turn. The harness must return each result with the correct identifier so that Gemini can associate each result with its corresponding request.

## 5. Important design decisions

### A. Explicit registration

Only functions registered in ToolRegistry can execute.

### B. Argument parsing

Gemini returns arguments as JSON strings. We convert them into Python dictionaries using json.loads().

### C. Error handling

Tool execution failures should become observations rather than immediately terminating the agent.

For example:

    ToolResult(
        tool_call_id="call_123",
        name="read_file",
        content="FileNotFoundError: main.py does not exist",
        success=False
    )

The agent can send this error back to Gemini, allowing it to reason about the failure and choose another action.

### D. Safety boundary

The registry restricts execution to explicitly registered functions.

This is not a complete security sandbox. Individual tools still require their own input validation, permission checks, and execution restrictions.

## 6. What we implement in this cell

We will implement:

- ToolResult dataclass.
- ToolRegistry class.
- register() method.
- get() method.
- execute() method.

For now, we will test the registry with a harmless demonstration function rather than executing arbitrary shell commands or Python code.


In [35]:

@dataclass
class ToolResult:
    """
    Normalized result of one tool execution.
    """
    tool_call_id: str
    name: str
    content: str
    success: bool


class ToolRegistry:
    """
    Registry of explicitly permitted Python functions.
    """

    def __init__(self):
        self._tools = {}

    def register(self, name: str, function: Callable):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Tool name must be a non-empty string.")

        if not callable(function):
            raise TypeError("Registered tool must be callable.")

        if name in self._tools:
            raise ValueError(f"Tool already registered: {name}")

        self._tools[name] = function

    def get(self, name: str) -> Callable:
        if name not in self._tools:
            raise KeyError(f"Unknown tool: {name}")

        return self._tools[name]

    def execute(self, tool_call: ToolCall) -> ToolResult:
        """
        Execute one registered tool call and normalize its result.
        """

        try:
            if not isinstance(tool_call, ToolCall):
                raise TypeError("Expected a ToolCall object.")

            function = self.get(tool_call.name)

            arguments = json.loads(tool_call.arguments)

            if not isinstance(arguments, dict):
                raise ValueError(
                    "Tool arguments must decode into a JSON object."
                )

            result = function(**arguments)

            return ToolResult(
                tool_call_id=tool_call.id,
                name=tool_call.name,
                content=str(result),
                success=True
            )

        except Exception as e:
            return ToolResult(
                tool_call_id=getattr(tool_call, "id", "unknown"),
                name=getattr(tool_call, "name", "unknown"),
                content=f"{type(e).__name__}: {e}",
                success=False
            )


print("ToolResult and ToolRegistry initialized successfully.")


ToolResult and ToolRegistry initialized successfully.


In [36]:

def calculate_sum(a: int, b: int) -> int:
    """Return the sum of two integers."""
    return a + b


registry = ToolRegistry()

registry.register(
    name="calculate_sum",
    function=calculate_sum
)

sum_call = ToolCall(
    id="call_sum_1",
    name="calculate_sum",
    arguments='{"a": 10, "b": 25}'
)

sum_result = registry.execute(sum_call)

print("Tool execution result:")
print(sum_result)

assert sum_result.success
assert sum_result.tool_call_id == "call_sum_1"
assert sum_result.name == "calculate_sum"
assert sum_result.content == "35"

print("\nBasic tool execution test passed.")


Tool execution result:
ToolResult(tool_call_id='call_sum_1', name='calculate_sum', content='35', success=True)

Basic tool execution test passed.


In [37]:

failure_cases = [
    ToolCall(
        id="call_unknown",
        name="unknown_tool",
        arguments="{}"
    ),
    ToolCall(
        id="call_bad_json",
        name="calculate_sum",
        arguments='{"a": 10, "b": 25'
    ),
    ToolCall(
        id="call_missing_arg",
        name="calculate_sum",
        arguments='{"a": 10}'
    ),
    ToolCall(
        id="call_wrong_type",
        name="calculate_sum",
        arguments='[10, 25]'
    )
]

for call in failure_cases:
    result = registry.execute(call)

    print(f"\nTool: {result.name}")
    print(f"Success: {result.success}")
    print(f"Observation: {result.content}")

    assert not result.success

print("\nAll tool execution failure tests passed.")



Tool: unknown_tool
Success: False
Observation: KeyError: 'Unknown tool: unknown_tool'

Tool: calculate_sum
Success: False
Observation: JSONDecodeError: Expecting ',' delimiter: line 1 column 18 (char 17)

Tool: calculate_sum
Success: False
Observation: TypeError: calculate_sum() missing 1 required positional argument: 'b'

Tool: calculate_sum
Success: False
Observation: ValueError: Tool arguments must decode into a JSON object.

All tool execution failure tests passed.



# Cell 12.4: Tool Results as Conversation Messages

## 1. Objective

Our ToolRegistry can execute functions and return ToolResult objects.

However, Gemini does not understand our custom ToolResult dataclass directly.

We need to convert each result into the standard conversation message format.

Recall our tool-call interaction:

1. Gemini generates an assistant message containing a tool call.
2. Our harness executes the requested function.
3. The function produces a ToolResult.
4. The harness converts ToolResult into a message with role="tool".
5. This message is appended to the conversation.
6. The updated conversation is sent back to Gemini.

## 2. Required message structure

Suppose Gemini requests:

    ToolCall(
        id="call_sum_1",
        name="calculate_sum",
        arguments='{"a": 10, "b": 25}'
    )

Our registry returns:

    ToolResult(
        tool_call_id="call_sum_1",
        name="calculate_sum",
        content="35",
        success=True
    )

We convert it into:

    {
        "role": "tool",
        "tool_call_id": "call_sum_1",
        "content": "35"
    }

Notice that the tool result contains the original tool_call_id.

This identifier is essential because the conversation protocol uses it to associate each execution result with the corresponding assistant tool call.

## 3. Handling failures

The same mechanism works when execution fails.

For example:

    ToolResult(
        tool_call_id="call_bad_1",
        name="calculate_sum",
        content="TypeError: Missing argument b",
        success=False
    )

The resulting tool message contains the error as its content.

Gemini can then observe the error and decide what to do next.

## 4. Important distinction

The conversation message does not need a separate success field.

The success field is useful internally for our harness, logging, and debugging.

The tool message sent to the LLM contains:
- role
- tool_call_id
- content

## 5. What we implement

We will add a to_message() method to ToolResult.

We will then verify:

1. Successful result conversion.
2. Failed result conversion.
3. Correct tool_call_id preservation.
4. Integration with validate_messages().


In [38]:

@dataclass
class ToolResult:
    """
    Normalized result of one tool execution.
    """

    tool_call_id: str
    name: str
    content: str
    success: bool

    def to_message(self) -> Dict[str, Any]:
        """
        Convert the execution result into a conversation protocol message.
        """

        return {
            "role": "tool",
            "tool_call_id": self.tool_call_id,
            "content": self.content
        }


print("ToolResult updated successfully.")


ToolResult updated successfully.


In [39]:

# Execute a successful tool call.
success_result = registry.execute(
    ToolCall(
        id="call_success",
        name="calculate_sum",
        arguments='{"a": 10, "b": 25}'
    )
)

# Execute a failing tool call.
failure_result = registry.execute(
    ToolCall(
        id="call_failure",
        name="calculate_sum",
        arguments='{"a": 10}'
    )
)

# Convert both results into protocol messages.
success_message = success_result.to_message()
failure_message = failure_result.to_message()

print("Successful tool message:")
print(success_message)

print("\nFailed tool message:")
print(failure_message)

# Verify successful conversion.
assert success_message == {
    "role": "tool",
    "tool_call_id": "call_success",
    "content": "35"
}

# Verify failed conversion.
assert failure_message["role"] == "tool"
assert failure_message["tool_call_id"] == "call_failure"
assert "TypeError" in failure_message["content"]

# Build a complete conversation containing a tool call and its result.
conversation = [
    {
        "role": "system",
        "content": "You are a code-fixing assistant."
    },
    {
        "role": "user",
        "content": "Calculate 10 + 25."
    },
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "id": "call_success",
                "type": "function",
                "function": {
                    "name": "calculate_sum",
                    "arguments": '{"a": 10, "b": 25}'
                }
            }
        ]
    },
    success_message,
    {
        "role": "assistant",
        "content": "The answer is 35."
    }
]

validate_messages(conversation)

print("\nConversation protocol validation passed.")
print("ToolResult integration tests passed.")


Successful tool message:
{'role': 'tool', 'tool_call_id': 'call_success', 'content': '35'}

Failed tool message:
{'role': 'tool', 'tool_call_id': 'call_failure', 'content': "TypeError: calculate_sum() missing 1 required positional argument: 'b'"}

Conversation protocol validation passed.
ToolResult integration tests passed.



# Cell 13: Batch Tool Execution

## 1. Objective

Until now, our ToolRegistry executes one ToolCall at a time.

However, an LLM can request multiple tools in a single assistant response.

For example, Gemini might request:

    ToolCall(
        id="call_1",
        name="calculate_sum",
        arguments='{"a": 10, "b": 20}'
    )

    ToolCall(
        id="call_2",
        name="calculate_sum",
        arguments='{"a": 30, "b": 40}'
    )

Both calls belong to the same assistant turn.

Our harness should execute both calls and return two corresponding ToolResult objects.

## 2. Why batch execution?

Consider an agent that needs to inspect three files.

Instead of making three separate LLM requests:

    Gemini -> read_file("main.py")
    Gemini -> read_file("utils.py")
    Gemini -> read_file("test.py")

It may request all three tools in one response:

    Gemini -> [
        read_file("main.py"),
        read_file("utils.py"),
        read_file("test.py")
    ]

Our harness can execute all three requests and then send all three results back to Gemini in one conversation update.

This avoids unnecessary model round trips.

## 3. Execution strategy

For now, we will execute tools sequentially.

The execution flow is:

    List[ToolCall]
          |
          v
    Validate tool calls
          |
          v
    Execute call 1
          |
          v
    Execute call 2
          |
          v
    Execute call 3
          |
          v
    List[ToolResult]

Why sequential execution?

- Predictable execution order.
- Easier debugging.
- No concurrency-related race conditions.
- Safer when tools modify files or shared state.

We can introduce parallel execution later for independent read-only tools if needed.

## 4. Important design decisions

### A. Preserve execution order

The first ToolCall must correspond to the first ToolResult, and so on.

### B. Preserve identifiers

Each ToolResult must retain the tool_call_id of its original request.

### C. Isolate failures

Suppose three tools are requested:

- Tool 1 succeeds.
- Tool 2 fails.
- Tool 3 succeeds.

The failure of Tool 2 must not prevent Tool 3 from executing.

Our existing execute() method already converts individual failures into ToolResult objects.

### D. Return standardized results

The batch method returns:

    List[ToolResult]

These can subsequently be converted into conversation messages using:

    result.to_message()

## 5. What we implement

We extend ToolRegistry with:

    execute_many(tool_calls)

It will:
- Accept a list of ToolCall objects.
- Reject invalid input and duplicate call IDs.
- Execute each call sequentially.
- Return results in the original order.


Update ToolRegistry
We'll redefine the class with all existing methods plus execute_many(). Your existing registry instance will still work, but we'll create a fresh instance in the test cell.

In [41]:

class ToolRegistry:
    """
    Registry of explicitly permitted Python functions.
    Supports individual and batch tool execution.
    """

    def __init__(self):
        self._tools = {}

    def register(self, name: str, function: Callable):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Tool name must be a non-empty string.")

        if not callable(function):
            raise TypeError("Registered tool must be callable.")

        if name in self._tools:
            raise ValueError(f"Tool already registered: {name}")

        self._tools[name] = function

    def get(self, name: str) -> Callable:
        if name not in self._tools:
            raise KeyError(f"Unknown tool: {name}")

        return self._tools[name]

    def execute(self, tool_call: ToolCall) -> ToolResult:
        """Execute one tool call and normalize its result."""

        try:
            if not isinstance(tool_call, ToolCall):
                raise TypeError("Expected a ToolCall object.")

            function = self.get(tool_call.name)

            arguments = json.loads(tool_call.arguments)

            if not isinstance(arguments, dict):
                raise ValueError(
                    "Tool arguments must decode into a JSON object."
                )

            result = function(**arguments)

            return ToolResult(
                tool_call_id=tool_call.id,
                name=tool_call.name,
                content=str(result),
                success=True
            )

        except Exception as e:
            return ToolResult(
                tool_call_id=getattr(tool_call, "id", "unknown"),
                name=getattr(tool_call, "name", "unknown"),
                content=f"{type(e).__name__}: {e}",
                success=False
            )

    def execute_many(
        self,
        tool_calls: List[ToolCall]
    ) -> List[ToolResult]:
        """
        Execute multiple tool calls sequentially.

        Individual execution failures are returned as ToolResult
        objects and do not stop the remaining calls.
        """

        if not isinstance(tool_calls, list):
            raise TypeError("tool_calls must be a list.")

        if not all(isinstance(call, ToolCall) for call in tool_calls):
            raise TypeError("Every item must be a ToolCall.")

        call_ids = [call.id for call in tool_calls]

        if len(call_ids) != len(set(call_ids)):
            raise ValueError("Duplicate tool_call IDs in batch.")

        results = []

        for call in tool_calls:
            result = self.execute(call)
            results.append(result)

        return results


print("ToolRegistry updated with batch execution.")


ToolRegistry updated with batch execution.


In [42]:

# Create a fresh registry.
batch_registry = ToolRegistry()

batch_registry.register(
    name="calculate_sum",
    function=calculate_sum
)

# Gemini-like batch of three tool calls.
batch_calls = [
    ToolCall(
        id="call_1",
        name="calculate_sum",
        arguments='{"a": 10, "b": 20}'
    ),
    ToolCall(
        id="call_2",
        name="calculate_sum",
        arguments='{"a": 30, "b": 40}'
    ),
    ToolCall(
        id="call_3",
        name="calculate_sum",
        arguments='{"a": 50, "b": 60}'
    )
]

batch_results = batch_registry.execute_many(batch_calls)

for result in batch_results:
    print(
        f"ID: {result.tool_call_id} | "
        f"Success: {result.success} | "
        f"Result: {result.content}"
    )

assert len(batch_results) == 3
assert [r.tool_call_id for r in batch_results] == [
    "call_1", "call_2", "call_3"
]
assert [r.content for r in batch_results] == ["30", "70", "110"]
assert all(r.success for r in batch_results)

print("\nBatch execution test passed.")


ID: call_1 | Success: True | Result: 30
ID: call_2 | Success: True | Result: 70
ID: call_3 | Success: True | Result: 110

Batch execution test passed.


In [43]:

mixed_calls = [
    ToolCall(
        id="call_4",
        name="calculate_sum",
        arguments='{"a": 5, "b": 10}'
    ),
    ToolCall(
        id="call_5",
        name="calculate_sum",
        arguments='{"a": 5}'
    ),
    ToolCall(
        id="call_6",
        name="calculate_sum",
        arguments='{"a": 20, "b": 30}'
    )
]

mixed_results = batch_registry.execute_many(mixed_calls)

for result in mixed_results:
    print(
        f"{result.tool_call_id}: "
        f"{'SUCCESS' if result.success else 'FAILED'} "
        f"| {result.content}"
    )

assert len(mixed_results) == 3
assert mixed_results[0].success
assert not mixed_results[1].success
assert mixed_results[2].success

print("\nFailure isolation test passed.")


call_4: SUCCESS | 15
call_5: FAILED | TypeError: calculate_sum() missing 1 required positional argument: 'b'
call_6: SUCCESS | 50

Failure isolation test passed.


In [44]:

batch_messages = [
    result.to_message()
    for result in mixed_results
]

print("Generated tool messages:")

for message in batch_messages:
    print(message)

assert len(batch_messages) == len(mixed_calls)

assert [
    message["tool_call_id"] for message in batch_messages
] == [
    call.id for call in mixed_calls
]

print("\nBatch result-to-message conversion passed.")


Generated tool messages:
{'role': 'tool', 'tool_call_id': 'call_4', 'content': '15'}
{'role': 'tool', 'tool_call_id': 'call_5', 'content': "TypeError: calculate_sum() missing 1 required positional argument: 'b'"}
{'role': 'tool', 'tool_call_id': 'call_6', 'content': '50'}

Batch result-to-message conversion passed.



# Cell 14: Tool Argument Validation

## 1. Objective

Our current ToolRegistry can:
- Find registered functions.
- Parse JSON arguments.
- Execute functions.
- Capture execution errors.

However, we need a separate validation stage before execution.

Consider this function:

    def calculate_sum(a: int, b: int) -> int:
        return a + b

Gemini may generate several types of arguments.

### Case 1: Valid arguments

    {"a": 10, "b": 20}

Both required arguments are present and have the expected types.

Expected: ACCEPT.

### Case 2: Missing argument

    {"a": 10}

The argument b is missing.

Expected: REJECT.

### Case 3: Unexpected argument

    {"a": 10, "b": 20, "c": 30}

The function does not accept an argument named c.

Expected: REJECT.

### Case 4: Incorrect argument type

    {"a": "ten", "b": 20}

The function expects an integer for a, but receives a string.

Expected: REJECT.

### Case 5: Invalid JSON

    {"a": 10, "b": 20

The argument string cannot be parsed.

Expected: REJECT.

## 2. Validation strategy

We will use Python's inspect.signature().

For a function:

    def calculate_sum(a: int, b: int) -> int:
        return a + b

Python can inspect its signature:

    (a: int, b: int) -> int

We can then use signature.bind(**arguments).

This checks:
- Required parameters.
- Unexpected keyword arguments.
- Whether the supplied arguments can bind to the function signature.

We will additionally inspect type annotations to check basic argument types.

## 3. Validation pipeline

    ToolCall
       |
       v
    Parse JSON arguments
       |
       v
    Check that arguments form a dictionary
       |
       v
    Inspect Python function signature
       |
       v
    Check required and unexpected arguments
       |
       v
    Check annotated argument types
       |
       v
    Execute function
       |
       v
    ToolResult

## 4. Important design decision

We separate validation from execution.

A malformed request should never reach the actual function.

For example, if Gemini requests:

    calculate_sum(a="ten", b=20)

The harness should reject it before calculate_sum is executed.

We will support common Python annotations:
- int
- float
- str
- bool
- list
- dict
- tuple
- Optional types
- Union types

Unannotated parameters will not receive runtime type checks.

This is lightweight validation based on Python function signatures. It is not a complete JSON Schema validator.


In [45]:

import inspect
import types

from typing import get_origin, get_args, Union, Any


def matches_type(value: Any, expected_type: Any) -> bool:
    """
    Check whether a value matches a supported Python type annotation.
    """

    if expected_type is Any:
        return True

    if expected_type is None or expected_type is type(None):
        return value is None

    origin = get_origin(expected_type)
    args = get_args(expected_type)

    # Optional and Union types.
    if origin in (Union, types.UnionType):
        return any(matches_type(value, t) for t in args)

    # Generic list, e.g. list[int].
    if origin is list:
        if not isinstance(value, list):
            return False

        element_type = args[0] if args else Any
        return all(matches_type(item, element_type) for item in value)

    # Generic dictionary, e.g. dict[str, int].
    if origin is dict:
        if not isinstance(value, dict):
            return False

        key_type, value_type = args if args else (Any, Any)

        return all(
            matches_type(k, key_type)
            and matches_type(v, value_type)
            for k, v in value.items()
        )

    # Generic tuple, e.g. tuple[int, str].
    if origin is tuple:
        if not isinstance(value, tuple):
            return False

        if len(args) == 2 and args[1] is Ellipsis:
            return all(matches_type(v, args[0]) for v in value)

        if args and len(value) != len(args):
            return False

        return all(
            matches_type(v, t)
            for v, t in zip(value, args)
        )

    # Primitive types.
    if expected_type is bool:
        return isinstance(value, bool)

    if expected_type is int:
        return isinstance(value, int) and not isinstance(value, bool)

    if expected_type is float:
        return (
            isinstance(value, (int, float))
            and not isinstance(value, bool)
        )

    if isinstance(expected_type, type):
        return isinstance(value, expected_type)

    # Unsupported annotation: do not reject based on it.
    return True


def validate_tool_arguments(
    function: Callable,
    arguments: Dict[str, Any]
) -> None:
    """
    Validate arguments against a Python function signature.

    Raises:
        TypeError: If arguments cannot bind or have incorrect types.
    """

    if not isinstance(arguments, dict):
        raise TypeError("Tool arguments must be a dictionary.")

    signature = inspect.signature(function)

    # Validate missing and unexpected arguments.
    try:
        bound_arguments = signature.bind(**arguments)
    except TypeError as e:
        raise TypeError(f"Invalid tool arguments: {e}") from e

    # Validate annotated parameter types.
    for name, value in bound_arguments.arguments.items():
        parameter = signature.parameters[name]
        annotation = parameter.annotation

        if annotation is inspect.Parameter.empty:
            continue

        if not matches_type(value, annotation):
            raise TypeError(
                f"Argument '{name}' must match {annotation}, "
                f"but received {type(value).__name__}."
            )


print("Tool argument validation initialized successfully.")


Tool argument validation initialized successfully.


In [46]:

class ToolRegistry:
    """
    Registry of explicitly permitted Python functions.
    Supports validated individual and batch execution.
    """

    def __init__(self):
        self._tools = {}

    def register(self, name: str, function: Callable):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Tool name must be a non-empty string.")

        if not callable(function):
            raise TypeError("Registered tool must be callable.")

        if name in self._tools:
            raise ValueError(f"Tool already registered: {name}")

        self._tools[name] = function

    def get(self, name: str) -> Callable:
        if name not in self._tools:
            raise KeyError(f"Unknown tool: {name}")

        return self._tools[name]

    def execute(self, tool_call: ToolCall) -> ToolResult:
        try:
            if not isinstance(tool_call, ToolCall):
                raise TypeError("Expected a ToolCall object.")

            function = self.get(tool_call.name)

            arguments = json.loads(tool_call.arguments)

            if not isinstance(arguments, dict):
                raise ValueError(
                    "Tool arguments must decode into a JSON object."
                )

            # Validate BEFORE executing the function.
            validate_tool_arguments(function, arguments)

            result = function(**arguments)

            return ToolResult(
                tool_call_id=tool_call.id,
                name=tool_call.name,
                content=str(result),
                success=True
            )

        except Exception as e:
            return ToolResult(
                tool_call_id=getattr(tool_call, "id", "unknown"),
                name=getattr(tool_call, "name", "unknown"),
                content=f"{type(e).__name__}: {e}",
                success=False
            )

    def execute_many(
        self,
        tool_calls: List[ToolCall]
    ) -> List[ToolResult]:

        if not isinstance(tool_calls, list):
            raise TypeError("tool_calls must be a list.")

        if not all(isinstance(call, ToolCall) for call in tool_calls):
            raise TypeError("Every item must be a ToolCall.")

        call_ids = [call.id for call in tool_calls]

        if len(call_ids) != len(set(call_ids)):
            raise ValueError("Duplicate tool_call IDs in batch.")

        return [self.execute(call) for call in tool_calls]


print("ToolRegistry updated with argument validation.")


ToolRegistry updated with argument validation.


In [47]:

# Create a fresh registry.
validated_registry = ToolRegistry()

validated_registry.register(
    "calculate_sum",
    calculate_sum
)

validation_cases = [
    (
        "Valid arguments",
        '{"a": 10, "b": 20}',
        True
    ),
    (
        "Missing argument",
        '{"a": 10}',
        False
    ),
    (
        "Unexpected argument",
        '{"a": 10, "b": 20, "c": 30}',
        False
    ),
    (
        "Incorrect argument type",
        '{"a": "ten", "b": 20}',
        False
    ),
    (
        "Boolean instead of integer",
        '{"a": true, "b": 20}',
        False
    ),
    (
        "Malformed JSON",
        '{"a": 10, "b": 20',
        False
    ),
    (
        "JSON array instead of object",
        '[10, 20]',
        False
    )
]

for index, (description, arguments, expected_success) in enumerate(
    validation_cases
):

    result = validated_registry.execute(
        ToolCall(
            id=f"validation_{index}",
            name="calculate_sum",
            arguments=arguments
        )
    )

    assert result.success == expected_success, (
        f"Unexpected result for case: {description}"
    )

    print(
        f"[PASS] {description}: "
        f"{'Accepted' if result.success else 'Rejected'}"
    )

print("\nAll tool argument validation tests passed.")


[PASS] Valid arguments: Accepted
[PASS] Missing argument: Rejected
[PASS] Unexpected argument: Rejected
[PASS] Incorrect argument type: Rejected
[PASS] Boolean instead of integer: Rejected
[PASS] Malformed JSON: Rejected
[PASS] JSON array instead of object: Rejected

All tool argument validation tests passed.



# Cell 15: Tool Definition and Registry Consistency

## 1. Objective

We now have two separate components:

1. ToolRegistry: Stores and executes Python functions.
2. Tool definitions: Describes available functions to Gemini.

These two components must remain consistent.

Consider our calculate_sum function:

    def calculate_sum(a: int, b: int) -> int:
        return a + b

We register it in Python:

    registry.register(
        name="calculate_sum",
        function=calculate_sum
    )

Separately, we describe it to Gemini using a tool schema.

    calculate_sum_tool = {
        "type": "function",
        "function": {
            "name": "calculate_sum",
            "description": "Calculate the sum of two integers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"}
                },
                "required": ["a", "b"]
            }
        }
    }

The problem is that these two definitions are independent.

For example:
- Gemini may be told that a function accepts three parameters.
- The actual Python function may accept only two.
- Gemini may be told that a parameter is a string when Python expects an integer.
- A tool may be advertised to Gemini but not registered in our registry.

We need to detect these inconsistencies before making an API request.

## 2. What will we validate?

For each tool definition, we check:

### A. Basic structure

- Tool type must be "function".
- Function name must be a non-empty string.
- Description must be a non-empty string.
- Parameters must be an object schema.

### B. Parameter definitions

- Every property must have a valid name.
- Every property must have a supported JSON Schema type.
- Required parameters must exist in properties.
- Required must be a list of strings.

### C. Registry consistency

- Every advertised tool must be registered.
- Every registered tool we intend to advertise must have a definition.
- Tool names must be unique.

## 3. Why is this important?

Suppose Gemini generates:

    calculate_sum(a=10, b=20)

But our Python registry contains:

    def calculate_sum(a, b, c):
        return a + b + c

The schema and implementation disagree.

This can cause repeated tool execution failures and unnecessary LLM calls.

Our objective is to catch these mistakes before the agent starts reasoning.

## 4. Scope

For now, we validate the basic JSON Schema structure and registry consistency.

We are not implementing a complete JSON Schema engine. More advanced constraints, such as numerical ranges, string patterns, and nested object validation, can be added later.


In [48]:

SUPPORTED_JSON_TYPES = {
    "string",
    "integer",
    "number",
    "boolean",
    "object",
    "array"
}


def validate_tool_definitions(
    tool_definitions: List[Dict[str, Any]],
    registry: ToolRegistry
) -> None:
    """
    Validate tool definitions and check consistency with ToolRegistry.

    Raises ValueError when a schema is malformed or a tool is
    missing from the registry.
    """

    if not isinstance(tool_definitions, list):
        raise TypeError("Tool definitions must be a list.")

    seen_names = set()

    for index, tool in enumerate(tool_definitions):

        if not isinstance(tool, dict):
            raise ValueError(f"Tool {index} must be a dictionary.")

        if tool.get("type") != "function":
            raise ValueError(f"Tool {index} must have type='function'.")

        function = tool.get("function")

        if not isinstance(function, dict):
            raise ValueError(f"Tool {index} has an invalid function object.")

        name = function.get("name")
        description = function.get("description")
        parameters = function.get("parameters")

        if not isinstance(name, str) or not name.strip():
            raise ValueError(f"Tool {index} has an invalid name.")

        if name in seen_names:
            raise ValueError(f"Duplicate tool definition: {name}")

        seen_names.add(name)

        if not isinstance(description, str) or not description.strip():
            raise ValueError(f"Tool '{name}' needs a description.")

        if not isinstance(parameters, dict):
            raise ValueError(
                f"Tool '{name}' needs a parameters schema."
            )

        if parameters.get("type") != "object":
            raise ValueError(
                f"Tool '{name}' parameters must have type='object'."
            )

        properties = parameters.get("properties")

        if not isinstance(properties, dict):
            raise ValueError(
                f"Tool '{name}' properties must be a dictionary."
            )

        for property_name, property_schema in properties.items():

            if not isinstance(property_name, str) or not property_name:
                raise ValueError(
                    f"Tool '{name}' contains an invalid property name."
                )

            if not isinstance(property_schema, dict):
                raise ValueError(
                    f"Invalid schema for '{name}.{property_name}'."
                )

            property_type = property_schema.get("type")

            if property_type not in SUPPORTED_JSON_TYPES:
                raise ValueError(
                    f"Unsupported JSON type for "
                    f"'{name}.{property_name}': {property_type}"
                )

        required = parameters.get("required", [])

        if not isinstance(required, list):
            raise ValueError(
                f"Tool '{name}' required must be a list."
            )

        if not all(isinstance(item, str) for item in required):
            raise ValueError(
                f"Tool '{name}' required entries must be strings."
            )

        unknown_required = set(required) - set(properties)

        if unknown_required:
            raise ValueError(
                f"Tool '{name}' requires undefined properties: "
                f"{unknown_required}"
            )

        # The advertised tool must exist in the registry.
        registry.get(name)

    print(
        f"Validated {len(tool_definitions)} tool definitions "
        "against the registry."
    )


print("Tool schema validator initialized.")


Tool schema validator initialized.


In [49]:

calculate_sum_tool = {
    "type": "function",
    "function": {
        "name": "calculate_sum",
        "description": "Calculate the sum of two integers.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "integer",
                    "description": "First integer."
                },
                "b": {
                    "type": "integer",
                    "description": "Second integer."
                }
            },
            "required": ["a", "b"],
            "additionalProperties": False
        }
    }
}

calculate_sum_definitions = [calculate_sum_tool]

validate_tool_definitions(
    calculate_sum_definitions,
    validated_registry
)

print("Tool definition and registry consistency test passed.")


Validated 1 tool definitions against the registry.
Tool definition and registry consistency test passed.



# Cell 16: Project Workspace and Filesystem Boundary

## 1. Objective

Our eventual code-fixing agent needs to inspect and modify multi-file Python projects.

Before implementing file tools, we need to establish a controlled workspace.

We will create a dedicated directory:

    /content/agent_workspace/

The agent will operate inside this directory rather than having unrestricted access to the Colab filesystem.

## 2. Why do we need a workspace boundary?

Suppose Gemini requests:

    read_file(path="src/model.py")

This should resolve to:

    /content/agent_workspace/src/model.py

But consider a malicious or accidental request:

    read_file(path="../../etc/passwd")

We must prevent the agent from accessing files outside its assigned workspace.

Therefore, every requested file path must be checked before accessing the filesystem.

## 3. Path resolution strategy

We will use Python's pathlib library.

The process is:

1. Receive a relative file path.
2. Reject absolute paths.
3. Resolve the path relative to the workspace.
4. Resolve symbolic links.
5. Verify that the final resolved path remains inside the workspace.
6. Return the validated path.

For example:

    safe_resolve_path("src/model.py")

    # Allowed:
    /content/agent_workspace/src/model.py

But:

    safe_resolve_path("../../outside.txt")

    # Rejected:
    ValueError

This also prevents a symbolic link inside the workspace from being used to access an external directory.

## 4. Design decisions

### A. Dedicated workspace

The agent only operates inside agent_workspace.

### B. Relative paths

Tools receive relative paths rather than arbitrary absolute filesystem paths.

### C. Centralized validation

All future filesystem tools will use the same safe_resolve_path() function.

This avoids duplicating security checks across read_file, write_file, edit_file, and other tools.

### D. No unrestricted execution

This workspace boundary is a filesystem restriction, not a complete operating-system sandbox. We will also restrict and control code execution separately.

## 5. What we implement

- WORKSPACE: Root directory of the agent's project.
- safe_resolve_path(): Validates requested paths.
- Basic tests for valid paths, traversal attempts, absolute paths, and symbolic links.


In [50]:

from pathlib import Path

WORKSPACE = Path("/content/agent_workspace").resolve()

WORKSPACE.mkdir(parents=True, exist_ok=True)

print("Agent workspace initialized.")
print("Workspace:", WORKSPACE)
print("Exists:", WORKSPACE.exists())
print("Is directory:", WORKSPACE.is_dir())


Agent workspace initialized.
Workspace: /content/agent_workspace
Exists: True
Is directory: True


In [51]:

def safe_resolve_path(relative_path: str) -> Path:
    """
    Resolve a relative path while ensuring it remains inside WORKSPACE.

    Rejects absolute paths and paths escaping the workspace,
    including escapes through existing symbolic links.
    """

    if not isinstance(relative_path, str) or not relative_path.strip():
        raise ValueError("Path must be a non-empty string.")

    requested_path = Path(relative_path)

    if requested_path.is_absolute():
        raise ValueError("Absolute paths are not permitted.")

    resolved_path = (WORKSPACE / requested_path).resolve()

    try:
        resolved_path.relative_to(WORKSPACE)
    except ValueError:
        raise ValueError(
            "Access denied: requested path is outside the workspace."
        )

    return resolved_path


print("Safe path resolver initialized.")


Safe path resolver initialized.


In [52]:

# Test 1: Normal relative path
path = safe_resolve_path("src/model.py")

assert path == WORKSPACE / "src/model.py"
print("[PASS] Valid relative path accepted.")


# Test 2: Nested directory
path = safe_resolve_path("src/models/encoder.py")

assert path.is_relative_to(WORKSPACE)
print("[PASS] Nested relative path accepted.")


# Test 3: Parent directory traversal
try:
    safe_resolve_path("../../outside.txt")
    raise AssertionError("Path traversal was not blocked.")
except ValueError:
    print("[PASS] Parent directory traversal blocked.")


# Test 4: Absolute path
try:
    safe_resolve_path("/etc/passwd")
    raise AssertionError("Absolute path was not blocked.")
except ValueError:
    print("[PASS] Absolute path blocked.")


# Test 5: Symbolic link escape
outside_directory = Path("/content/agent_workspace_test_outside")
outside_directory.mkdir(exist_ok=True)

symlink = WORKSPACE / "external_link"

try:
    symlink.symlink_to(outside_directory, target_is_directory=True)

    try:
        safe_resolve_path("external_link/secret.txt")
        raise AssertionError("Symbolic link escape was not blocked.")
    except ValueError:
        print("[PASS] Symbolic link escape blocked.")

finally:
    if symlink.is_symlink():
        symlink.unlink()

    outside_directory.rmdir()


print("\nAll workspace boundary tests passed.")


[PASS] Valid relative path accepted.
[PASS] Nested relative path accepted.
[PASS] Parent directory traversal blocked.
[PASS] Absolute path blocked.
[PASS] Symbolic link escape blocked.

All workspace boundary tests passed.



# Cell 17: Project Exploration Tools

## 1. Objective

A code-fixing agent must first understand the project before attempting to modify it.

For a multi-file Python project, it needs to answer questions such as:

- What files and directories exist?
- Where is the model implementation?
- Where is the training loop?
- Which file defines a particular function?
- What does a particular source file contain?

We implement three tools for these tasks.

## 2. Tool 1: list_files

Purpose:
Explore the directory structure of the project.

Example request:

    list_files(path=".")

Expected output:

    src/
        model.py
        dataset.py
        trainer.py
    tests/
        test_model.py
        test_dataset.py
    config.py
    requirements.txt

Design decisions:

- Supports recursive directory exploration.
- Uses relative paths.
- Skips common generated directories such as __pycache__ and .git.
- Limits the number of returned entries.

## 3. Tool 2: read_file

Purpose:
Read the contents of a source or configuration file.

Example request:

    read_file(path="src/model.py")

Expected output:

    class Model:
        def forward(self, x):
            return self.network(x)

Design decisions:

- Only reads files inside WORKSPACE.
- Rejects directories and non-regular files.
- Uses UTF-8 decoding.
- Limits file size to avoid unexpectedly large tool outputs.

The size limit is important because the entire tool output may subsequently become part of the conversation sent to Gemini.

## 4. Tool 3: search_code

Purpose:
Search project files for a particular string or regular expression.

Example request:

    search_code(
        query="def forward",
        path="."
    )

Expected output:

    src/model.py:15: def forward(self, x):
    src/encoder.py:42: def forward(self, x):

Design decisions:

- Searches recursively.
- Supports literal text and optional regular expressions.
- Skips binary and oversized files.
- Limits the number of matches returned.

## 5. Architecture

The tools follow the same filesystem boundary:

    Gemini tool request
           |
           v
    ToolRegistry
           |
           v
    list_files / read_file / search_code
           |
           v
    safe_resolve_path()
           |
           v
    Workspace filesystem
           |
           v
    ToolResult

No filesystem tool should directly trust a path generated by the model.

## 6. Why these three tools first?

A debugging agent generally needs to follow this sequence:

1. Explore the project.
2. Locate relevant files.
3. Read the implementation.
4. Search for related functions and references.
5. Understand the code before proposing changes.

These three tools establish that exploration capability.


In [53]:

import re
import os
import stat

IGNORED_DIRECTORIES = {
    ".git",
    "__pycache__",
    ".pytest_cache",
    ".mypy_cache",
    ".venv",
    "venv",
    "node_modules"
}

MAX_READ_BYTES = 1_000_000
MAX_SEARCH_FILE_BYTES = 1_000_000
MAX_LIST_ENTRIES = 300
MAX_SEARCH_MATCHES = 100


def list_files(path: str = ".") -> str:
    """
    Recursively list files and directories inside the workspace.
    """

    target = safe_resolve_path(path)

    if not target.exists():
        raise FileNotFoundError(f"Directory does not exist: {path}")

    if not target.is_dir():
        raise NotADirectoryError(f"Not a directory: {path}")

    entries = []
    count = 0

    for root, dirs, files in os.walk(target, followlinks=False):

        # Skip ignored directories and symbolic links.
        dirs[:] = sorted(
            d for d in dirs
            if d not in IGNORED_DIRECTORIES
            and not (Path(root) / d).is_symlink()
        )

        root_path = Path(root)

        for directory in dirs:
            full_path = root_path / directory
            relative = full_path.relative_to(WORKSPACE)

            entries.append(f"[DIR]  {relative}")
            count += 1

            if count >= MAX_LIST_ENTRIES:
                break

        if count >= MAX_LIST_ENTRIES:
            break

        for filename in sorted(files):
            full_path = root_path / filename

            if full_path.is_symlink():
                continue

            relative = full_path.relative_to(WORKSPACE)

            entries.append(f"[FILE] {relative}")
            count += 1

            if count >= MAX_LIST_ENTRIES:
                break

        if count >= MAX_LIST_ENTRIES:
            break

    if not entries:
        return "The directory is empty."

    output = "\n".join(entries)

    if count >= MAX_LIST_ENTRIES:
        output += "\n... Output truncated at entry limit."

    return output


def read_file(path: str) -> str:
    """
    Read a UTF-8 text file inside the workspace.
    """

    target = safe_resolve_path(path)

    if not target.exists():
        raise FileNotFoundError(f"File does not exist: {path}")

    if target.is_symlink() or not target.is_file():
        raise ValueError("Requested path is not a regular file.")

    if target.stat().st_size > MAX_READ_BYTES:
        raise ValueError(
            f"File exceeds the {MAX_READ_BYTES}-byte read limit."
        )

    try:
        return target.read_text(encoding="utf-8")
    except UnicodeDecodeError as e:
        raise ValueError("File is not valid UTF-8 text.") from e


def search_code(
    query: str,
    path: str = ".",
    use_regex: bool = False
) -> str:
    """
    Search text files recursively inside the workspace.

    Returns matching file paths, line numbers, and matching lines.
    """

    if not isinstance(query, str) or not query:
        raise ValueError("Search query must be a non-empty string.")

    target = safe_resolve_path(path)

    if not target.exists():
        raise FileNotFoundError(f"Path does not exist: {path}")

    if use_regex:
        try:
            pattern = re.compile(query)
        except re.error as e:
            raise ValueError(f"Invalid regular expression: {e}") from e
    else:
        pattern = None

    if target.is_file():
        files = [target]
    else:
        files = []

        for root, dirs, filenames in os.walk(target, followlinks=False):
            dirs[:] = sorted(
                d for d in dirs
                if d not in IGNORED_DIRECTORIES
                and not (Path(root) / d).is_symlink()
            )

            for filename in filenames:
                file_path = Path(root) / filename

                if not file_path.is_symlink():
                    files.append(file_path)

    matches = []

    for file_path in sorted(files):

        if len(matches) >= MAX_SEARCH_MATCHES:
            break

        try:
            if file_path.stat().st_size > MAX_SEARCH_FILE_BYTES:
                continue

            content = file_path.read_text(encoding="utf-8")

        except (OSError, UnicodeDecodeError):
            continue

        for line_number, line in enumerate(content.splitlines(), start=1):

            found = (
                pattern.search(line) is not None
                if use_regex
                else query in line
            )

            if found:
                relative = file_path.relative_to(WORKSPACE)

                matches.append(
                    f"{relative}:{line_number}: {line.strip()}"
                )

                if len(matches) >= MAX_SEARCH_MATCHES:
                    break

    if not matches:
        return f"No matches found for: {query}"

    output = "\n".join(matches)

    if len(matches) >= MAX_SEARCH_MATCHES:
        output += "\n... Match limit reached."

    return output


print("Project exploration tools initialized.")


Project exploration tools initialized.


In [54]:

# Create a sample project inside the controlled workspace.

sample_project = {
    "src/model.py": '''
class SimpleModel:
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

    def forward(self, x):
        return x
''',

    "src/trainer.py": '''
def train_model(model, dataset, epochs=10):
    for epoch in range(epochs):
        for batch in dataset:
            predictions = model.forward(batch)
    return model
''',

    "src/dataset.py": '''
class CustomDataset:
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)
''',

    "tests/test_model.py": '''
def test_model_forward():
    model = SimpleModel(10, 20)
    output = model.forward([1, 2, 3])
    assert output is not None
''',

    "config.py": '''
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 10
''',

    "requirements.txt": '''
numpy
pytest
torch
'''
}

for relative_path, content in sample_project.items():

    file_path = safe_resolve_path(relative_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content.strip() + "\n", encoding="utf-8")

print("Sample project created.")
print("Number of files:", len(sample_project))
print("Project root:", WORKSPACE)


Sample project created.
Number of files: 6
Project root: /content/agent_workspace


In [55]:

# Test 1: List project files

print("=" * 55)
print("PROJECT STRUCTURE")
print("=" * 55)

structure = list_files(".")

print(structure)

assert "src/model.py" in structure
assert "src/trainer.py" in structure
assert "tests/test_model.py" in structure

print("\n[PASS] Project exploration.")


# Test 2: Read a source file

print("\n" + "=" * 55)
print("READ SOURCE FILE")
print("=" * 55)

model_source = read_file("src/model.py")

print(model_source)

assert "class SimpleModel" in model_source
assert "def forward" in model_source

print("[PASS] File reading.")


# Test 3: Search for a function

print("\n" + "=" * 55)
print("SEARCH CODE")
print("=" * 55)

search_results = search_code("def forward")

print(search_results)

assert "src/model.py" in search_results

print("\n[PASS] Literal code search.")


# Test 4: Regular expression search

regex_results = search_code(
    r"def (forward|train_model)",
    use_regex=True
)

assert "forward" in regex_results
assert "train_model" in regex_results

print("[PASS] Regular expression search.")


print("\nAll project exploration tests passed.")


PROJECT STRUCTURE
[DIR]  src
[DIR]  tests
[FILE] config.py
[FILE] requirements.txt
[FILE] src/dataset.py
[FILE] src/model.py
[FILE] src/trainer.py
[FILE] tests/test_model.py

[PASS] Project exploration.

READ SOURCE FILE
class SimpleModel:
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

    def forward(self, x):
        return x

[PASS] File reading.

SEARCH CODE
src/model.py:6: def forward(self, x):

[PASS] Literal code search.
[PASS] Regular expression search.

All project exploration tests passed.



# Cell 18: Register Project Tools and Generate Gemini Schemas

## 1. Objective

We have implemented three actual Python functions:

- list_files()
- read_file()
- search_code()

However, Gemini does not automatically know that these functions exist.

We need to connect three components:

1. Python implementations: Actual functions that perform operations.
2. ToolRegistry: Stores and executes the functions.
3. Tool schemas: Describes the available functions to Gemini.

## 2. Registering tools

We create a project registry and register each function.

Example:

    project_registry.register(
        name="read_file",
        function=read_file
    )

The registry now knows how to execute read_file.

## 3. Generating schemas

We also define the corresponding Gemini function schemas.

For example, read_file accepts one required string parameter:

    {
        "path": "src/model.py"
    }

The schema tells Gemini:
- The function name.
- What the function does.
- What arguments it expects.
- Which arguments are required.

## 4. Why maintain both implementations and schemas?

The registry contains executable Python functions.

The schemas contain descriptions of those functions for the LLM.

They serve different purposes and must remain consistent.

## 5. Expected interaction

When we send a request such as:

"Find where the forward method is implemented."

Gemini may request:

    search_code(query="def forward", path=".")

Our harness will:

1. Receive the tool call.
2. Convert it into ToolCall.
3. Execute it through ToolRegistry.
4. Convert ToolResult into a tool message.
5. Append the result to the conversation.

In this cell, we only register tools and validate schemas. The actual Gemini interaction comes next.


In [56]:

project_registry = ToolRegistry()

project_registry.register(
    name="list_files",
    function=list_files
)

project_registry.register(
    name="read_file",
    function=read_file
)

project_registry.register(
    name="search_code",
    function=search_code
)

print("Registered project tools:")

for name in project_registry._tools:
    print("-", name)

assert len(project_registry._tools) == 3

print("\nProject tool registration passed.")


Registered project tools:
- list_files
- read_file
- search_code

Project tool registration passed.


In [57]:

project_tool_definitions = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": (
                "List files and directories recursively inside the project workspace."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Relative directory path. Use '.' for the project root."
                    }
                },
                "required": [],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": (
                "Read the UTF-8 text contents of a file inside the project workspace."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "Relative path of the file to read."
                    }
                },
                "required": ["path"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_code",
            "description": (
                "Search project files for a literal string or regular expression."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Text or regular expression to search for."
                    },
                    "path": {
                        "type": "string",
                        "description": "Relative directory or file path. Defaults to '.'."
                    },
                    "use_regex": {
                        "type": "boolean",
                        "description": "Whether to interpret query as a regular expression."
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    }
]

print("Gemini tool schemas defined.")
print("Number of tools:", len(project_tool_definitions))


Gemini tool schemas defined.
Number of tools: 3


In [58]:

validate_tool_definitions(
    project_tool_definitions,
    project_registry
)

registered_names = set(project_registry._tools)

schema_names = {
    tool["function"]["name"]
    for tool in project_tool_definitions
}

assert registered_names == schema_names

print("\nRegistered tools and advertised schemas match.")


Validated 3 tool definitions against the registry.

Registered tools and advertised schemas match.



# Cell 19: First End-to-End Tool Interaction with Gemini

## 1. Objective

We now combine the components developed so far:

- GeminiClient
- ToolCall
- AssistantTurn
- ToolRegistry
- ToolResult
- Project exploration tools
- Conversation protocol validator

Previously, we tested Gemini function calling, but we did not execute the requested functions.

Now we will implement the complete interaction:

    User request
         |
         v
    Gemini receives conversation + tool schemas
         |
         v
    Gemini generates tool calls
         |
         v
    GeminiClient normalizes the response
         |
         v
    ToolRegistry executes the calls
         |
         v
    ToolResult objects are generated
         |
         v
    Results become role="tool" messages
         |
         v
    Updated conversation is sent back to Gemini
         |
         v
    Gemini produces its final answer

## 2. Example task

We ask Gemini:

"Inspect this project. Find where the forward method is implemented and explain what it currently does."

Gemini should determine that it needs to:

1. Explore the project if necessary.
2. Search for the forward method.
3. Read the relevant source file.
4. Explain its implementation.

The exact tool sequence is chosen by Gemini, not hardcoded by us.

## 3. Why is this important?

This is the first time our model and execution infrastructure work together.

We are testing whether:

- Gemini understands the available tool descriptions.
- Gemini generates valid ToolCall objects.
- Our registry executes the requested functions.
- Tool results are correctly associated with their call IDs.
- Gemini can use the returned observations to produce an answer.

## 4. Important limitation

This is a single tool-execution round, not yet a complete autonomous agent loop.

If Gemini requests tools, we execute them and send their results back once.

The agent does not yet repeatedly reason, call tools, inspect errors, and continue until the task is complete.

We will build that ReAct loop separately.


In [59]:

project_messages = [
    {
        "role": "system",
        "content": (
            "You are a Python code analysis assistant. "
            "You have access to tools for exploring a project. "
            "Use the tools to inspect the actual source code before "
            "making claims. Do not invent file contents."
        )
    },
    {
        "role": "user",
        "content": (
            "Inspect the project. Find where the forward method is "
            "implemented and explain what it currently does."
        )
    }
]

project_response = gemini.chat(
    messages=project_messages,
    tools=project_tool_definitions,
    tool_choice="required"
)

print("Gemini response received.")
print("Finish reason:", project_response.finish_reason)
print("Content:", project_response.content)
print("Number of tool calls:", len(project_response.tool_calls))

for call in project_response.tool_calls:
    print("\nTool:", call.name)
    print("Arguments:", call.arguments)

assert len(project_response.tool_calls) > 0

print("\nGemini project exploration request passed.")


Gemini response received.
Finish reason: tool_calls
Content: None
Number of tool calls: 1

Tool: search_code
Arguments: {"query":"def forward"}

Gemini project exploration request passed.


In [60]:

# Convert normalized assistant response into protocol format.
assistant_message = project_response.to_message()

# Append the assistant tool-call message.
project_messages.append(assistant_message)

# Execute all requested tools.
project_results = project_registry.execute_many(
    project_response.tool_calls
)

# Convert execution results into conversation messages.
tool_messages = [
    result.to_message()
    for result in project_results
]

project_messages.extend(tool_messages)

print("Tool execution results:")
print("=" * 50)

for result in project_results:
    print("Tool:", result.name)
    print("Success:", result.success)
    print("Observation:")
    print(result.content)
    print("-" * 50)

# Validate the updated conversation.
validate_messages(project_messages)

print("\nTool execution and protocol validation passed.")


Tool execution results:
Tool: search_code
Success: True
Observation:
src/model.py:6: def forward(self, x):
--------------------------------------------------

Tool execution and protocol validation passed.


In [61]:

# Gemini now receives its original request and the tool observations.

final_project_response = gemini.chat(
    messages=project_messages,
    tools=project_tool_definitions
)

print("Gemini final response:")
print("=" * 60)
print(final_project_response.content)

assert isinstance(final_project_response, AssistantTurn)
assert final_project_response.content is not None

print("\nEnd-to-end tool interaction completed.")


LLMBadRequestError: Gemini rejected the request: HTTP 400


# Cell 19.4: Diagnose the Gemini HTTP 400 Error

## Objective

The local tool execution succeeded, but Gemini rejected the follow-up conversation.

Before modifying anything, inspect the exact message structures.

Important fields to inspect:

1. Assistant message content.
2. Assistant tool-call ID.
3. Assistant tool-call name.
4. Tool response ID.
5. Tool response content.

A possible compatibility issue is that the OpenAI-compatible Gemini endpoint may reject an assistant tool-call message whose content is null.

Some developers have reported this behavior with Google's compatibility endpoint.

We will inspect the actual values in our conversation rather than assume this is the cause.

The diagnostic is entirely local and does not make another API request.


In [62]:

print("ASSISTANT TOOL-CALL MESSAGE")
print("=" * 55)

assistant_tool_message = project_messages[2]

print(json.dumps(assistant_tool_message, indent=2))

print("\nTOOL RESULT MESSAGES")
print("=" * 55)

for message in project_messages[3:]:
    print(json.dumps(message, indent=2))

print("\nDIAGNOSTIC CHECKS")
print("=" * 55)

for call in project_response.tool_calls:
    print("Tool ID:", repr(call.id))
    print("Tool name:", repr(call.name))
    print("Arguments:", call.arguments)

print("Assistant content:", repr(project_response.content))
print("Assistant content is None:", project_response.content is None)


ASSISTANT TOOL-CALL MESSAGE
{
  "role": "assistant",
  "content": null,
  "tool_calls": [
    {
      "id": "call_6134954952079648274",
      "type": "function",
      "function": {
        "name": "search_code",
        "arguments": "{\"query\":\"def forward\"}"
      }
    }
  ]
}

TOOL RESULT MESSAGES
{
  "role": "tool",
  "tool_call_id": "call_6134954952079648274",
  "content": "src/model.py:6: def forward(self, x):"
}

DIAGNOSTIC CHECKS
Tool ID: 'call_6134954952079648274'
Tool name: 'search_code'
Arguments: {"query":"def forward"}
Assistant content: None
Assistant content is None: True



# Cell 19.5: Gemini OpenAI-Compatible Message Serialization

## 1. Problem

Our normalized AssistantTurn correctly represents a tool-only response with:

    content=None

This is valid in our internal data model.

However, Google's OpenAI-compatible endpoint may reject an assistant message containing tool_calls when its content is null.

This is a provider-specific serialization issue, not a failure of our ToolRegistry or conversation protocol.

## 2. Solution

We preserve the original conversation and create a separate copy for Gemini.

Before sending the request:

    provider_messages = copy.deepcopy(messages)

We inspect assistant messages containing tool_calls.

If their content is None, we replace it with:

    "[TOOL CALL]"

This placeholder is only for API compatibility. It is not a real assistant explanation and is not stored in our canonical conversation history.

## 3. Why not modify AssistantTurn.to_message()?

Because we want to keep the internal protocol provider-independent.

If we later add another LLM provider, its serialization requirements may differ.

The correct architectural separation is:

    Internal conversation
            |
            v
    Provider-specific serializer
            |
            v
    Gemini API request

## 4. Diagnostic test

We will resend the same conversation with the placeholder applied.

We will also expose the actual HTTP error body if Gemini rejects the request again.

This will help distinguish a null-content problem from other compatibility issues, such as tool-call metadata or thought signatures.


In [63]:

# Preserve the canonical conversation.
provider_messages = copy.deepcopy(project_messages)

# Apply a Gemini-specific compatibility adjustment.
for message in provider_messages:
    if (
        message.get("role") == "assistant"
        and message.get("tool_calls")
        and message.get("content") is None
    ):
        message["content"] = "[TOOL CALL]"

print("Canonical assistant content:",
      repr(project_messages[2]["content"]))

print("Gemini request assistant content:",
      repr(provider_messages[2]["content"]))

assert project_messages[2]["content"] is None
assert provider_messages[2]["content"] == "[TOOL CALL]"

try:
    diagnostic_completion = gemini.client.chat.completions.create(
        model=gemini.model,
        messages=provider_messages,
        tools=project_tool_definitions,
        temperature=DEFAULT_TEMPERATURE,
        max_tokens=MAX_OUTPUT_TOKENS
    )

    diagnostic_message = diagnostic_completion.choices[0].message

    print("\nGemini accepted the serialized conversation.")
    print("Response:", diagnostic_message.content)
    print("New tool calls:", diagnostic_message.tool_calls)

except Exception as e:
    print("\nGemini rejected the serialized conversation.")
    print("Exception:", type(e).__name__)

    response = getattr(e, "response", None)

    if response is not None:
        print("HTTP status:", response.status_code)
        print("Provider error body:", response.text)
    else:
        print("Details:", str(e))


Canonical assistant content: None
Gemini request assistant content: '[TOOL CALL]'

Gemini rejected the serialized conversation.
Exception: BadRequestError
HTTP status: 400
Provider error body: [{
  "error": {
    "code": 400,
    "message": "Function call is missing a thought_signature in functionCall parts. This is required for tools to work correctly, and missing thought_signature may lead to degraded model performance. Additional data, function call `default_api:search_code` , position 2. Please refer to https://aistudio.google.com/docs/thought-signatures for more details.",
    "status": "INVALID_ARGUMENT"
  }
}
]



# Cell 19.6: Inspect Raw Gemini Response Metadata

## 1. Objective

Our GeminiClient stores the original provider response in:

    AssistantTurn.raw

We need to inspect this response to determine where Gemini placed the thought signature.

The OpenAI-compatible response may contain additional metadata such as:

    tool_call.extra_content.google.thought_signature

This field is not part of our current ToolCall dataclass.

## 2. What we will inspect

- Original completion response.
- Original assistant message.
- Original tool-call object.
- Additional provider metadata.
- Whether the thought signature exists.

We will inspect metadata keys and signature length rather than printing the complete opaque signature.

## 3. Important design decision

We should preserve provider metadata without interpreting or modifying it.

Our normalized ToolCall should eventually retain both:

- Standard fields: id, name, arguments.
- Provider metadata: additional fields required for round-trip compatibility.

This also makes our architecture more extensible for other providers.


In [64]:

raw_completion = project_response.raw

raw_assistant_message = raw_completion.choices[0].message

raw_tool_calls = raw_assistant_message.tool_calls

print("RAW ASSISTANT MESSAGE")
print("=" * 55)

print("Message type:", type(raw_assistant_message))
print("Available attributes:", raw_assistant_message.model_fields.keys())

print("\nRAW TOOL CALL")
print("=" * 55)

raw_tool_call = raw_tool_calls[0]

print("Tool call type:", type(raw_tool_call))
print("Available attributes:", raw_tool_call.model_fields.keys())

# Inspect all non-null fields.
raw_tool_dump = raw_tool_call.model_dump(exclude_none=True)

print("\nSerialized tool-call keys:")
print(list(raw_tool_dump.keys()))

# Look for provider-specific metadata.
print("\nPROVIDER METADATA")
print("=" * 55)

extra_content = raw_tool_dump.get("extra_content")

if extra_content is None:
    extra_content = getattr(raw_tool_call, "extra_content", None)

print("Extra content available:", extra_content is not None)

if extra_content is not None:
    print("Metadata keys:", list(extra_content.keys()))

    google_metadata = extra_content.get("google", {})

    print("Google metadata keys:", list(google_metadata.keys()))

    signature = google_metadata.get("thought_signature")

    print("Thought signature exists:", signature is not None)

    if signature is not None:
        print("Signature type:", type(signature).__name__)
        print("Signature length:", len(signature))

else:
    print("No extra_content found.")

    print("\nFull raw tool-call representation:")
    print(raw_tool_dump)


RAW ASSISTANT MESSAGE
Message type: <class 'openai.types.chat.chat_completion_message.ChatCompletionMessage'>
Available attributes: dict_keys(['content', 'refusal', 'role', 'annotations', 'audio', 'function_call', 'tool_calls'])

RAW TOOL CALL
Tool call type: <class 'openai.types.chat.chat_completion_message_function_tool_call.ChatCompletionMessageFunctionToolCall'>
Available attributes: dict_keys(['id', 'function', 'type'])

Serialized tool-call keys:
['id', 'function', 'type', 'extra_content']

PROVIDER METADATA
Extra content available: True
Metadata keys: ['google']
Google metadata keys: ['thought_signature']
Thought signature exists: True
Signature type: str
Signature length: 132


/tmp/ipykernel_1048/2691012285.py:11: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  print("Available attributes:", raw_assistant_message.model_fields.keys())
/tmp/ipykernel_1048/2691012285.py:19: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  print("Available attributes:", raw_tool_call.model_fields.keys())



### Why this change is necessary

Our normalized `ToolCall` currently stores only three fields:

- `id`
- `name`
- `arguments`

But Gemini also returns provider-specific metadata inside each tool call. In particular, Gemini 3 requires the `thought_signature` to be returned with the corresponding function call in the next request.

For example, the original tool call contains additional metadata conceptually structured like this:

```python
{
    "id": "call_123",
    "type": "function",
    "function": {
        "name": "search_code",
        "arguments": "{\"query\": \"def forward\"}"
    },
    "extra_content": {
        "google": {
            "thought_signature": "..."
        }
    }
}
```

Our current `to_message()` method reconstructs the message without this final metadata field.

**For now, we will preserve the original SDK message as a dictionary.** This is a minimal change that lets us verify the complete Gemini tool-calling round trip before changing the normalized data structures.


In [65]:

# Find the assistant message containing the tool call
assistant_message_index = next(
    i
    for i in range(len(project_messages) - 1, -1, -1)
    if project_messages[i].get("role") == "assistant"
    and project_messages[i].get("tool_calls")
)

# Recover the complete original assistant message from Gemini's response
original_assistant_message = (
    project_response.raw.choices[0].message.model_dump(exclude_none=True)
)

# Replace our reconstructed message with the original message
project_messages[assistant_message_index] = original_assistant_message

# Verify that the thought signature has been preserved
preserved_call = original_assistant_message["tool_calls"][0]

preserved_signature = (
    preserved_call
    .get("extra_content", {})
    .get("google", {})
    .get("thought_signature")
)

assert preserved_signature is not None, "Thought signature was not preserved!"

# Validate the conversation again
validate_messages(project_messages)

print("Original assistant message restored.")
print("Thought signature preserved:", preserved_signature is not None)
print("Signature length:", len(preserved_signature))
print("Conversation protocol validation passed.")

print("\nAssistant message keys:", list(original_assistant_message.keys()))
print("Tool-call keys:", list(preserved_call.keys()))
print("Tool result:", project_messages[assistant_message_index + 1])


Original assistant message restored.
Thought signature preserved: True
Signature length: 132
Conversation protocol validation passed.

Assistant message keys: ['role', 'tool_calls']
Tool-call keys: ['id', 'function', 'type', 'extra_content']
Tool result: {'role': 'tool', 'tool_call_id': 'call_6134954952079648274', 'content': 'src/model.py:6: def forward(self, x):'}



### What happens in this step?

Our conversation now contains:

1. The system instruction.
2. The user's request to inspect the project.
3. Gemini's original assistant message, including its `thought_signature`.
4. The result returned by our `search_code` tool.

We now send this complete conversation back to Gemini.

There are two possible outcomes:

- **Successful tool round trip:** Gemini understands the tool result and responds with an explanation or another tool call.
- **Another API error:** We inspect the exact response and adjust the provider-specific message format.

We will not execute any additional tool calls yet. First, we want to confirm that Gemini accepts the conversation.


In [66]:

# Validate the complete conversation before sending it
validate_messages(project_messages)

# Send the conversation back to Gemini
try:
    project_followup_response = gemini.chat(
        messages=project_messages,
        tools=project_tool_definitions
    )

    print("Second Gemini request succeeded!")
    print("\nAssistant response:")
    print(project_followup_response.content)

    print("\nAdditional tool calls:", len(project_followup_response.tool_calls))

    for i, call in enumerate(project_followup_response.tool_calls, start=1):
        print(f"\nTool call {i}:")
        print("Name:", call.name)
        print("Arguments:", call.arguments)

    print("\nFinish reason:", project_followup_response.finish_reason)
    print("Usage:", project_followup_response.usage)

except Exception as e:
    print("Second Gemini request failed.")
    print("Exception type:", type(e).__name__)
    print("Error:", str(e))


Second Gemini request succeeded!

Assistant response:
None

Additional tool calls: 1

Tool call 1:
Name: read_file
Arguments: {"path":"src/model.py"}

Finish reason: tool_calls
Usage: {'prompt_tokens': 333, 'completion_tokens': 20, 'total_tokens': 353}



### What happens in this step?

Gemini requested:

```python
read_file(path="src/model.py")
```

Our harness must now:

1. Append Gemini's original assistant message, preserving its provider metadata.
2. Execute `read_file` through the tool registry.
3. Append the tool result as a tool message.
4. Validate the complete conversation.
5. Send the updated conversation back to Gemini.

This is the complete tool interaction pattern. Gemini can request multiple tools in succession before producing its final textual answer.


In [67]:

# 1. Preserve Gemini's original assistant message
raw_assistant_message = (
    project_followup_response.raw.choices[0].message.model_dump(
        exclude_none=True
    )
)

project_messages.append(raw_assistant_message)

# 2. Execute the requested tool
read_results = project_registry.execute_many(
    project_followup_response.tool_calls
)

# 3. Append tool observations to the conversation
for result in read_results:
    project_messages.append(result.to_message())

    print(f"Tool: {result.tool_name}")
    print(f"Success: {result.success}")
    print("\nObservation:")
    print(result.content)

# 4. Validate the complete conversation
validate_messages(project_messages)

print("\nConversation protocol validation passed.")
print("Total messages:", len(project_messages))

# 5. Ask Gemini to continue
project_final_response = gemini.chat(
    messages=project_messages,
    tools=project_tool_definitions
)

print("\n" + "=" * 60)
print("GEMINI'S RESPONSE")
print("=" * 60)

print(project_final_response.content)

print("\nAdditional tool calls:", len(project_final_response.tool_calls))
print("Finish reason:", project_final_response.finish_reason)
print("Usage:", project_final_response.usage)


AttributeError: 'ToolResult' object has no attribute 'tool_name'

In [68]:

from dataclasses import fields, is_dataclass

result = read_results[0]

print("ToolResult fields:")
if is_dataclass(result):
    for field in fields(result):
        print(f"  {field.name}: {getattr(result, field.name)!r}")
else:
    print(vars(result))

print("\nLast conversation message:")
print(project_messages[-1])

print("\nNumber of tool results:", len(read_results))


ToolResult fields:
  tool_call_id: 'call_3532307127395177360'
  name: 'read_file'
  content: 'class SimpleModel:\n    def __init__(self, input_dim, hidden_dim):\n        self.input_dim = input_dim\n        self.hidden_dim = hidden_dim\n\n    def forward(self, x):\n        return x\n'
  success: True

Last conversation message:
{'role': 'tool', 'tool_call_id': 'call_3532307127395177360', 'content': 'class SimpleModel:\n    def __init__(self, input_dim, hidden_dim):\n        self.input_dim = input_dim\n        self.hidden_dim = hidden_dim\n\n    def forward(self, x):\n        return x\n'}

Number of tool results: 1



The conversation now contains:

1. User's request.
2. Gemini's `search_code` call and its observation.
3. Gemini's `read_file` call and its observation.

We will send this complete conversation to Gemini again.

This time, Gemini should have enough information to explain the implementation of `forward()`. It may still request another tool if it needs more information.


In [69]:

# Confirm the latest tool execution succeeded
assert read_results[0].success is True
assert project_messages[-1]["role"] == "tool"

# Validate the complete conversation
validate_messages(project_messages)

# Ask Gemini to continue
project_final_response = gemini.chat(
    messages=project_messages,
    tools=project_tool_definitions
)

print("=" * 60)
print("GEMINI RESPONSE")
print("=" * 60)

print("Content:", project_final_response.content)
print("Tool calls:", len(project_final_response.tool_calls))
print("Finish reason:", project_final_response.finish_reason)
print("Usage:", project_final_response.usage)

for i, call in enumerate(project_final_response.tool_calls, start=1):
    print(f"\nAdditional tool call {i}:")
    print("Name:", call.name)
    print("Arguments:", call.arguments)


GEMINI RESPONSE
Content: None
Tool calls: 1
Finish reason: tool_calls
Usage: {'prompt_tokens': 422, 'completion_tokens': 16, 'total_tokens': 438}

Additional tool call 1:
Name: list_files
Arguments: {"path":"."}



### What happens here?

Gemini requested the `list_files` tool with the project root as its path.

We will:

1. Preserve Gemini's complete original assistant message, including its thought signature.
2. Execute `list_files`.
3. Append the result to the conversation.
4. Ask Gemini to continue reasoning.

Notice that we are still manually coordinating these steps. In the next stage, we will put them inside a loop so this process happens automatically.


In [70]:

# 1. Preserve Gemini's original assistant message
raw_assistant_message = (
    project_final_response.raw.choices[0].message.model_dump(
        exclude_none=True
    )
)

project_messages.append(raw_assistant_message)

# 2. Execute Gemini's requested tool call
list_results = project_registry.execute_many(
    project_final_response.tool_calls
)

# 3. Append tool observations
for result in list_results:
    project_messages.append(result.to_message())

    print(f"Tool: {result.name}")
    print(f"Success: {result.success}")
    print("\nObservation:")
    print(result.content)

# 4. Validate the updated conversation
validate_messages(project_messages)

print("\nConversation protocol validation passed.")

# 5. Ask Gemini to continue
project_next_response = gemini.chat(
    messages=project_messages,
    tools=project_tool_definitions
)

print("\n" + "=" * 60)
print("GEMINI RESPONSE")
print("=" * 60)

print("Content:", project_next_response.content)
print("Tool calls:", len(project_next_response.tool_calls))
print("Finish reason:", project_next_response.finish_reason)
print("Usage:", project_next_response.usage)

for i, call in enumerate(project_next_response.tool_calls, start=1):
    print(f"\nTool call {i}:")
    print("Name:", call.name)
    print("Arguments:", call.arguments)


Tool: list_files
Success: True

Observation:
[DIR]  src
[DIR]  tests
[FILE] config.py
[FILE] requirements.txt
[FILE] src/dataset.py
[FILE] src/model.py
[FILE] src/trainer.py
[FILE] tests/test_model.py

Conversation protocol validation passed.

GEMINI RESPONSE
Content: None
Tool calls: 1
Finish reason: tool_calls
Usage: {'prompt_tokens': 514, 'completion_tokens': 16, 'total_tokens': 530}

Tool call 1:
Name: search_code
Arguments: {"query":"forward"}



### 1. The ReAct interaction cycle

ReAct stands for **Reason + Act**. The model alternates between deciding what to do and using tools to gather information.

Our harness implements this cycle:

1. **Reason:** Gemini receives the conversation and decides what information it needs.
2. **Act:** Gemini returns one or more tool calls.
3. **Observe:** Our tool registry executes those calls and returns observations.
4. **Continue:** We append the observations and ask Gemini what to do next.
5. **Finish:** When Gemini returns a response without tool calls, the agent stops.

For example, a project-inspection task might follow this sequence:

| Round | Gemini requests | Harness returns |
|---|---|---|
| 1 | `search_code` | Matching source locations |
| 2 | `read_file` | File contents |
| 3 | `list_files` | Project structure |
| 4 | `search_code` | Additional matching locations |
| 5 | No tool call | Final explanation |

The number of rounds is not fixed. Gemini decides when it has enough information.

### 2. Important implementation details

- We preserve the complete original assistant message using `model_dump()`. This retains Gemini's `thought_signature`.
- We execute tools through `ToolRegistry`, not arbitrary model-generated Python.
- We validate the conversation after every tool-execution round.
- We impose a maximum number of rounds to prevent an endless tool-calling cycle.
- The function returns the complete conversation history, final response, and number of rounds.

For this first implementation, tool execution remains sequential through our existing `execute_many()` method.


In [71]:

import copy


def run_agent(
    llm,
    messages,
    registry,
    tool_definitions,
    initial_response=None,
    max_steps=10,
    verbose=True,
):
    """
    Run a ReAct-style agent loop.

    Parameters
    ----------
    llm:
        LLM client implementing chat().
    messages:
        Initial conversation history.
    registry:
        ToolRegistry containing executable tools.
    tool_definitions:
        Function definitions sent to Gemini.
    initial_response:
        Optional already-obtained AssistantTurn.
        Useful when Gemini has already returned the first tool call.
    max_steps:
        Maximum number of tool-execution rounds.
    verbose:
        Whether to print tool execution details.

    Returns
    -------
    history:
        Complete conversation history.
    response:
        Last AssistantTurn returned by the model.
    steps:
        Number of tool-execution rounds.
    """

    if max_steps < 1:
        raise ValueError("max_steps must be at least 1.")

    # Work on a copy so the caller's original conversation is not modified.
    history = copy.deepcopy(messages)

    validate_messages(history)

    # Use an existing response if one has already been obtained.
    # Otherwise, ask Gemini for the first response.
    response = initial_response

    if response is None:
        response = llm.chat(
            messages=history,
            tools=tool_definitions,
        )

    steps = 0

    while response.tool_calls:

        if steps >= max_steps:
            if verbose:
                print("\nMaximum agent steps reached.")
            break

        steps += 1

        if verbose:
            print("\n" + "=" * 65)
            print(f"AGENT STEP {steps}")
            print("=" * 65)

        # 1. Preserve the complete original Gemini assistant message.
        assistant_message = (
            response.raw.choices[0].message.model_dump(
                exclude_none=True
            )
        )

        history.append(assistant_message)

        # 2. Execute all requested tool calls.
        results = registry.execute_many(response.tool_calls)

        # 3. Append tool observations.
        for result in results:
            history.append(result.to_message())

            if verbose:
                print(f"\nTool: {result.name}")
                print(f"Success: {result.success}")
                print("Observation:")
                print(result.content)

        # 4. Validate the updated conversation.
        validate_messages(history)

        # 5. Ask Gemini to continue.
        response = llm.chat(
            messages=history,
            tools=tool_definitions,
        )

        if verbose:
            print("\nGemini finish reason:", response.finish_reason)
            print("New tool calls:", len(response.tool_calls))

    if verbose:
        print("\n" + "=" * 65)
        print("AGENT FINISHED")
        print("=" * 65)

        if response.tool_calls:
            print("Stopped because the maximum step limit was reached.")
        else:
            print("Final response received.")

        print("Tool-execution rounds:", steps)
        print("Total conversation messages:", len(history))

        if response.content:
            print("\nFinal answer:")
            print(response.content)

    return history, response, steps


In [72]:

agent_history, agent_response, agent_steps = run_agent(
    llm=gemini,
    messages=project_messages,
    registry=project_registry,
    tool_definitions=project_tool_definitions,
    initial_response=project_next_response,
    max_steps=10,
    verbose=True,
)

print("\nFinal response object:")
print("Content:", agent_response.content)
print("Finish reason:", agent_response.finish_reason)
print("Tool calls remaining:", len(agent_response.tool_calls))



AGENT STEP 1

Tool: search_code
Success: True
Observation:
src/model.py:6: def forward(self, x):
src/trainer.py:4: predictions = model.forward(batch)
tests/test_model.py:1: def test_model_forward():
tests/test_model.py:3: output = model.forward([1, 2, 3])

Gemini finish reason: tool_calls
New tool calls: 1

AGENT STEP 2

Tool: read_file
Success: True
Observation:
def test_model_forward():
    model = SimpleModel(10, 20)
    output = model.forward([1, 2, 3])
    assert output is not None


Gemini finish reason: tool_calls
New tool calls: 1

AGENT STEP 3

Tool: read_file
Success: True
Observation:
def train_model(model, dataset, epochs=10):
    for epoch in range(epochs):
        for batch in dataset:
            predictions = model.forward(batch)
    return model


Gemini finish reason: stop
New tool calls: 0

AGENT FINISHED
Final response received.
Tool-execution rounds: 3
Total conversation messages: 14

Final answer:
After inspecting the project, here is where the `forward` method i


### Objective

Implement a tool that allows Gemini to create or modify files inside the project workspace.

The tool will have two arguments:

- `path`: Relative path of the file to write.
- `content`: Complete text content to write.

Example:

```python
write_file(
    path="src/model.py",
    content="class SimpleModel:\n    pass\n"
)
```

### Safety constraints

1. Only paths inside `WORKSPACE` are permitted.
2. Absolute paths and path traversal such as `../../outside.py` are rejected.
3. Symlink escapes are rejected by our existing `safe_resolve_path`.
4. Parent directories are created when necessary.
5. A maximum write size prevents unexpectedly large outputs.
6. The tool reports success or raises an exception; it never silently claims a failed write succeeded.

For now, this tool replaces the complete contents of a file. Later, we can implement a dedicated `edit_file` tool for targeted changes.


In [73]:

# Maximum content size permitted for one write operation.
MAX_WRITE_BYTES = 200_000


def write_file(path: str, content: str) -> str:
    """
    Create or overwrite a text file inside WORKSPACE.
    """

    if not isinstance(path, str) or not path.strip():
        raise ValueError("path must be a non-empty string.")

    if not isinstance(content, str):
        raise TypeError("content must be a string.")

    content_size = len(content.encode("utf-8"))

    if content_size > MAX_WRITE_BYTES:
        raise ValueError(
            f"Content exceeds the maximum allowed size "
            f"({MAX_WRITE_BYTES} bytes)."
        )

    # Resolve and validate the path against workspace restrictions.
    target = safe_resolve_path(path)

    # Create missing parent directories.
    target.parent.mkdir(parents=True, exist_ok=True)

    # Write the file.
    target.write_text(content, encoding="utf-8")

    return (
        f"Successfully wrote {content_size} bytes to "
        f"{target.relative_to(WORKSPACE)}."
    )



We should test both normal behaviour and security restrictions before giving this tool to Gemini.

The tests below verify:

- Creating a new file.
- Overwriting an existing file.
- Creating a file inside a previously nonexistent directory.
- Rejecting path traversal.
- Rejecting absolute paths.
- Rejecting oversized content.

The test files are temporary and will be removed afterward.


In [74]:

# Test 1: Create a file
test_path = ".agent_scratch/write_test.txt"

message = write_file(test_path, "Hello from the agent!")
target = safe_resolve_path(test_path)

assert target.read_text(encoding="utf-8") == "Hello from the agent!"
print("Test 1 passed: File creation")

# Test 2: Overwrite an existing file
write_file(test_path, "Updated content")

assert target.read_text(encoding="utf-8") == "Updated content"
print("Test 2 passed: File overwrite")

# Test 3: Create missing parent directories
nested_path = ".agent_scratch/nested/example.txt"

write_file(nested_path, "Nested file")

nested_target = safe_resolve_path(nested_path)
assert nested_target.read_text(encoding="utf-8") == "Nested file"
print("Test 3 passed: Parent directory creation")

# Test 4: Reject path traversal
try:
    write_file("../../outside.txt", "Not allowed")
    raise AssertionError("Path traversal was not rejected.")
except (ValueError, PermissionError):
    print("Test 4 passed: Path traversal rejected")

# Test 5: Reject absolute paths
try:
    write_file(str(WORKSPACE.parent / "outside.txt"), "Not allowed")
    raise AssertionError("Absolute path was not rejected.")
except (ValueError, PermissionError):
    print("Test 5 passed: Absolute path rejected")

# Test 6: Reject oversized content
try:
    write_file(
        ".agent_scratch/large.txt",
        "x" * (MAX_WRITE_BYTES + 1)
    )
    raise AssertionError("Oversized content was not rejected.")
except ValueError:
    print("Test 6 passed: Oversized content rejected")

# Cleanup temporary test directory
import shutil

shutil.rmtree(WORKSPACE / ".agent_scratch", ignore_errors=True)

print("\nAll write_file tests passed.")


Test 1 passed: File creation
Test 2 passed: File overwrite
Test 3 passed: Parent directory creation
Test 4 passed: Path traversal rejected
Test 5 passed: Absolute path rejected
Test 6 passed: Oversized content rejected

All write_file tests passed.



Our registry currently contains three tools:

- `list_files`
- `read_file`
- `search_code`

We want to add `write_file` without disturbing the existing tools.

First, inspect the registry's available methods and the existing Gemini function definitions. Then we can use exactly the same registration convention.


In [75]:

import inspect

print("REGISTRY TYPE")
print(type(project_registry))

print("\nREGISTRY METHODS")
for name in dir(project_registry):
    if not name.startswith("_"):
        attribute = getattr(project_registry, name)
        if callable(attribute):
            try:
                print(f"{name}{inspect.signature(attribute)}")
            except (TypeError, ValueError):
                print(name)

print("\nEXISTING TOOL DEFINITIONS")
for definition in project_tool_definitions:
    print(definition)

print("\nWRITE_FILE SIGNATURE")
print(inspect.signature(write_file))


REGISTRY TYPE
<class '__main__.ToolRegistry'>

REGISTRY METHODS
execute(tool_call: __main__.ToolCall) -> __main__.ToolResult
execute_many(tool_calls: List[__main__.ToolCall]) -> List[__main__.ToolResult]
get(name: str) -> Callable
register(name: str, function: Callable)

EXISTING TOOL DEFINITIONS
{'type': 'function', 'function': {'name': 'list_files', 'description': 'List files and directories recursively inside the project workspace.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string', 'description': "Relative directory path. Use '.' for the project root."}}, 'required': [], 'additionalProperties': False}}}
{'type': 'function', 'function': {'name': 'read_file', 'description': 'Read the UTF-8 text contents of a file inside the project workspace.', 'parameters': {'type': 'object', 'properties': {'path': {'type': 'string', 'description': 'Relative path of the file to read.'}}, 'required': ['path'], 'additionalProperties': False}}}
{'type': 'function', 'function': 


### 1. Register the Python function

The registry maps a tool name to its executable Python function.

When Gemini requests `write_file`, the registry will look up that name and invoke our `write_file()` implementation.

### 2. Define the Gemini function schema

We expose two required parameters:

- `path` (string): Relative file path inside the workspace.
- `content` (string): Complete content to write.

We also set `additionalProperties` to false so Gemini cannot supply unexpected arguments.

### 3. Validate consistency

Our existing `validate_tool_definitions()` function checks that the advertised tool definitions correspond to registered Python functions.


In [76]:

# 1. Register the Python function
project_registry.register("write_file", write_file)


# 2. Define its Gemini function schema
write_file_definition = {
    "type": "function",
    "function": {
        "name": "write_file",
        "description": (
            "Create or overwrite a UTF-8 text file inside the project workspace. "
            "Only relative paths inside the workspace are permitted."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "description": "Relative path of the file to create or overwrite."
                },
                "content": {
                    "type": "string",
                    "description": "Complete text content to write into the file."
                }
            },
            "required": ["path", "content"],
            "additionalProperties": False
        }
    }
}


# 3. Extend the existing tool definitions
project_tool_definitions.append(write_file_definition)


# 4. Validate tool definitions against the registry
validate_tool_definitions(
    project_tool_definitions,
    project_registry
)


# 5. Verify registration
print("Registered tools:")

for definition in project_tool_definitions:
    name = definition["function"]["name"]
    print(" -", name)

assert project_registry.get("write_file") is write_file

print("\nwrite_file registration successful.")
print("Total registered tools:", len(project_tool_definitions))


Validated 4 tool definitions against the registry.
Registered tools:
 - list_files
 - read_file
 - search_code
 - write_file

write_file registration successful.
Total registered tools: 4



### Objective

Create a tool that allows the agent to execute pytest tests inside the project workspace.

For example, Gemini could request:

```python
run_tests(test_path="tests")
```

The harness will execute the equivalent of:

```bash
python -m pytest -q tests
```

### Design decisions

1. **No shell execution:** Use Python's `subprocess.run()` with an argument list and `shell=False`.
2. **Workspace restriction:** Only execute tests from paths resolved inside `WORKSPACE`.
3. **Timeout:** Stop tests that run for too long. The maximum allowed timeout will be 120 seconds.
4. **Capture output:** Return both standard output and standard error so Gemini can inspect failures.
5. **Bounded observations:** Limit returned output to avoid flooding the conversation with enormous test logs.
6. **Exit status:** Include the pytest exit code. Exit code zero indicates success; a nonzero code indicates failure or another pytest error.

**Security note:** Pytest executes Python code from the project. This tool is intended for projects you trust; workspace path restrictions and timeouts do not constitute a full operating-system sandbox.


In [77]:

import subprocess
import sys

MAX_TEST_TIMEOUT_SECONDS = 120
MAX_TEST_OUTPUT_CHARS = 12000


def run_tests(
    test_path: str = "tests",
    timeout_seconds: int = 60,
) -> str:
    """
    Run pytest on a file or directory inside WORKSPACE.

    Returns the exit code and captured test output.
    """

    if not isinstance(test_path, str) or not test_path.strip():
        raise ValueError("test_path must be a non-empty string.")

    if not isinstance(timeout_seconds, int) or isinstance(timeout_seconds, bool):
        raise TypeError("timeout_seconds must be an integer.")

    if not 1 <= timeout_seconds <= MAX_TEST_TIMEOUT_SECONDS:
        raise ValueError(
            f"timeout_seconds must be between 1 and "
            f"{MAX_TEST_TIMEOUT_SECONDS}."
        )

    # Resolve and validate the requested test path.
    target = safe_resolve_path(test_path)

    if not target.exists():
        raise FileNotFoundError(f"Test path does not exist: {test_path}")

    if not (target.is_file() or target.is_dir()):
        raise ValueError("test_path must refer to a file or directory.")

    # Execute pytest without invoking a shell.
    command = [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        str(target),
    ]

    try:
        completed = subprocess.run(
            command,
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
            shell=False,
            check=False,
        )

        output = (
            f"Pytest exit code: {completed.returncode}\n\n"
            f"STDOUT:\n{completed.stdout}\n\n"
            f"STDERR:\n{completed.stderr}"
        )

    except subprocess.TimeoutExpired as exc:
        stdout = exc.stdout or ""
        stderr = exc.stderr or ""

        if isinstance(stdout, bytes):
            stdout = stdout.decode("utf-8", errors="replace")

        if isinstance(stderr, bytes):
            stderr = stderr.decode("utf-8", errors="replace")

        output = (
            f"TEST TIMEOUT: pytest exceeded {timeout_seconds} seconds.\n\n"
            f"STDOUT:\n{stdout}\n\n"
            f"STDERR:\n{stderr}"
        )

    # Keep observations within a manageable size.
    if len(output) > MAX_TEST_OUTPUT_CHARS:
        omitted = len(output) - MAX_TEST_OUTPUT_CHARS
        output = (
            output[:MAX_TEST_OUTPUT_CHARS]
            + f"\n\n[Output truncated: {omitted} characters omitted.]"
        )

    return output


print("run_tests implementation created.")
print("Workspace:", WORKSPACE)
print("Maximum timeout:", MAX_TEST_TIMEOUT_SECONDS, "seconds")
print("Maximum output:", MAX_TEST_OUTPUT_CHARS, "characters")


run_tests implementation created.
Workspace: /content/agent_workspace
Maximum timeout: 120 seconds
Maximum output: 12000 characters



We will test the tool against our existing sample project.

The tests check:

- Running the complete test directory.
- Running a specific test file.
- Rejecting a nonexistent path.
- Rejecting path traversal.
- Rejecting an invalid timeout.

We will also inspect the actual pytest output. A nonzero pytest exit code is a valid tool result, not necessarily a failure of the tool itself.


In [78]:

# Test 1: Run the complete test directory
result = run_tests("tests")

print("TEST DIRECTORY RESULT")
print(result)

assert "Pytest exit code: 0" in result, (
    "The existing project tests did not all pass."
)

print("\nTest 1 passed: Test directory execution")


# Test 2: Run one test file
result = run_tests("tests/test_model.py")

assert "Pytest exit code: 0" in result
print("Test 2 passed: Individual test-file execution")


# Test 3: Reject nonexistent paths
try:
    run_tests("tests/does_not_exist.py")
    raise AssertionError("Nonexistent path was not rejected.")
except FileNotFoundError:
    print("Test 3 passed: Nonexistent path rejected")


# Test 4: Reject path traversal
try:
    run_tests("../../outside")
    raise AssertionError("Path traversal was not rejected.")
except (ValueError, PermissionError):
    print("Test 4 passed: Path traversal rejected")


# Test 5: Reject invalid timeout
try:
    run_tests("tests", timeout_seconds=500)
    raise AssertionError("Invalid timeout was not rejected.")
except ValueError:
    print("Test 5 passed: Invalid timeout rejected")


print("\nAll run_tests implementation tests passed.")


TEST DIRECTORY RESULT
Pytest exit code: 1

STDOUT:
F                                                                        [100%]
=================================== FAILURES ===================================
______________________________ test_model_forward ______________________________

    def test_model_forward():
>       model = SimpleModel(10, 20)
                ^^^^^^^^^^^
E       NameError: name 'SimpleModel' is not defined

tests/test_model.py:2: NameError
=========================== short test summary info ============================
FAILED tests/test_model.py::test_model_forward - NameError: name 'SimpleModel...
1 failed in 0.04s


STDERR:



AssertionError: The existing project tests did not all pass.


### Why did pytest fail?

When Python executes a test file, it does not automatically know about classes defined in other modules.

Our class is defined in:

`src/model.py`

But the test uses:

`SimpleModel(...)`

without importing it.

The missing statement is:

```python
from src.model import SimpleModel
```

This is a useful first debugging example because it tests whether our agent can:

1. Interpret a Python traceback.
2. Identify the file responsible.
3. Read the relevant source and test files.
4. Propose a minimal correction.
5. Modify the file.
6. Run pytest again to verify the correction.

**Important:** We should not change the assertion or the model implementation to hide this error. The correct fix is to import the class.


In [79]:

print("PROJECT STRUCTURE")
print(list_files("."))

print("\nTEST FILE")
print(read_file("tests/test_model.py"))

print("\nMODEL FILE")
print(read_file("src/model.py"))

print("\nSRC INIT FILE EXISTS:",
      safe_resolve_path("src/__init__.py").exists())


PROJECT STRUCTURE
[DIR]  src
[DIR]  tests
[FILE] config.py
[FILE] requirements.txt
[FILE] src/dataset.py
[FILE] src/model.py
[FILE] src/trainer.py
[FILE] tests/test_model.py

TEST FILE
def test_model_forward():
    model = SimpleModel(10, 20)
    output = model.forward([1, 2, 3])
    assert output is not None


MODEL FILE
class SimpleModel:
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

    def forward(self, x):
        return x


SRC INIT FILE EXISTS: False



We will modify only `tests/test_model.py`.

Current code:

```python
def test_model_forward():
    model = SimpleModel(10, 20)
    output = model.forward([1, 2, 3])
    assert output is not None
```

Corrected code:

```python
from src.model import SimpleModel


def test_model_forward():
    model = SimpleModel(10, 20)
    output = model.forward([1, 2, 3])
    assert output is not None
```

We are not changing `SimpleModel.forward()` because the reported failure occurs before that method is even called.


In [80]:

corrected_test = '''from src.model import SimpleModel


def test_model_forward():
    model = SimpleModel(10, 20)
    output = model.forward([1, 2, 3])
    assert output is not None
'''

# Write the corrected test file
print(write_file("tests/test_model.py", corrected_test))

# Verify the updated contents
print("\nUpdated test file:")
print(read_file("tests/test_model.py"))


Successfully wrote 162 bytes to tests/test_model.py.

Updated test file:
from src.model import SimpleModel


def test_model_forward():
    model = SimpleModel(10, 20)
    output = model.forward([1, 2, 3])
    assert output is not None



In [81]:

test_result = run_tests("tests")

print(test_result)

# Check the actual pytest exit code
assert "Pytest exit code: 0" in test_result, (
    "Tests are still failing. Inspect the pytest output."
)

print("\nAll project tests passed!")


Pytest exit code: 0

STDOUT:
.                                                                        [100%]
1 passed in 0.01s


STDERR:


All project tests passed!



We will verify that `run_tests` correctly handles:

- Running an individual test file.
- Rejecting nonexistent paths.
- Rejecting path traversal.
- Rejecting invalid timeout values.
- Rejecting non-integer timeout values.

These checks validate the tool itself, independently of whether the project's tests pass.


In [82]:

# Test 1: Individual test file
result = run_tests("tests/test_model.py")
assert "Pytest exit code: 0" in result
print("Test 1 passed: Individual test execution")


# Test 2: Nonexistent path
try:
    run_tests("tests/nonexistent.py")
    raise AssertionError("Nonexistent path was not rejected.")
except FileNotFoundError:
    print("Test 2 passed: Nonexistent path rejected")


# Test 3: Path traversal
try:
    run_tests("../../outside")
    raise AssertionError("Path traversal was not rejected.")
except (ValueError, PermissionError):
    print("Test 3 passed: Path traversal rejected")


# Test 4: Excessive timeout
try:
    run_tests("tests", timeout_seconds=500)
    raise AssertionError("Excessive timeout was not rejected.")
except ValueError:
    print("Test 4 passed: Excessive timeout rejected")


# Test 5: Invalid timeout type
try:
    run_tests("tests", timeout_seconds=2.5)
    raise AssertionError("Invalid timeout type was not rejected.")
except TypeError:
    print("Test 5 passed: Invalid timeout type rejected")


print("\nAll remaining run_tests validation tests passed.")


Test 1 passed: Individual test execution
Test 2 passed: Nonexistent path rejected
Test 3 passed: Path traversal rejected
Test 4 passed: Excessive timeout rejected
Test 5 passed: Invalid timeout type rejected

All remaining run_tests validation tests passed.



### Function registration

We register the Python function in `ToolRegistry` and expose its schema to Gemini.

The schema contains:

- `test_path`: Relative path of the test file or directory. Defaults to `tests`.
- `timeout_seconds`: Maximum execution time in seconds. Defaults to 60.

We deliberately constrain the timeout to 1–120 seconds in the Python implementation. The model's schema describes the argument types, while the Python function enforces the actual limits.

After registration, we validate all five tool definitions against the registry.


In [83]:

# 1. Register the Python function
project_registry.register("run_tests", run_tests)


# 2. Define the Gemini tool schema
run_tests_definition = {
    "type": "function",
    "function": {
        "name": "run_tests",
        "description": (
            "Execute pytest tests inside the project workspace. "
            "Returns the test output and exit code. "
            "Use this tool to verify code changes."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "test_path": {
                    "type": "string",
                    "description": (
                        "Relative path of a test file or directory. "
                        "Defaults to 'tests'."
                    )
                },
                "timeout_seconds": {
                    "type": "integer",
                    "description": (
                        "Maximum test execution time in seconds, "
                        "between 1 and 120."
                    )
                }
            },
            "required": [],
            "additionalProperties": False
        }
    }
}


# 3. Add it to the existing definitions
project_tool_definitions.append(run_tests_definition)


# 4. Validate registry/schema consistency
validate_tool_definitions(
    project_tool_definitions,
    project_registry
)


# 5. Verify
print("Registered tools:")

for definition in project_tool_definitions:
    print(" -", definition["function"]["name"])

assert project_registry.get("run_tests") is run_tests

print("\nrun_tests registration successful.")
print("Total registered tools:", len(project_tool_definitions))


Validated 5 tool definitions against the registry.
Registered tools:
 - list_files
 - read_file
 - search_code
 - write_file
 - run_tests

run_tests registration successful.
Total registered tools: 5



### Inspect the existing project

We will inspect four files:

1. `src/model.py` — model implementation.
2. `src/dataset.py` — dataset implementation.
3. `src/trainer.py` — training logic.
4. `config.py` — project configuration.

We will also inspect the existing tests.

Our next debugging task will involve a genuine logical error in the project, with a corresponding test that captures the expected behaviour.

The goal is to make Gemini independently:

- Locate the relevant implementation.
- Understand how multiple files interact.
- Identify the root cause.
- Make the smallest appropriate correction.
- Run tests and verify the result.

We will preserve the existing project and its passing tests.


In [84]:

files_to_inspect = [
    "src/model.py",
    "src/dataset.py",
    "src/trainer.py",
    "tests/test_model.py",
    "config.py",
]

for path in files_to_inspect:
    print("\n" + "=" * 70)
    print(f"FILE: {path}")
    print("=" * 70)

    try:
        print(read_file(path))
    except Exception as e:
        print(f"Could not read file: {e}")



FILE: src/model.py
class SimpleModel:
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

    def forward(self, x):
        return x


FILE: src/dataset.py
class CustomDataset:
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)


FILE: src/trainer.py
def train_model(model, dataset, epochs=10):
    for epoch in range(epochs):
        for batch in dataset:
            predictions = model.forward(batch)
    return model


FILE: tests/test_model.py
from src.model import SimpleModel


def test_model_forward():
    model = SimpleModel(10, 20)
    output = model.forward([1, 2, 3])
    assert output is not None


FILE: config.py
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 10




### Challenge specification: Dataset and training pipeline

**Expected behaviour**

1. `CustomDataset` stores a collection of samples and corresponding labels.
2. `len(dataset)` returns the number of samples.
3. `dataset[i]` returns the sample and its corresponding label.
4. `train_model()` iterates through the dataset and passes each sample to `model.forward()`.
5. The model's predictions must be collected and returned.

**Intentional bug**

The trainer currently returns the model but discards all predictions.

We will also add tests that verify the expected behaviour of the dataset and trainer.

The agent will not be told which implementation is incorrect. It must inspect the project, interpret the failing tests, and decide which files need modification.

This first challenge tests multi-file debugging and verification. We will introduce more complicated bugs involving tensor dimensions, configuration mismatches, and model–dataset interfaces in later challenges.


In [85]:

# Create tests for the dataset and trainer.

dataset_tests = '''from src.dataset import CustomDataset


def test_dataset_length():
    dataset = CustomDataset(
        samples=[[1, 2], [3, 4], [5, 6]],
        labels=[0, 1, 0],
    )
    assert len(dataset) == 3


def test_dataset_indexing():
    dataset = CustomDataset(
        samples=[[1, 2], [3, 4]],
        labels=[0, 1],
    )

    assert dataset[0] == ([1, 2], 0)
    assert dataset[1] == ([3, 4], 1)
'''

trainer_tests = '''from src.model import SimpleModel
from src.dataset import CustomDataset
from src.trainer import train_model


def test_trainer_returns_predictions():
    model = SimpleModel(2, 4)

    dataset = CustomDataset(
        samples=[[1, 2], [3, 4], [5, 6]],
        labels=[0, 1, 0],
    )

    predictions = train_model(model, dataset, epochs=1)

    assert predictions is not None
    assert len(predictions) == 3
    assert predictions == [[1, 2], [3, 4], [5, 6]]
'''

print(write_file("tests/test_dataset.py", dataset_tests))
print(write_file("tests/test_trainer.py", trainer_tests))

print("\nNew tests created.")


Successfully wrote 397 bytes to tests/test_dataset.py.
Successfully wrote 461 bytes to tests/test_trainer.py.

New tests created.


In [86]:

challenge_result = run_tests("tests")

print(challenge_result)


Pytest exit code: 1

STDOUT:
FF.F                                                                     [100%]
=================================== FAILURES ===================================
_____________________________ test_dataset_length ______________________________

    def test_dataset_length():
>       dataset = CustomDataset(
            samples=[[1, 2], [3, 4], [5, 6]],
            labels=[0, 1, 0],
        )
E       TypeError: CustomDataset.__init__() got an unexpected keyword argument 'labels'

tests/test_dataset.py:5: TypeError
____________________________ test_dataset_indexing _____________________________

    def test_dataset_indexing():
>       dataset = CustomDataset(
            samples=[[1, 2], [3, 4]],
            labels=[0, 1],
        )
E       TypeError: CustomDataset.__init__() got an unexpected keyword argument 'labels'

tests/test_dataset.py:13: TypeError
_______________________ test_trainer_returns_predictions _______________________

    def test_trainer_ret


### Agent task

You are working inside a Python project.

The project's pytest suite contains failing tests. Your task is to investigate and fix the underlying implementation problems.

Requirements:

1. Inspect the project structure and relevant source files.
2. Run the test suite to understand the failures.
3. Trace each failure to its underlying cause.
4. Modify the necessary implementation files.
5. Run the complete test suite again.
6. Continue debugging until all tests pass, or explain precisely why you cannot resolve a failure.

Constraints:

- Do not modify tests to make them pass.
- Do not remove assertions.
- Do not hardcode expected test outputs.
- Preserve existing functionality.
- Make minimal, logically justified changes.
- Verify all changes using pytest.

Report the files modified, root causes, fixes, and final test results.

You have access to the project's file exploration, reading, writing, searching, and pytest tools.


In [87]:

# Start a fresh task conversation.
# Reuse the existing system instruction, but not the previous task history.

debug_messages = [
    copy.deepcopy(project_messages[0]),
    {
        "role": "user",
        "content": (
            "Investigate and fix the failing tests in this project. "
            "Follow the debugging task requirements. "
            "Do not modify the tests. "
            "Use the available tools to inspect, diagnose, edit, and verify "
            "the implementation. Continue until all tests pass or you "
            "can explain a blocker."
        ),
    },
]

# Run the autonomous agent.
debug_history, debug_response, debug_steps = run_agent(
    llm=gemini,
    messages=debug_messages,
    registry=project_registry,
    tool_definitions=project_tool_definitions,
    max_steps=15,
    verbose=True,
)

print("\n" + "=" * 65)
print("DEBUGGING EXPERIMENT SUMMARY")
print("=" * 65)

print("Tool-execution rounds:", debug_steps)
print("Finish reason:", debug_response.finish_reason)
print("Remaining tool calls:", len(debug_response.tool_calls))
print("\nAgent report:\n")
print(debug_response.content)



AGENT STEP 1

Tool: list_files
Success: True
Observation:
[DIR]  src
[DIR]  tests
[FILE] config.py
[FILE] requirements.txt
[FILE] src/dataset.py
[FILE] src/model.py
[FILE] src/trainer.py
[FILE] tests/test_dataset.py
[FILE] tests/test_model.py
[FILE] tests/test_trainer.py

Gemini finish reason: tool_calls
New tool calls: 1

AGENT STEP 2

Tool: run_tests
Success: True
Observation:
Pytest exit code: 1

STDOUT:
FF.F                                                                     [100%]
=================================== FAILURES ===================================
_____________________________ test_dataset_length ______________________________

    def test_dataset_length():
>       dataset = CustomDataset(
            samples=[[1, 2], [3, 4], [5, 6]],
            labels=[0, 1, 0],
        )
E       TypeError: CustomDataset.__init__() got an unexpected keyword argument 'labels'

tests/test_dataset.py:5: TypeError
____________________________ test_dataset_indexing ____________________


We will inspect three things:

1. The pending tool call returned at the end of Step 15.
2. The updated `src/dataset.py`.
3. The still-existing `src/trainer.py`.

This is important because we should resume from the existing conversation rather than restart the debugging task. Restarting would discard the agent's investigation history.

We will not execute the pending tool call yet.


In [88]:

print("=" * 65)
print("PENDING GEMINI ACTION")
print("=" * 65)

print("Pending tool calls:", len(debug_response.tool_calls))

for call in debug_response.tool_calls:
    print("\nTool:", call.name)
    print("Arguments:", call.arguments)

print("\n" + "=" * 65)
print("CURRENT DATASET IMPLEMENTATION")
print("=" * 65)

print(read_file("src/dataset.py"))

print("\n" + "=" * 65)
print("CURRENT TRAINER IMPLEMENTATION")
print("=" * 65)

print(read_file("src/trainer.py"))

print("\n" + "=" * 65)
print("CURRENT TEST RESULTS")
print("=" * 65)

print(run_tests("tests"))


PENDING GEMINI ACTION
Pending tool calls: 1

Tool: read_file
Arguments: {"path":"tests/test_trainer.py"}

CURRENT DATASET IMPLEMENTATION
class CustomDataset:
    def __init__(self, samples, labels=None):
        self.samples = samples
        self.labels = labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.samples[idx], self.labels[idx]
        return self.samples[idx]


CURRENT TRAINER IMPLEMENTATION
def train_model(model, dataset, epochs=10):
    for epoch in range(epochs):
        for batch in dataset:
            predictions = model.forward(batch)
    return model


CURRENT TEST RESULTS
Pytest exit code: 1

STDOUT:
...F                                                                     [100%]
=================================== FAILURES ===================================
_______________________ test_trainer_returns_predictions _______________________

    def test_trainer_ret


### Resuming an interrupted agent

Our previous `run_agent()` stopped after 15 tool-execution rounds. It returned:

- `debug_history`: The complete conversation, including all tool observations.
- `debug_response`: Gemini's latest response, containing one pending tool call.

We can resume without losing context.

We pass:
- The existing conversation as `messages`.
- The pending response as `initial_response`.
- An additional budget of 10 tool-execution rounds.

The pending tool call will be executed first. Gemini will then continue from where it stopped.

This is not a new debugging session. It is a continuation of the same session.


In [89]:

# Resume the existing debugging session.
resumed_history, resumed_response, resumed_steps = run_agent(
    llm=gemini,
    messages=debug_history,
    registry=project_registry,
    tool_definitions=project_tool_definitions,
    initial_response=debug_response,
    max_steps=10,
    verbose=True,
)

print("\n" + "=" * 65)
print("RESUMED DEBUGGING SUMMARY")
print("=" * 65)

print("Additional tool-execution rounds:", resumed_steps)
print("Finish reason:", resumed_response.finish_reason)
print("Remaining tool calls:", len(resumed_response.tool_calls))

print("\nAgent report:")
print(resumed_response.content)



AGENT STEP 1

Tool: read_file
Success: True
Observation:
from src.model import SimpleModel
from src.dataset import CustomDataset
from src.trainer import train_model


def test_trainer_returns_predictions():
    model = SimpleModel(2, 4)

    dataset = CustomDataset(
        samples=[[1, 2], [3, 4], [5, 6]],
        labels=[0, 1, 0],
    )

    predictions = train_model(model, dataset, epochs=1)

    assert predictions is not None
    assert len(predictions) == 3
    assert predictions == [[1, 2], [3, 4], [5, 6]]


Gemini finish reason: tool_calls
New tool calls: 1

AGENT STEP 2

Tool: write_file
Success: True
Observation:
Successfully wrote 261 bytes to src/trainer.py.

Gemini finish reason: tool_calls
New tool calls: 1

AGENT STEP 3

Tool: run_tests
Success: True
Observation:
Pytest exit code: 0

STDOUT:
....                                                                     [100%]
4 passed in 0.02s


STDERR:


Gemini finish reason: tool_calls
New tool calls: 1

AGENT STEP 4

Tool: 

In [90]:

print("=" * 65)
print("FINAL PROJECT FILES")
print("=" * 65)

for path in ["src/dataset.py", "src/trainer.py"]:
    print(f"\n--- {path} ---")
    print(read_file(path))

print("\n" + "=" * 65)
print("INDEPENDENT PYTEST VERIFICATION")
print("=" * 65)

final_test_result = run_tests("tests")
print(final_test_result)


FINAL PROJECT FILES

--- src/dataset.py ---
class CustomDataset:
    def __init__(self, samples, labels=None):
        self.samples = samples
        self.labels = labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.labels is not None:
            return self.samples[idx], self.labels[idx]
        return self.samples[idx]


--- src/trainer.py ---
def train_model(model, dataset, epochs=10):
    predictions = []
    for epoch in range(epochs):
        predictions = []
        for sample, label in dataset:
            pred = model.forward(sample)
            predictions.append(pred)
    return predictions


INDEPENDENT PYTEST VERIFICATION
Pytest exit code: 0

STDOUT:
....                                                                     [100%]
4 passed in 0.01s


STDERR:




### 1. Agent execution states

We will support two explicit states initially:

| State | Meaning |
|---|---|
| `completed` | Gemini returned a response without tool calls |
| `max_steps_reached` | The execution budget was exhausted while tool calls remained |

Our result object will contain:

- `status`: Current execution state.
- `history`: Conversation history.
- `response`: Latest Gemini response.
- `steps`: Number of tool-execution rounds.
- `pending_tool_calls`: Calls that have not yet been executed.

### 2. Why preserve pending tool calls separately?

Suppose the agent reaches its limit after requesting `read_file`.

We must not discard that request or execute it twice.

Instead, we preserve the response and resume using it as `initial_response` in the next execution.

This gives us a clean separation between the agent's state and the execution budget.

### 3. Another important correction

When Gemini finishes, we will append its final assistant message to the returned history. This makes the completed history a complete conversation rather than just a sequence of tool interactions.


In [91]:

from dataclasses import dataclass
from typing import List, Dict, Any


@dataclass
class AgentRunResult:
    status: str
    history: List[Dict[str, Any]]
    response: AssistantTurn
    steps: int
    pending_tool_calls: List[ToolCall]


def get_original_assistant_message(response):
    """
    Preserve the original Gemini response, including provider metadata.
    """

    if response.raw is not None:
        return (
            response.raw.choices[0]
            .message.model_dump(exclude_none=True)
        )

    # Fallback for providers or scripted responses without raw SDK objects.
    return response.to_message()


def run_agent_v2(
    llm,
    messages,
    registry,
    tool_definitions,
    initial_response=None,
    max_steps=10,
    verbose=True,
):
    """
    Resumable ReAct agent loop with explicit execution states.
    """

    if max_steps < 1:
        raise ValueError("max_steps must be at least 1.")

    history = copy.deepcopy(messages)
    validate_messages(history)

    response = initial_response

    if response is None:
        response = llm.chat(
            messages=history,
            tools=tool_definitions,
        )

    steps = 0

    while response.tool_calls:

        # Preserve the pending response without executing it.
        if steps >= max_steps:
            status = "max_steps_reached"
            break

        if verbose:
            print("\n" + "=" * 65)
            print(f"AGENT STEP {steps + 1}")
            print("=" * 65)

        # Preserve original assistant message and Gemini metadata.
        history.append(get_original_assistant_message(response))

        # Execute requested tools.
        results = registry.execute_many(response.tool_calls)

        # Append observations.
        for result in results:
            history.append(result.to_message())

            if verbose:
                print(f"\nTool: {result.name}")
                print(f"Success: {result.success}")
                print("Observation:")
                print(result.content)

        validate_messages(history)

        steps += 1

        # Continue the reasoning cycle.
        response = llm.chat(
            messages=history,
            tools=tool_definitions,
        )

    else:
        # No pending tool calls: the agent has completed its task.
        status = "completed"

        # Include the final assistant message in the history.
        history.append(get_original_assistant_message(response))

    result = AgentRunResult(
        status=status,
        history=history,
        response=response,
        steps=steps,
        pending_tool_calls=copy.deepcopy(response.tool_calls),
    )

    if verbose:
        print("\n" + "=" * 65)
        print("AGENT EXECUTION SUMMARY")
        print("=" * 65)

        print("Status:", result.status)
        print("Tool-execution rounds:", result.steps)
        print("Pending tool calls:", len(result.pending_tool_calls))
        print("Conversation messages:", len(result.history))

        if result.response.content:
            print("\nFinal response:")
            print(result.response.content)

    return result



We will test this using `ScriptedLLM`, our deterministic fake model.

Instead of spending Gemini API calls, we will prescribe the following responses:

1. First response: request `list_files`.
2. Second response: request `list_files` again.
3. Third response: return a final textual answer.

We will initially allow only one tool-execution round.

Expected behaviour:

- First execution stops with `max_steps_reached`.
- One tool call remains pending.
- We resume using the saved response and history.
- Second execution completes successfully.
- The final history includes the assistant's final answer.

This tests the control-flow mechanism independently of Gemini.


In [92]:

# Prescribe two tool calls followed by a final response.

first_call = AssistantTurn(
    content=None,
    tool_calls=[
        ToolCall(
            id="resume_test_1",
            name="list_files",
            arguments='{"path":"."}',
        )
    ],
    finish_reason="tool_calls",
)

second_call = AssistantTurn(
    content=None,
    tool_calls=[
        ToolCall(
            id="resume_test_2",
            name="list_files",
            arguments='{"path":"src"}',
        )
    ],
    finish_reason="tool_calls",
)

final_turn = AssistantTurn(
    content="The project contains source and test directories.",
    tool_calls=[],
    finish_reason="stop",
)

scripted_llm = ScriptedLLM([
    second_call,
    final_turn,
])

test_messages = [
    {"role": "system", "content": "You are a code assistant."},
    {"role": "user", "content": "Inspect the project."},
]


# First execution: only one tool round is permitted.
first_run = run_agent_v2(
    llm=scripted_llm,
    messages=test_messages,
    registry=project_registry,
    tool_definitions=project_tool_definitions,
    initial_response=first_call,
    max_steps=1,
    verbose=False,
)

assert first_run.status == "max_steps_reached"
assert first_run.steps == 1
assert len(first_run.pending_tool_calls) == 1

print("Test 1 passed: Maximum-step interruption")
print("Status:", first_run.status)
print("Pending tool calls:", len(first_run.pending_tool_calls))


# Resume from the saved state.
second_run = run_agent_v2(
    llm=scripted_llm,
    messages=first_run.history,
    registry=project_registry,
    tool_definitions=project_tool_definitions,
    initial_response=first_run.response,
    max_steps=1,
    verbose=False,
)

assert second_run.status == "completed"
assert second_run.steps == 1
assert len(second_run.pending_tool_calls) == 0
assert second_run.history[-1]["role"] == "assistant"
assert second_run.history[-1]["content"] == (
    "The project contains source and test directories."
)

print("\nTest 2 passed: Agent successfully resumed")
print("Status:", second_run.status)
print("Final response:", second_run.response.content)


# Verify that the final conversation is complete.
validate_messages(second_run.history)

print("\nTest 3 passed: Completed conversation validation")
print("Total messages:", len(second_run.history))


Test 1 passed: Maximum-step interruption
Status: max_steps_reached
Pending tool calls: 1

Test 2 passed: Agent successfully resumed
Status: completed
Final response: The project contains source and test directories.

Test 3 passed: Completed conversation validation
Total messages: 7



### Objective

Before implementing context compaction, we need to understand how quickly our context grows.

Gemini already returns token usage in each `AssistantTurn`.

We will inspect the latest completed debugging experiment and calculate:

- Number of conversation messages.
- Total characters in the conversation.
- Number of assistant and tool messages.
- Latest prompt and completion token counts.

This gives us a baseline for measuring the effect of memory management later.

Note that prompt token counts measure the request sent to Gemini, not the total size of our locally stored history.


In [93]:

# Inspect the most recent completed debugging run.

print("CONVERSATION STATISTICS")
print("=" * 55)

print("Total messages:", len(resumed_history))

role_counts = {}

for message in resumed_history:
    role = message.get("role", "unknown")
    role_counts[role] = role_counts.get(role, 0) + 1

print("Messages by role:", role_counts)

total_characters = sum(
    len(str(message.get("content", "")))
    + len(str(message.get("tool_calls", "")))
    for message in resumed_history
)

print("Total conversation characters:", total_characters)

print("\nLATEST GEMINI TOKEN USAGE")
print("=" * 55)

print(resumed_response.usage)

print("\nWORKING MEMORY")
print("=" * 55)

print("Not implemented yet.")


CONVERSATION STATISTICS
Total messages: 40
Messages by role: {'system': 1, 'user': 1, 'assistant': 19, 'tool': 19}
Total conversation characters: 13520

LATEST GEMINI TOKEN USAGE
{'prompt_tokens': 3146, 'completion_tokens': 172, 'total_tokens': 3318}

WORKING MEMORY
Not implemented yet.



### Objective

Implement a `UsageTracker` that maintains:

1. Number of Gemini requests.
2. Cumulative prompt tokens.
3. Cumulative completion tokens.
4. Cumulative total tokens.
5. Most recent prompt token count.
6. Largest prompt encountered.

Why track these separately?

- `prompt_tokens`: Measures the context sent to Gemini.
- `completion_tokens`: Measures the model's generated output.
- `total_tokens`: Measures combined usage for that request.
- `largest_prompt`: Helps us understand peak context consumption.

We will use Gemini's actual API usage statistics rather than estimating tokens from characters.

Later, the context manager will use the latest prompt size and a configured threshold to decide when compaction is needed.


In [94]:

from dataclasses import dataclass, field


@dataclass
class UsageTracker:
    requests: int = 0
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0

    latest_prompt_tokens: int = 0
    largest_prompt_tokens: int = 0

    def record(self, usage):
        """
        Record token usage from one AssistantTurn.

        Expected usage:
        {
            "prompt_tokens": ...,
            "completion_tokens": ...,
            "total_tokens": ...
        }
        """

        if not usage:
            return

        prompt = int(usage.get("prompt_tokens", 0) or 0)
        completion = int(usage.get("completion_tokens", 0) or 0)
        total = int(
            usage.get("total_tokens", prompt + completion) or 0
        )

        self.requests += 1

        self.prompt_tokens += prompt
        self.completion_tokens += completion
        self.total_tokens += total

        self.latest_prompt_tokens = prompt
        self.largest_prompt_tokens = max(
            self.largest_prompt_tokens,
            prompt
        )

    def summary(self):
        return {
            "requests": self.requests,
            "prompt_tokens": self.prompt_tokens,
            "completion_tokens": self.completion_tokens,
            "total_tokens": self.total_tokens,
            "latest_prompt_tokens": self.latest_prompt_tokens,
            "largest_prompt_tokens": self.largest_prompt_tokens,
        }

    def display(self):
        print("=" * 50)
        print("TOKEN USAGE SUMMARY")
        print("=" * 50)

        for key, value in self.summary().items():
            print(f"{key:25s}: {value:,}")



We will first test the tracker with synthetic requests to verify that aggregation works correctly.

Then we will record the actual usage from our latest Gemini response.

The tracker is deliberately independent of the Gemini client. This means we can use the same component with another model provider or our `ScriptedLLM`.


In [95]:

# Test with synthetic requests.
usage_tracker = UsageTracker()

usage_tracker.record({
    "prompt_tokens": 100,
    "completion_tokens": 50,
    "total_tokens": 150,
})

usage_tracker.record({
    "prompt_tokens": 200,
    "completion_tokens": 80,
    "total_tokens": 280,
})

assert usage_tracker.requests == 2
assert usage_tracker.prompt_tokens == 300
assert usage_tracker.completion_tokens == 130
assert usage_tracker.total_tokens == 430
assert usage_tracker.latest_prompt_tokens == 200
assert usage_tracker.largest_prompt_tokens == 200

print("Synthetic aggregation tests passed.")


# Test with actual Gemini usage from our completed run.
actual_tracker = UsageTracker()

actual_tracker.record(resumed_response.usage)

print("\nActual latest Gemini request:")
actual_tracker.display()


Synthetic aggregation tests passed.

Actual latest Gemini request:
TOKEN USAGE SUMMARY
requests                 : 1
prompt_tokens            : 3,146
completion_tokens        : 172
total_tokens             : 3,318
latest_prompt_tokens     : 3,146
largest_prompt_tokens    : 3,146



## Step 25.2 — Automatic token tracking

The agent makes multiple LLM requests during a ReAct loop:

1. Gemini receives the conversation and decides which tool to call.
2. The tool executes and returns an observation.
3. Gemini receives the updated conversation.
4. This repeats until the task is completed.

We want to record token usage after **every API request**, not just the final request.

For resumability, the tracker is passed into the agent and retained across runs. When resuming, we must avoid counting the already-recorded initial response twice.

**Example:** If a run makes 5 requests and is interrupted, then resumes for 3 more requests, the tracker should report 8 requests—not 4 or 9.


In [96]:

def run_agent_v3(
    llm,
    messages,
    registry,
    tool_definitions,
    usage_tracker,
    initial_response=None,
    initial_response_already_counted=False,
    max_steps=10,
    verbose=True,
):
    history = copy.deepcopy(messages)
    validate_messages(history)

    response = initial_response

    # Obtain the first response if one was not supplied.
    if response is None:
        response = llm.chat(
            messages=history,
            tools=tool_definitions,
        )
        usage_tracker.record(response.usage)

    elif not initial_response_already_counted:
        usage_tracker.record(response.usage)

    steps = 0

    while response.tool_calls:

        if steps >= max_steps:
            status = "max_steps_reached"
            break

        # Preserve the original Gemini assistant message,
        # including its thought signature.
        history.append(get_original_assistant_message(response))

        results = registry.execute_many(response.tool_calls)

        for result in results:
            history.append(result.to_message())

        validate_messages(history)

        steps += 1

        # Request the next assistant response.
        response = llm.chat(
            messages=history,
            tools=tool_definitions,
        )

        usage_tracker.record(response.usage)

        if verbose:
            print(f"Completed agent step {steps}")

    else:
        status = "completed"
        history.append(get_original_assistant_message(response))

    return AgentRunResult(
        status=status,
        history=history,
        response=response,
        steps=steps,
        pending_tool_calls=copy.deepcopy(response.tool_calls),
    )


In [99]:

# Create two synthetic responses with usage information.


response_1 = AssistantTurn(
    content=None,
    tool_calls=[
        ToolCall(
            id="call_1",
            name="list_files",
            arguments='{"path": "."}',
        )
    ],
    finish_reason="tool_calls",
    usage={
        "prompt_tokens": 100,
        "completion_tokens": 20,
        "total_tokens": 120,
    },
)


response_2 = AssistantTurn(
    content="The project files have been listed.",
    tool_calls=[],
    finish_reason="stop",
    usage={
        "prompt_tokens": 150,
        "completion_tokens": 30,
        "total_tokens": 180,
    },
)

synthetic_llm = ScriptedLLM([response_2])


synthetic_tracker = UsageTracker()

synthetic_result = run_agent_v3(
    llm=synthetic_llm,
    messages=[
        {
            "role": "system",
            "content": "You are a code-fixing agent. Use tools when needed."
        },
        {
            "role": "user",
            "content": "List the project files."
        }
    ],
    registry=project_registry,
    tool_definitions=project_tool_definitions,
    usage_tracker=synthetic_tracker,
    initial_response=response_1,
    max_steps=5,
    verbose=False,
)

synthetic_tracker.display()

assert synthetic_tracker.requests == 2
assert synthetic_tracker.prompt_tokens == 250
assert synthetic_tracker.completion_tokens == 50
assert synthetic_tracker.total_tokens == 300

print("\nAll token-tracking integration tests passed.")



TOKEN USAGE SUMMARY
requests                 : 2
prompt_tokens            : 250
completion_tokens        : 50
total_tokens             : 300
latest_prompt_tokens     : 150
largest_prompt_tokens    : 150

All token-tracking integration tests passed.



## Why different tools need different output limits

A tool's output has a different level of importance depending on its purpose.

| Tool | Output policy |
|---|---|
| `list_files` | Moderate limit; file paths are relatively compact |
| `read_file` | Larger limit; source code must remain readable |
| `search_code` | Moderate limit; preserve relevant matches |
| `run_tests` | Larger limit; preserve failure information |
| `write_file` | Smaller limit; confirmation is usually sufficient |

We will use character limits as a simple first approximation of token limits.

For example, suppose a tool produces 20,000 characters but its limit is 8,000. Instead of discarding everything after the first 8,000 characters, we retain both the beginning and the end.

This is useful because:
- The beginning often contains the function definitions or initial context.
- The end may contain the actual error, traceback, or test summary.

The full output will not be retained in the conversation message when it is truncated. Later, we will introduce working memory and persistent storage to manage information that needs to survive beyond the active context.


In [100]:

# Character limits for different tools.
# These are initial engineering choices, not universal limits.

TOOL_OUTPUT_LIMITS = {
    "list_files": 6000,
    "read_file": 12000,
    "search_code": 8000,
    "run_tests": 12000,
    "write_file": 2000,
}

DEFAULT_TOOL_OUTPUT_LIMIT = 8000


def truncate_tool_output(content, max_chars):
    """
    Limit tool output while preserving its beginning and end.
    """

    if not isinstance(content, str):
        content = str(content)

    if len(content) <= max_chars:
        return content

    marker = "\n\n... [OUTPUT TRUNCATED] ...\n\n"

    available_chars = max_chars - len(marker)

    if available_chars <= 0:
        return content[:max_chars]

    head_chars = available_chars // 2
    tail_chars = available_chars - head_chars

    return (
        content[:head_chars]
        + marker
        + content[-tail_chars:]
    )


def manage_tool_result(result):
    """
    Apply a tool-specific output limit to a ToolResult.

    Returns a new ToolResult, leaving the original result unchanged.
    """

    max_chars = TOOL_OUTPUT_LIMITS.get(
        result.name,
        DEFAULT_TOOL_OUTPUT_LIMIT,
    )

    original_content = result.content

    managed_content = truncate_tool_output(
        original_content,
        max_chars,
    )

    if len(str(original_content)) > max_chars:
        managed_content += (
            f"\n\n[Original output: {len(str(original_content)):,} "
            f"characters; displayed output: {len(managed_content):,} characters.]"
        )

    return ToolResult(
        tool_call_id=result.tool_call_id,
        name=result.name,
        content=managed_content,
        success=result.success,
    )


print("Tool-output management utilities defined.")


Tool-output management utilities defined.


In [101]:

# Test 1: Small output should remain unchanged.

small_output = "All 4 tests passed."

assert truncate_tool_output(small_output, 100) == small_output


# Test 2: Large output should be truncated.

large_output = "\n".join(
    [f"Line number {i}: example source code" for i in range(1000)]
)

managed_output = truncate_tool_output(
    large_output,
    max_chars=1000,
)

assert len(managed_output) <= 1000
assert "[OUTPUT TRUNCATED]" in managed_output


# Test 3: Beginning and end should both be preserved.

assert "Line number 0" in managed_output
assert "Line number 999" in managed_output


# Test 4: Tool-specific limits should be applied.

test_result = ToolResult(
    tool_call_id="test_call",
    name="read_file",
    content=large_output,
    success=True,
)

managed_result = manage_tool_result(test_result)

assert len(managed_result.content) > 0
assert "[OUTPUT TRUNCATED]" in managed_result.content
assert managed_result.tool_call_id == "test_call"
assert managed_result.success is True

print("All tool-output management tests passed.")
print("Original output characters:", len(large_output))
print("Managed output characters:", len(managed_result.content))


All tool-output management tests passed.
Original output characters: 36889
Managed output characters: 12076



## Step 26.2 — Integrating output management

The current execution flow is:

1. Gemini requests a tool.
2. ToolRegistry executes the tool.
3. ToolResult is generated.
4. ToolResult is appended to conversation history.
5. Gemini receives the conversation.

We will insert output management between steps 3 and 4.

New flow:

Tool execution → ToolResult → Output management → Conversation history

This ensures that oversized outputs are controlled before they enter the conversation.

We will create `run_agent_v4()` rather than modifying `run_agent_v3()`. This preserves our previous implementation for comparison.

All other behavior must remain unchanged:
- Gemini thought signatures must be preserved.
- Token tracking must continue.
- Resumable execution must continue.
- Pending tool calls must remain available.


In [102]:

def run_agent_v4(
    llm,
    messages,
    registry,
    tool_definitions,
    usage_tracker,
    initial_response=None,
    initial_response_already_counted=False,
    max_steps=10,
    verbose=True,
):
    history = copy.deepcopy(messages)
    validate_messages(history)

    response = initial_response

    if response is None:
        response = llm.chat(
            messages=history,
            tools=tool_definitions,
        )
        usage_tracker.record(response.usage)

    elif not initial_response_already_counted:
        usage_tracker.record(response.usage)

    steps = 0

    while response.tool_calls:

        if steps >= max_steps:
            status = "max_steps_reached"
            break

        # Preserve the original assistant message.
        history.append(get_original_assistant_message(response))

        # Execute requested tools.
        results = registry.execute_many(response.tool_calls)

        # Apply output management before adding observations.
        managed_results = [
            manage_tool_result(result)
            for result in results
        ]

        for result in managed_results:
            history.append(result.to_message())

        validate_messages(history)

        steps += 1

        # Ask Gemini to continue.
        response = llm.chat(
            messages=history,
            tools=tool_definitions,
        )

        usage_tracker.record(response.usage)

        if verbose:
            print(f"Completed agent step {steps}")

    else:
        status = "completed"
        history.append(get_original_assistant_message(response))

    return AgentRunResult(
        status=status,
        history=history,
        response=response,
        steps=steps,
        pending_tool_calls=copy.deepcopy(response.tool_calls),
    )


In [105]:

# 1. Register the tool using your actual registry interface.

def large_output_tool():
    return "X" * 20000


large_output_registry = ToolRegistry()

large_output_registry.register(
    "large_output",
    large_output_tool,
)


# 2. Define the Gemini tool schema separately.

large_output_definitions = [
    {
        "type": "function",
        "function": {
            "name": "large_output",
            "description": "Returns a large string.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
                "additionalProperties": False,
            },
        },
    }
]


# 3. First response requests the tool.

large_response_1 = AssistantTurn(
    content=None,
    tool_calls=[
        ToolCall(
            id="large_call_1",
            name="large_output",
            arguments="{}",
        )
    ],
    finish_reason="tool_calls",
    usage={
        "prompt_tokens": 100,
        "completion_tokens": 20,
        "total_tokens": 120,
    },
)


# 4. Second response finishes the task.

large_response_2 = AssistantTurn(
    content="The large output was successfully managed.",
    tool_calls=[],
    finish_reason="stop",
    usage={
        "prompt_tokens": 150,
        "completion_tokens": 30,
        "total_tokens": 180,
    },
)


# 5. Execute the agent.

large_test_llm = ScriptedLLM([large_response_2])
large_test_tracker = UsageTracker()

large_test_result = run_agent_v4(
    llm=large_test_llm,
    messages=[
        {"role": "system", "content": "You are a code-fixing agent."},
        {"role": "user", "content": "Run the large output tool."},
    ],
    registry=large_output_registry,
    tool_definitions=large_output_definitions,
    usage_tracker=large_test_tracker,
    initial_response=large_response_1,
    max_steps=5,
    verbose=False,
)


# 6. Verify that the tool output was truncated.

tool_observation = next(
    message
    for message in large_test_result.history
    if message.get("role") == "tool"
)

print("Managed output length:", len(tool_observation["content"]))
print(
    "Contains truncation marker:",
    "[OUTPUT TRUNCATED]" in tool_observation["content"],
)

assert len(tool_observation["content"]) < 20000
assert "[OUTPUT TRUNCATED]" in tool_observation["content"]
assert large_test_tracker.requests == 2

print("\nAll agent integration tests passed.")


Managed output length: 8075
Contains truncation marker: True

All agent integration tests passed.



## Step 27.1 — Design the Working Memory structure

Working Memory represents the current state of the debugging task.

It is separate from conversation history.

We will maintain the following information:

1. **Task:** What the user asked the agent to accomplish.
2. **Project structure:** Important files and directories discovered.
3. **Inspected files:** Files already read or analyzed.
4. **Identified issues:** Bugs and their suspected causes.
5. **Modified files:** Files changed by the agent and why.
6. **Test status:** Latest test results.
7. **Decisions:** Important decisions made during debugging.
8. **Pending tasks:** What the agent still needs to investigate or fix.

Example:

```text
Task:
Fix the dataset indexing and training functionality.

Inspected files:
- src/dataset.py
- src/trainer.py
- tests/test_dataset.py

Identified issues:
- Dataset did not support labels.
- Dataset indexing was not implemented.

Modified files:
- src/dataset.py
- src/trainer.py

Test status:
4 tests passed.

Pending tasks:
- None.
```

Unlike conversation history, this memory is updated as the agent progresses.

For now, we will implement the data structure and basic update operations. Later, we will connect it to the agent loop and use it during context compaction.


In [106]:

from dataclasses import dataclass, field
from typing import List


@dataclass
class WorkingMemory:

    task: str = ""

    project_structure: List[str] = field(default_factory=list)

    inspected_files: List[str] = field(default_factory=list)

    identified_issues: List[str] = field(default_factory=list)

    modified_files: List[str] = field(default_factory=list)

    decisions: List[str] = field(default_factory=list)

    test_status: str = "Not run"

    pending_tasks: List[str] = field(default_factory=list)


    def add_unique(self, collection, item):
        """Add an item without creating duplicates."""
        if item not in collection:
            collection.append(item)


    def add_inspected_file(self, path):
        self.add_unique(self.inspected_files, path)


    def add_issue(self, issue):
        self.add_unique(self.identified_issues, issue)


    def add_modified_file(self, path):
        self.add_unique(self.modified_files, path)


    def add_decision(self, decision):
        self.add_unique(self.decisions, decision)


    def add_pending_task(self, task):
        self.add_unique(self.pending_tasks, task)


    def complete_task(self, task):
        if task in self.pending_tasks:
            self.pending_tasks.remove(task)


    def set_test_status(self, status):
        self.test_status = status


    def set_project_structure(self, paths):
        self.project_structure = list(dict.fromkeys(paths))


    def to_dict(self):
        return {
            "task": self.task,
            "project_structure": self.project_structure.copy(),
            "inspected_files": self.inspected_files.copy(),
            "identified_issues": self.identified_issues.copy(),
            "modified_files": self.modified_files.copy(),
            "decisions": self.decisions.copy(),
            "test_status": self.test_status,
            "pending_tasks": self.pending_tasks.copy(),
        }


    def display(self):
        print("=" * 60)
        print("AGENT WORKING MEMORY")
        print("=" * 60)

        for section, value in self.to_dict().items():
            print(f"\n{section.upper().replace('_', ' ')}:")

            if isinstance(value, list):
                if value:
                    for item in value:
                        print(f"  - {item}")
                else:
                    print("  - None")

            else:
                print(f"  {value}")


In [107]:

memory = WorkingMemory()

memory.task = "Fix dataset indexing and training functionality."

memory.set_project_structure([
    "src/model.py",
    "src/dataset.py",
    "src/trainer.py",
    "tests/test_dataset.py",
    "tests/test_trainer.py",
])

memory.add_inspected_file("src/dataset.py")
memory.add_inspected_file("src/trainer.py")

memory.add_issue(
    "CustomDataset did not support labels or indexing."
)

memory.add_modified_file("src/dataset.py")
memory.add_modified_file("src/trainer.py")

memory.add_decision(
    "Preserve compatibility with datasets that do not have labels."
)

memory.set_test_status("4 tests passed.")

memory.add_pending_task("Review trainer behavior for zero epochs.")

# Test duplicate prevention.
memory.add_inspected_file("src/dataset.py")

assert len(memory.inspected_files) == 2

# Test task completion.
memory.complete_task("Review trainer behavior for zero epochs.")

assert len(memory.pending_tasks) == 0

memory.display()

print("\nAll WorkingMemory tests passed.")


AGENT WORKING MEMORY

TASK:
  Fix dataset indexing and training functionality.

PROJECT STRUCTURE:
  - src/model.py
  - src/dataset.py
  - src/trainer.py
  - tests/test_dataset.py
  - tests/test_trainer.py

INSPECTED FILES:
  - src/dataset.py
  - src/trainer.py

IDENTIFIED ISSUES:
  - CustomDataset did not support labels or indexing.

MODIFIED FILES:
  - src/dataset.py
  - src/trainer.py

DECISIONS:
  - Preserve compatibility with datasets that do not have labels.

TEST STATUS:
  4 tests passed.

PENDING TASKS:
  - None

All WorkingMemory tests passed.



## Serialization and deserialization

Serialization converts our WorkingMemory object into a JSON-compatible dictionary.

For example:

WorkingMemory object → Dictionary → JSON file

Deserialization performs the reverse operation:

JSON file → Dictionary → WorkingMemory object

We will implement three methods:

- `to_dict()` — already implemented.
- `save(path)` — save memory to a JSON file.
- `load(path)` — restore memory from a JSON file.

We will also implement `to_prompt()`.

This last method is important: it converts the memory into a concise text representation that can be included in Gemini's system or user context.

We are not saving the entire conversation here. That will be handled separately in our persistent session storage stage.


In [108]:

import json
from pathlib import Path


def working_memory_save(self, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(
            self.to_dict(),
            f,
            indent=4,
            ensure_ascii=False,
        )


def working_memory_load(path):
    path = Path(path)

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    return WorkingMemory(**data)


def working_memory_to_prompt(self):
    """
    Convert working memory into compact text for the LLM.
    """

    sections = [
        f"CURRENT TASK:\n{self.task}",
        f"PROJECT STRUCTURE:\n{self.project_structure}",
        f"INSPECTED FILES:\n{self.inspected_files}",
        f"IDENTIFIED ISSUES:\n{self.identified_issues}",
        f"MODIFIED FILES:\n{self.modified_files}",
        f"DECISIONS:\n{self.decisions}",
        f"TEST STATUS:\n{self.test_status}",
        f"PENDING TASKS:\n{self.pending_tasks}",
    ]

    return "\n\n".join(sections)


# Attach methods to the existing class.
WorkingMemory.save = working_memory_save
WorkingMemory.load = staticmethod(working_memory_load)
WorkingMemory.to_prompt = working_memory_to_prompt

print("WorkingMemory serialization methods added.")


WorkingMemory serialization methods added.


In [109]:

MEMORY_PATH = "/content/agent_workspace/agent_memory.json"

# Save the existing memory.
memory.save(MEMORY_PATH)

print("Memory saved:", MEMORY_PATH)


# Restore it as a new object.
restored_memory = WorkingMemory.load(MEMORY_PATH)


# Verify that all fields were preserved.
assert restored_memory.to_dict() == memory.to_dict()

print("Memory successfully restored.")
print("\nRestored memory:")

restored_memory.display()


# Verify LLM prompt representation.
memory_prompt = restored_memory.to_prompt()

assert "Fix dataset indexing" in memory_prompt
assert "src/dataset.py" in memory_prompt
assert "4 tests passed" in memory_prompt

print("\nMemory prompt generated.")
print("Prompt characters:", len(memory_prompt))


Memory saved: /content/agent_workspace/agent_memory.json
Memory successfully restored.

Restored memory:
AGENT WORKING MEMORY

TASK:
  Fix dataset indexing and training functionality.

PROJECT STRUCTURE:
  - src/model.py
  - src/dataset.py
  - src/trainer.py
  - tests/test_dataset.py
  - tests/test_trainer.py

INSPECTED FILES:
  - src/dataset.py
  - src/trainer.py

IDENTIFIED ISSUES:
  - CustomDataset did not support labels or indexing.

MODIFIED FILES:
  - src/dataset.py
  - src/trainer.py

DECISIONS:
  - Preserve compatibility with datasets that do not have labels.

TEST STATUS:
  4 tests passed.

PENDING TASKS:
  - None

Memory prompt generated.
Prompt characters: 495



## Step 27.3 — Automatic memory updates

We will create a function called `update_working_memory()`.

It receives:
- The tool name.
- The tool arguments.
- The tool result.
- The current WorkingMemory object.

It then updates the relevant memory fields.

Examples:

1. `read_file({"path": "src/model.py"})`
   → Add `src/model.py` to inspected files.

2. `write_file({"path": "src/model.py", ...})`
   → Add `src/model.py` to modified files, only if successful.

3. `run_tests({"test_path": "tests"})`
   → Update the test status using the actual output.

For this first version, we will track verified execution events. Extracting deeper findings from source code and test failures can be added later.


In [110]:

import json


def update_working_memory(memory, result, arguments):
    """
    Update working memory from a completed tool execution.

    result: ToolResult
    arguments: Parsed tool arguments dictionary
    """

    if not result.success:
        return

    path = arguments.get("path")

    if result.name == "list_files":
        try:
            data = json.loads(result.content)

            if isinstance(data, list):
                memory.set_project_structure(data)

            elif isinstance(data, dict):
                files = data.get("files", [])
                if isinstance(files, list):
                    memory.set_project_structure(files)

        except (json.JSONDecodeError, TypeError):
            pass

    elif result.name == "read_file":
        if path:
            memory.add_inspected_file(path)

    elif result.name == "write_file":
        if path:
            memory.add_modified_file(path)

    elif result.name == "run_tests":
        memory.set_test_status(result.content[:1500])


print("Working memory update function defined.")


Working memory update function defined.


In [111]:

test_memory = WorkingMemory(
    task="Debug the sample Python project."
)

# Simulate successful file inspection.
read_result = ToolResult(
    tool_call_id="call_1",
    name="read_file",
    content="class SimpleModel:\n    pass",
    success=True,
)

update_working_memory(
    test_memory,
    read_result,
    {"path": "src/model.py"},
)


# Simulate successful file modification.
write_result = ToolResult(
    tool_call_id="call_2",
    name="write_file",
    content="File written successfully.",
    success=True,
)

update_working_memory(
    test_memory,
    write_result,
    {"path": "src/model.py"},
)


# Simulate test execution.
test_result = ToolResult(
    tool_call_id="call_3",
    name="run_tests",
    content="4 passed in 0.12s",
    success=True,
)

update_working_memory(
    test_memory,
    test_result,
    {"test_path": "tests"},
)


# Verify updates.
assert "src/model.py" in test_memory.inspected_files
assert "src/model.py" in test_memory.modified_files
assert "4 passed" in test_memory.test_status

test_memory.display()

print("\nAll automatic memory update tests passed.")


AGENT WORKING MEMORY

TASK:
  Debug the sample Python project.

PROJECT STRUCTURE:
  - None

INSPECTED FILES:
  - src/model.py

IDENTIFIED ISSUES:
  - None

MODIFIED FILES:
  - src/model.py

DECISIONS:
  - None

TEST STATUS:
  4 passed in 0.12s

PENDING TASKS:
  - None

All automatic memory update tests passed.



## Updated execution pipeline

For every tool call:

1. Gemini requests a tool.
2. ToolRegistry executes it.
3. The harness updates Working Memory from the original result.
4. The harness truncates the result if necessary.
5. The managed result enters conversation history.
6. Gemini receives the updated conversation.

The WorkingMemory object will be passed into the agent and returned as part of the execution result.

We will also save memory after each completed tool round so that progress is less likely to be lost if the Colab session is interrupted.

This is not yet full session persistence: conversation history and pending execution state will be handled separately later.


In [112]:

from dataclasses import dataclass
from typing import Optional


@dataclass
class AgentRunResult:
    status: str
    history: list
    response: AssistantTurn
    steps: int
    pending_tool_calls: list
    working_memory: Optional[WorkingMemory] = None


In [113]:

def run_agent_v5(
    llm,
    messages,
    registry,
    tool_definitions,
    usage_tracker,
    working_memory,
    initial_response=None,
    initial_response_already_counted=False,
    memory_path=None,
    max_steps=10,
    verbose=True,
):
    history = copy.deepcopy(messages)
    validate_messages(history)

    response = initial_response

    if response is None:
        response = llm.chat(
            messages=history,
            tools=tool_definitions,
        )
        usage_tracker.record(response.usage)

    elif not initial_response_already_counted:
        usage_tracker.record(response.usage)

    steps = 0

    while response.tool_calls:

        if steps >= max_steps:
            status = "max_steps_reached"
            break

        # Preserve the original assistant message,
        # including Gemini thought signatures.
        history.append(get_original_assistant_message(response))

        # Execute tools.
        results = registry.execute_many(response.tool_calls)

        # Update memory from ORIGINAL tool results.
        for result in results:
            try:
                arguments = json.loads(
                    next(
                        call.arguments
                        for call in response.tool_calls
                        if call.id == result.tool_call_id
                    )
                )

                update_working_memory(
                    working_memory,
                    result,
                    arguments,
                )

            except (ValueError, TypeError, StopIteration):
                if verbose:
                    print(
                        f"Could not update memory for tool: {result.name}"
                    )

        # Manage output before adding observations to history.
        managed_results = [
            manage_tool_result(result)
            for result in results
        ]

        for result in managed_results:
            history.append(result.to_message())

        validate_messages(history)

        steps += 1

        # Save working memory after each completed tool round.
        if memory_path is not None:
            working_memory.save(memory_path)

        if verbose:
            print(f"Completed agent step {steps}")

        # Ask Gemini for the next action.
        response = llm.chat(
            messages=history,
            tools=tool_definitions,
        )

        usage_tracker.record(response.usage)

    else:
        status = "completed"
        history.append(get_original_assistant_message(response))

        if memory_path is not None:
            working_memory.save(memory_path)

    return AgentRunResult(
        status=status,
        history=history,
        response=response,
        steps=steps,
        pending_tool_calls=copy.deepcopy(response.tool_calls),
        working_memory=working_memory,
    )


In [114]:

# Synthetic tool
def inspect_sample_file(path):
    return "class SimpleModel:\n    def forward(self, x):\n        return x"


integration_registry = ToolRegistry()
integration_registry.register(
    "read_file",
    inspect_sample_file,
)

integration_definitions = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a project file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string"},
                },
                "required": ["path"],
                "additionalProperties": False,
            },
        },
    }
]

# Gemini requests the read_file tool.
integration_response_1 = AssistantTurn(
    content=None,
    tool_calls=[
        ToolCall(
            id="integration_call_1",
            name="read_file",
            arguments='{"path": "src/model.py"}',
        )
    ],
    finish_reason="tool_calls",
    usage={
        "prompt_tokens": 100,
        "completion_tokens": 20,
        "total_tokens": 120,
    },
)

# Gemini finishes.
integration_response_2 = AssistantTurn(
    content="I inspected src/model.py.",
    tool_calls=[],
    finish_reason="stop",
    usage={
        "prompt_tokens": 150,
        "completion_tokens": 30,
        "total_tokens": 180,
    },
)

integration_llm = ScriptedLLM([integration_response_2])
integration_tracker = UsageTracker()
integration_memory = WorkingMemory(
    task="Inspect the model implementation."
)

integration_result = run_agent_v5(
    llm=integration_llm,
    messages=[
        {
            "role": "system",
            "content": "You are a code-fixing agent.",
        },
        {
            "role": "user",
            "content": "Inspect src/model.py.",
        },
    ],
    registry=integration_registry,
    tool_definitions=integration_definitions,
    usage_tracker=integration_tracker,
    working_memory=integration_memory,
    initial_response=integration_response_1,
    memory_path="/content/agent_workspace/integration_memory.json",
    max_steps=5,
    verbose=True,
)

# Verify memory was automatically updated.
assert "src/model.py" in integration_result.working_memory.inspected_files

# Verify both API responses were tracked.
assert integration_tracker.requests == 2
assert integration_tracker.total_tokens == 300

# Verify conversation protocol.
validate_messages(integration_result.history)

print("\nWORKING MEMORY:")
integration_result.working_memory.display()

print("\nTOKEN USAGE:")
integration_tracker.display()

print("\nAll agent-memory integration tests passed.")


Completed agent step 1

WORKING MEMORY:
AGENT WORKING MEMORY

TASK:
  Inspect the model implementation.

PROJECT STRUCTURE:
  - None

INSPECTED FILES:
  - src/model.py

IDENTIFIED ISSUES:
  - None

MODIFIED FILES:
  - None

DECISIONS:
  - None

TEST STATUS:
  Not run

PENDING TASKS:
  - None

TOKEN USAGE:
TOKEN USAGE SUMMARY
requests                 : 2
prompt_tokens            : 250
completion_tokens        : 50
total_tokens             : 300
latest_prompt_tokens     : 150
largest_prompt_tokens    : 150

All agent-memory integration tests passed.



## Context builder

Input:
- Complete conversation history
- Working Memory
- Number of recent messages to retain

Output:
- A smaller conversation for the next Gemini request

Important design considerations:

1. Keep the system prompt.
2. Keep the original user task.
3. Include the latest Working Memory.
4. Retain recent messages for continuity.
5. Never retain an isolated tool response without its corresponding assistant tool call.

The fifth condition is particularly important for Gemini. Its function-calling protocol requires assistant tool calls and their corresponding tool responses to remain consistent. Gemini may also require provider-specific thought-signature metadata.

For this first implementation, we will build a conservative context selector that retains complete tool-call rounds.


In [118]:

def build_compact_context(
    history,
    working_memory,
    recent_messages=8,
):
    """
    Build compact context while preserving the conversation protocol.
    Working memory is appended to the original system message.
    """

    if not history:
        raise ValueError("Conversation history cannot be empty.")

    system_messages = [
        msg for msg in history
        if msg.get("role") == "system"
    ]

    user_messages = [
        msg for msg in history
        if msg.get("role") == "user"
    ]

    if len(system_messages) != 1:
        raise ProtocolError(
            "Expected exactly one system message."
        )

    if not user_messages:
        raise ProtocolError(
            "Conversation must contain a user message."
        )

    # Copy the original system message.
    compact_system = copy.deepcopy(system_messages[0])

    # Append working memory to the existing system prompt.
    compact_system["content"] += (
        "\n\nCURRENT WORKING MEMORY:\n\n"
        + working_memory.to_prompt()
    )

    # Preserve the original user request.
    original_user_message = copy.deepcopy(user_messages[0])

    # Select recent messages.
    recent = copy.deepcopy(history[-recent_messages:])

    # Remove the original system and user messages from recent selection.
    recent = [
        msg for msg in recent
        if msg.get("role") not in ("system", "user")
    ]

    compact_history = [
        compact_system,
        original_user_message,
        *recent,
    ]

    validate_messages(compact_history)

    return compact_history


print("Corrected context builder defined.")


Corrected context builder defined.


In [119]:

compact_test_history = integration_result.history

compact_context = build_compact_context(
    history=compact_test_history,
    working_memory=integration_memory,
    recent_messages=4,
)

print("Original messages:", len(compact_test_history))
print("Compact messages:", len(compact_context))

print("\nCompact context roles:")
for i, message in enumerate(compact_context):
    print(i, message["role"])

print(
    "\nWorking memory included:",
    "CURRENT WORKING MEMORY" in compact_context[0]["content"],
)

assert compact_context[0]["role"] == "system"
assert compact_context[1]["role"] == "user"
assert "CURRENT WORKING MEMORY" in compact_context[0]["content"]

validate_messages(compact_context)

print("\nBasic context builder tests passed.")


Original messages: 5
Compact messages: 5

Compact context roles:
0 system
1 user
2 assistant
3 tool
4 assistant

Working memory included: True

Basic context builder tests passed.



## Tool-call-aware context selection

We will introduce a helper:

get_complete_recent_messages()

It will:

1. Start from the end of conversation history.
2. Identify complete assistant tool-call messages.
3. Retain all corresponding tool responses.
4. Retain ordinary user and assistant messages.
5. Never include an incomplete tool-call group.

For now, we will use a message-count budget, not a token budget.

Later, we will introduce token-aware selection and automatic compaction.


In [120]:

def get_complete_recent_messages(history, max_messages=8):
    """
    Select recent messages while preserving complete tool-call groups.

    The system message and original user task are handled separately
    by build_compact_context().
    """

    selected = []
    i = len(history) - 1

    while i >= 0:

        message = history[i]
        role = message.get("role")

        # Tool responses must be associated with an earlier
        # assistant tool-call message.
        if role == "tool":

            tool_ids = set()

            while i >= 0 and history[i].get("role") == "tool":
                tool_ids.add(history[i].get("tool_call_id"))
                i -= 1

            if i < 0:
                break

            assistant_message = history[i]

            if assistant_message.get("role") != "assistant":
                break

            calls = assistant_message.get("tool_calls", [])

            call_ids = {
                call.get("id")
                for call in calls
            }

            # Only retain complete tool-call groups.
            if not tool_ids.issubset(call_ids):
                break

            group = history[i + 1:i + 1 + len(tool_ids)]

            selected = (
                [assistant_message]
                + group
                + selected
            )

            i -= 1

        elif role == "assistant" and message.get("tool_calls"):

            # An assistant tool-call message without its results
            # is not a complete group.
            break

        else:
            selected.insert(0, message)
            i -= 1

        if len(selected) >= max_messages:
            break

    return selected[-max_messages:]


print("Tool-call-aware selector defined.")


Tool-call-aware selector defined.


In [121]:

synthetic_history = [
    {"role": "system", "content": "System prompt"},
    {"role": "user", "content": "Debug this project."},

    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "id": "call_1",
                "type": "function",
                "function": {
                    "name": "read_file",
                    "arguments": '{"path": "src/model.py"}',
                },
            }
        ],
    },
    {
        "role": "tool",
        "tool_call_id": "call_1",
        "content": "Source code...",
    },

    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "id": "call_2",
                "type": "function",
                "function": {
                    "name": "run_tests",
                    "arguments": '{"test_path": "tests"}',
                },
            }
        ],
    },
    {
        "role": "tool",
        "tool_call_id": "call_2",
        "content": "4 passed",
    },

    {
        "role": "assistant",
        "content": "The project is fixed.",
    },
]


recent = get_complete_recent_messages(
    synthetic_history,
    max_messages=3,
)

print("Selected messages:")

for message in recent:
    print(message["role"], message.get("tool_call_id", ""))


# Verify the latest tool interaction is complete.
assert recent[0]["role"] == "assistant"
assert recent[0]["tool_calls"][0]["id"] == "call_2"
assert recent[1]["role"] == "tool"
assert recent[1]["tool_call_id"] == "call_2"
assert recent[2]["role"] == "assistant"

print("\nTool-call grouping tests passed.")


Selected messages:
assistant 
tool call_2
assistant 

Tool-call grouping tests passed.


Step 28.4 — Token-aware context budgeting
Now we'll improve our context builder so it selects recent interactions based on an estimated token budget rather than an arbitrary number of messages.

This is important because 5 messages containing large source files can consume more tokens than 30 messages containing short tool confirmations.
Our approach
We'll divide the conversation into complete interaction groups:
- An ordinary user or assistant message is one group.
- An assistant tool-call message and all its corresponding tool responses form one group.
Then we'll select the most recent complete groups that fit within the available budget.

One important distinction: We don't currently have a tokenizer for Gemini's exact tokenization. We'll initially estimate tokens using approximately 4 characters per token. This is a heuristic, not an exact count. We'll compare it with actual Gemini usage later.

In [122]:

import json


def estimate_tokens(text):
    """
    Approximate token count using 4 characters per token.

    This is a rough estimate, not Gemini's exact tokenizer.
    """
    return (len(str(text)) + 3) // 4


def estimate_message_tokens(message):
    serialized = json.dumps(
        message,
        ensure_ascii=False,
        default=str,
    )

    # Add a small allowance for message formatting overhead.
    return estimate_tokens(serialized) + 4


def get_interaction_groups(history, start_index):
    """
    Divide conversation history into complete interaction groups.

    An assistant tool call and all its corresponding tool
    responses are treated as one indivisible group.
    """

    groups = []
    i = start_index

    while i < len(history):

        message = history[i]
        role = message.get("role")

        if role == "tool":
            raise ProtocolError(
                f"Unexpected standalone tool message at index {i}."
            )

        if role == "assistant" and message.get("tool_calls"):

            expected_ids = {
                call["id"]
                for call in message["tool_calls"]
            }

            group = [message]
            observed_ids = set()

            i += 1

            while i < len(history) and history[i].get("role") == "tool":
                tool_id = history[i].get("tool_call_id")

                if tool_id in observed_ids:
                    raise ProtocolError(
                        f"Duplicate tool response: {tool_id}"
                    )

                observed_ids.add(tool_id)
                group.append(history[i])
                i += 1

            if observed_ids != expected_ids:
                raise ProtocolError(
                    "Incomplete assistant tool-call group. "
                    f"Expected {expected_ids}, received {observed_ids}."
                )

            groups.append(group)

        else:
            groups.append([message])
            i += 1

    return groups


print("Token estimation and interaction grouping defined.")


Token estimation and interaction grouping defined.


In [123]:

def build_compact_context_v2(
    history,
    working_memory,
    max_input_tokens=6000,
):
    """
    Build compact context under an approximate input-token budget.

    Preserves:
    - Original system message
    - Original user request
    - Current working memory
    - Most recent complete interaction groups that fit
    """

    if not history:
        raise ValueError("Conversation history cannot be empty.")

    validate_messages(history)

    system_messages = [
        msg for msg in history
        if msg.get("role") == "system"
    ]

    user_messages = [
        msg for msg in history
        if msg.get("role") == "user"
    ]

    if len(system_messages) != 1:
        raise ProtocolError("Expected exactly one system message.")

    if not user_messages:
        raise ProtocolError("No user request found.")

    original_system = copy.deepcopy(system_messages[0])
    original_user = copy.deepcopy(user_messages[0])

    # Add current memory to the existing system message.
    original_system["content"] += (
        "\n\nCURRENT WORKING MEMORY:\n\n"
        + working_memory.to_prompt()
    )

    fixed_context = [
        original_system,
        original_user,
    ]

    fixed_tokens = sum(
        estimate_message_tokens(message)
        for message in fixed_context
    )

    if fixed_tokens > max_input_tokens:
        raise ValueError(
            f"System prompt, user request and memory require "
            f"approximately {fixed_tokens} tokens, exceeding "
            f"the budget of {max_input_tokens}."
        )

    remaining_budget = max_input_tokens - fixed_tokens

    # Find the original user request's position.
    original_user_index = history.index(user_messages[0])

    groups = get_interaction_groups(
        history,
        start_index=original_user_index + 1,
    )

    selected_groups = []
    used_tokens = 0

    # Select complete groups starting from the newest.
    for group in reversed(groups):

        group_tokens = sum(
            estimate_message_tokens(message)
            for message in group
        )

        if used_tokens + group_tokens > remaining_budget:
            # Do not include a partial group.
            break

        selected_groups.append(group)
        used_tokens += group_tokens

    selected_groups.reverse()

    recent_messages = [
        message
        for group in selected_groups
        for message in group
    ]

    compact_history = fixed_context + recent_messages

    validate_messages(compact_history)

    estimated_total = fixed_tokens + used_tokens

    return compact_history, {
        "estimated_tokens": estimated_total,
        "fixed_context_tokens": fixed_tokens,
        "recent_context_tokens": used_tokens,
        "budget": max_input_tokens,
        "original_messages": len(history),
        "compact_messages": len(compact_history),
        "selected_groups": len(selected_groups),
    }


print("Token-aware context builder defined.")


Token-aware context builder defined.


In [124]:

compact_context, context_stats = build_compact_context_v2(
    history=synthetic_history,
    working_memory=integration_memory,
    max_input_tokens=2000,
)

print("=" * 55)
print("CONTEXT BUDGET REPORT")
print("=" * 55)

for key, value in context_stats.items():
    print(f"{key:25s}: {value:,}")

print("\nSelected messages:")

for message in compact_context:
    print(
        message["role"],
        message.get("tool_call_id", ""),
    )

# Verify budget.
assert context_stats["estimated_tokens"] <= context_stats["budget"]

# Verify original task and memory are preserved.
assert compact_context[1]["content"] == synthetic_history[1]["content"]
assert "CURRENT WORKING MEMORY" in compact_context[0]["content"]

# Verify conversation protocol.
validate_messages(compact_context)

# Verify that the latest tool interaction is complete.
selected_call_ids = {
    call["id"]
    for message in compact_context
    if message.get("role") == "assistant"
    for call in message.get("tool_calls", [])
}

selected_result_ids = {
    message["tool_call_id"]
    for message in compact_context
    if message.get("role") == "tool"
}

assert selected_call_ids == selected_result_ids

print("\nAll token-aware context tests passed.")


CONTEXT BUDGET REPORT
estimated_tokens         : 253
fixed_context_tokens     : 97
recent_context_tokens    : 156
budget                   : 2,000
original_messages        : 7
compact_messages         : 7
selected_groups          : 3

Selected messages:
system 
user 
assistant 
tool call_1
assistant 
tool call_2
assistant 

All token-aware context tests passed.



## Automatic context management

We'll introduce two parameters:

CONTEXT_LIMIT = 6000

COMPACTION_TARGET = 4500

Why two values?

- CONTEXT_LIMIT: When we trigger compaction.
- COMPACTION_TARGET: How much context we aim to retain afterward.

This creates a buffer and prevents compaction from happening on every request.

For example:

Current estimated context: 6,300 tokens

Threshold: 6,000 tokens

Action: Compact

Target: 4,500 tokens

The original conversation remains available separately; only the active history is shortened.

One limitation: this is an approximate input-token budget. Gemini's actual prompt usage can differ because of tokenization, API formatting, and provider-specific metadata. We will measure that difference later.


In [125]:

CONTEXT_LIMIT = 6000
COMPACTION_TARGET = 4500


def manage_context(
    history,
    working_memory,
    context_limit=CONTEXT_LIMIT,
    compaction_target=COMPACTION_TARGET,
):
    """
    Compact conversation only when the estimated context
    exceeds the configured threshold.
    """

    # Estimate the current conversation size.
    current_tokens = sum(
        estimate_message_tokens(message)
        for message in history
    )

    if current_tokens <= context_limit:
        return history, {
            "compacted": False,
            "before_tokens": current_tokens,
            "after_tokens": current_tokens,
        }

    # Build a compact context.
    compact_history, stats = build_compact_context_v2(
        history=history,
        working_memory=working_memory,
        max_input_tokens=compaction_target,
    )

    return compact_history, {
        "compacted": True,
        "before_tokens": current_tokens,
        "after_tokens": stats["estimated_tokens"],
        "messages_removed": (
            len(history) - len(compact_history)
        ),
    }


print("Automatic context manager defined.")


Automatic context manager defined.


In [127]:

large_history = [
    {"role": "system", "content": "You are a code-fixing agent."},
    {"role": "user", "content": "Fix the project bugs."},
]

# Add 20 complete interaction groups.
for i in range(20):

    call_id = f"large_call_{i}"

    large_history.append({
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "id": call_id,
                "type": "function",
                "function": {
                    "name": "read_file",
                    "arguments": '{"path": "src/model.py"}',
                },
            }
        ],
    })

    large_history.append({
        "role": "tool",
        "tool_call_id": call_id,
        "content": "X" * 2000,
    })

    # Add a user message to create the next interaction boundary.
    large_history.append({
        "role": "user",
        "content": f"Continue debugging. Inspection round {i + 1}.",
    })


validate_messages(large_history)

print("Original message count:", len(large_history))

managed_history, compaction_stats = manage_context(
    history=large_history,
    working_memory=integration_memory,
    context_limit=6000,
    compaction_target=4500,
)

print("\nCOMPACTION REPORT")

for key, value in compaction_stats.items():
    print(f"{key:25s}: {value:,}" if isinstance(value, int) else f"{key:25s}: {value}")

validate_messages(managed_history)

assert compaction_stats["compacted"] is True
assert len(managed_history) < len(large_history)
assert compaction_stats["after_tokens"] <= 4500

print("\nAutomatic compaction tests passed.")


Original message count: 62

COMPACTION REPORT
compacted                : 1
before_tokens            : 11,857
after_tokens             : 4,260
messages_removed         : 38

Automatic compaction tests passed.


In [128]:

@dataclass
class AgentRunResult:
    status: str
    history: list
    response: AssistantTurn
    steps: int
    pending_tool_calls: list
    working_memory: Optional[WorkingMemory] = None
    full_history: Optional[list] = None
    compaction_events: list = field(default_factory=list)


In [129]:

def run_agent_v6(
    llm,
    messages,
    registry,
    tool_definitions,
    usage_tracker,
    working_memory,
    initial_response=None,
    initial_response_already_counted=False,
    full_history=None,
    memory_path=None,
    context_limit=CONTEXT_LIMIT,
    compaction_target=COMPACTION_TARGET,
    max_steps=10,
    verbose=True,
):
    # Active conversation and complete audit history.
    history = copy.deepcopy(messages)

    if full_history is None:
        full_history = copy.deepcopy(messages)
    else:
        full_history = copy.deepcopy(full_history)

    validate_messages(history)
    validate_messages(full_history)

    response = initial_response

    compaction_events = []
    steps = 0

    # Obtain the first response if one was not supplied.
    if response is None:

        history, stats = manage_context(
            history,
            working_memory,
            context_limit,
            compaction_target,
        )

        if stats["compacted"]:
            compaction_events.append(stats)

        response = llm.chat(
            messages=history,
            tools=tool_definitions,
        )

        usage_tracker.record(response.usage)

    elif not initial_response_already_counted:
        usage_tracker.record(response.usage)

    while response.tool_calls:

        if steps >= max_steps:
            status = "max_steps_reached"
            break

        # Preserve original assistant message and Gemini metadata.
        assistant_message = get_original_assistant_message(response)

        history.append(copy.deepcopy(assistant_message))
        full_history.append(copy.deepcopy(assistant_message))

        # Execute requested tools.
        results = registry.execute_many(response.tool_calls)

        for result in results:

            # Retrieve the original tool arguments.
            arguments = next(
                (
                    json.loads(call.arguments)
                    for call in response.tool_calls
                    if call.id == result.tool_call_id
                ),
                {},
            )

            # Update memory using the ORIGINAL result.
            update_working_memory(
                working_memory,
                result,
                arguments,
            )

            # Preserve original, untruncated output in full history.
            original_tool_message = result.to_message()

            full_history.append(
                copy.deepcopy(original_tool_message)
            )

            # Add managed output to active history.
            managed_result = manage_tool_result(result)

            history.append(managed_result.to_message())

        validate_messages(history)
        validate_messages(full_history)

        steps += 1

        # Save memory after each completed tool round.
        if memory_path is not None:
            working_memory.save(memory_path)

        # Compact active history before the next Gemini request.
        history, stats = manage_context(
            history,
            working_memory,
            context_limit,
            compaction_target,
        )

        if stats["compacted"]:
            compaction_events.append(stats)

            if verbose:
                print(
                    "Context compacted:",
                    stats["before_tokens"],
                    "->",
                    stats["after_tokens"],
                    "estimated tokens",
                )

        if verbose:
            print(f"Completed agent step {steps}")

        # Continue the ReAct loop.
        response = llm.chat(
            messages=history,
            tools=tool_definitions,
        )

        usage_tracker.record(response.usage)

    else:
        status = "completed"

        final_message = get_original_assistant_message(response)

        history.append(copy.deepcopy(final_message))
        full_history.append(copy.deepcopy(final_message))

        if memory_path is not None:
            working_memory.save(memory_path)

    return AgentRunResult(
        status=status,
        history=history,
        response=response,
        steps=steps,
        pending_tool_calls=copy.deepcopy(response.tool_calls),
        working_memory=working_memory,
        full_history=full_history,
        compaction_events=compaction_events,
    )


In [130]:

# Use the large conversation generated in Step 28.5.

integration_tracker_v6 = UsageTracker()

integration_memory_v6 = WorkingMemory(
    task="Inspect and debug the sample Python project."
)

integration_llm_v6 = ScriptedLLM([
    integration_response_2
])

integration_result_v6 = run_agent_v6(
    llm=integration_llm_v6,
    messages=large_history,
    registry=integration_registry,
    tool_definitions=integration_definitions,
    usage_tracker=integration_tracker_v6,
    working_memory=integration_memory_v6,
    initial_response=integration_response_1,
    full_history=large_history,
    memory_path="/content/agent_workspace/working_memory_v6.json",
    context_limit=6000,
    compaction_target=4500,
    max_steps=5,
    verbose=True,
)


# Compare complete and active histories.
print("\n" + "=" * 55)
print("FINAL AGENT REPORT")
print("=" * 55)

print("Status:", integration_result_v6.status)
print("Execution steps:", integration_result_v6.steps)

print("\nFull history messages:",
      len(integration_result_v6.full_history))

print("Active history messages:",
      len(integration_result_v6.history))

print("Compaction events:",
      len(integration_result_v6.compaction_events))


# Verify memory.
print("\nWorking memory:")
integration_result_v6.working_memory.display()


# Verify token tracking.
print("\nToken usage:")
integration_tracker_v6.display()


# Verify protocol.
validate_messages(integration_result_v6.history)
validate_messages(integration_result_v6.full_history)

assert integration_result_v6.status == "completed"
assert len(integration_result_v6.full_history) > len(integration_result_v6.history)
assert len(integration_result_v6.compaction_events) >= 1

assert (
    "src/model.py"
    in integration_result_v6.working_memory.inspected_files
)

print("\nAll integrated context-management tests passed.")


Context compacted: 11944 -> 4350 estimated tokens
Completed agent step 1

FINAL AGENT REPORT
Status: completed
Execution steps: 1

Full history messages: 65
Active history messages: 27
Compaction events: 1

Working memory:
AGENT WORKING MEMORY

TASK:
  Inspect and debug the sample Python project.

PROJECT STRUCTURE:
  - None

INSPECTED FILES:
  - src/model.py

IDENTIFIED ISSUES:
  - None

MODIFIED FILES:
  - None

DECISIONS:
  - None

TEST STATUS:
  Not run

PENDING TASKS:
  - None

Token usage:
TOKEN USAGE SUMMARY
requests                 : 2
prompt_tokens            : 250
completion_tokens        : 50
total_tokens             : 300
latest_prompt_tokens     : 150
largest_prompt_tokens    : 150

All integrated context-management tests passed.



### AgentSession

The session object will hold:

- session_id
- full_history
- working_memory
- usage_tracker
- status
- execution_steps
- pending_tool_calls

It will provide:

- to_dict(): Convert the complete session into JSON-compatible data.
- save(): Persist the session to disk.
- load(): Restore the session from disk.

The active compacted history will also be saved because it is the exact context used by the agent for subsequent reasoning.

We will keep the execution response separately in memory when a run is paused. A later step will serialize that response too, so a pending tool call can be resumed after a restart.


In [131]:

def usage_tracker_to_dict(self):
    return self.summary()


def usage_tracker_from_dict(data):
    return UsageTracker(**data)


UsageTracker.to_dict = usage_tracker_to_dict
UsageTracker.from_dict = staticmethod(usage_tracker_from_dict)

print("UsageTracker serialization added.")


UsageTracker serialization added.


In [132]:

import uuid
from datetime import datetime, timezone


@dataclass
class AgentSession:

    session_id: str = field(
        default_factory=lambda: str(uuid.uuid4())
    )

    full_history: list = field(default_factory=list)

    active_history: list = field(default_factory=list)

    working_memory: Optional[WorkingMemory] = None

    usage_tracker: UsageTracker = field(
        default_factory=UsageTracker
    )

    status: str = "created"

    execution_steps: int = 0

    pending_tool_calls: list = field(default_factory=list)

    created_at: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )

    updated_at: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )


    def to_dict(self):

        return {
            "session_id": self.session_id,
            "full_history": self.full_history,
            "active_history": self.active_history,
            "working_memory": (
                self.working_memory.to_dict()
                if self.working_memory is not None
                else None
            ),
            "usage_tracker": self.usage_tracker.to_dict(),
            "status": self.status,
            "execution_steps": self.execution_steps,
            "pending_tool_calls": self.pending_tool_calls,
            "created_at": self.created_at,
            "updated_at": self.updated_at,
        }


    def save(self, path):

        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        self.updated_at = datetime.now(timezone.utc).isoformat()

        # Write to a temporary file first to reduce
        # the risk of leaving a partially written session.
        temporary_path = path.with_suffix(path.suffix + ".tmp")

        with temporary_path.open("w", encoding="utf-8") as f:
            json.dump(
                self.to_dict(),
                f,
                indent=2,
                ensure_ascii=False,
            )

        temporary_path.replace(path)


    @classmethod
    def load(cls, path):

        path = Path(path)

        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        memory_data = data.get("working_memory")

        memory = (
            WorkingMemory(**memory_data)
            if memory_data is not None
            else None
        )

        tracker = UsageTracker.from_dict(
            data.get("usage_tracker", {})
        )

        return cls(
            session_id=data["session_id"],
            full_history=data["full_history"],
            active_history=data["active_history"],
            working_memory=memory,
            usage_tracker=tracker,
            status=data["status"],
            execution_steps=data["execution_steps"],
            pending_tool_calls=data.get("pending_tool_calls", []),
            created_at=data["created_at"],
            updated_at=data["updated_at"],
        )


print("AgentSession class defined.")


AgentSession class defined.


In [133]:

session = AgentSession(
    full_history=copy.deepcopy(
        integration_result_v6.full_history
    ),

    active_history=copy.deepcopy(
        integration_result_v6.history
    ),

    working_memory=integration_result_v6.working_memory,

    usage_tracker=integration_tracker_v6,

    status=integration_result_v6.status,

    execution_steps=integration_result_v6.steps,

    pending_tool_calls=[
        {
            "id": call.id,
            "name": call.name,
            "arguments": call.arguments,
        }
        for call in integration_result_v6.pending_tool_calls
    ],
)

SESSION_PATH = "/content/agent_workspace/session.json"

# Save.
session.save(SESSION_PATH)

print("Session saved:", SESSION_PATH)
print("Session ID:", session.session_id)


# Restore.
restored_session = AgentSession.load(SESSION_PATH)

print("\nRestored session:")
print("Session ID:", restored_session.session_id)
print("Status:", restored_session.status)
print("Execution steps:", restored_session.execution_steps)

print(
    "Full history messages:",
    len(restored_session.full_history),
)

print(
    "Active history messages:",
    len(restored_session.active_history),
)


# Verify preservation.
assert restored_session.session_id == session.session_id

assert (
    restored_session.full_history
    == session.full_history
)

assert (
    restored_session.active_history
    == session.active_history
)

assert (
    restored_session.working_memory.to_dict()
    == session.working_memory.to_dict()
)

assert (
    restored_session.usage_tracker.to_dict()
    == session.usage_tracker.to_dict()
)

assert restored_session.status == session.status

print("\nAll session persistence tests passed.")


Session saved: /content/agent_workspace/session.json
Session ID: 6a774a57-04e1-48cb-805f-e52cd5a1d091

Restored session:
Session ID: 6a774a57-04e1-48cb-805f-e52cd5a1d091
Status: completed
Execution steps: 1
Full history messages: 65
Active history messages: 27

All session persistence tests passed.



### Bug scenario

Current behavior:
- Labelled dataset: `[(sample, label), ...]`
- Unlabelled dataset: `[sample, ...]`

The trainer assumes both have labels and unpacks every item into two variables.

Expected behavior:
- The trainer should support both dataset types.
- It should return predictions for the original samples.
- Existing labelled-dataset functionality must continue working.

We will preserve a snapshot of the project before either agent modifies it. This allows us to start both experiments from exactly the same code.


In [134]:

import shutil
from pathlib import Path

trainer_test_path = WORKSPACE / "tests" / "test_trainer.py"

new_test = '''

def test_trainer_supports_unlabeled_dataset():
    model = SimpleModel(2, 4)

    dataset = CustomDataset(
        samples=[[1, 2], [3, 4]],
    )

    predictions = train_model(
        model,
        dataset,
        epochs=1,
    )

    assert predictions == [[1, 2], [3, 4]]
'''

# Add the test only once.
existing_content = trainer_test_path.read_text(encoding="utf-8")

if "test_trainer_supports_unlabeled_dataset" not in existing_content:
    with trainer_test_path.open("a", encoding="utf-8") as f:
        f.write(new_test)

# Preserve the initial buggy project.
SNAPSHOT_PATH = Path("/content/agent_workspace_initial")

if SNAPSHOT_PATH.exists():
    shutil.rmtree(SNAPSHOT_PATH)

shutil.copytree(WORKSPACE, SNAPSHOT_PATH)

print("Project snapshot saved:", SNAPSHOT_PATH)
print("Regression test added:", trainer_test_path)


# Run tests before asking the agent to fix anything.
baseline_test_output = run_tests(test_path="tests")

print("\nINITIAL TEST RESULTS")
print(baseline_test_output)


Project snapshot saved: /content/agent_workspace_initial
Regression test added: /content/agent_workspace/tests/test_trainer.py

INITIAL TEST RESULTS
Pytest exit code: 1

STDOUT:
....F                                                                    [100%]
=================================== FAILURES ===================================
___________________ test_trainer_supports_unlabeled_dataset ____________________

    def test_trainer_supports_unlabeled_dataset():
        model = SimpleModel(2, 4)
    
        dataset = CustomDataset(
            samples=[[1, 2], [3, 4]],
        )
    
        predictions = train_model(
            model,
            dataset,
            epochs=1,
        )
    
>       assert predictions == [[1, 2], [3, 4]]
E       assert [1, 3] == [[1, 2], [3, 4]]
E         
E         At index 0 diff: 1 != [1, 2]
E         Use -v to get more diff

tests/test_trainer.py:34: AssertionError
=========================== short test summary info ========================


### Baseline configuration

Enabled:
- Gemini reasoning
- ReAct tool execution
- File exploration and editing
- Pytest execution
- Tool-output limits

Disabled:
- Working Memory
- Context compaction

Task: Fix the unlabelled dataset compatibility bug without breaking existing tests.

We'll record the actual Gemini token usage and final test results.


In [135]:

# Restore the exact initial project.
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)

shutil.copytree(SNAPSHOT_PATH, WORKSPACE)

# Create a fresh baseline tracker.
baseline_tracker = UsageTracker()

baseline_messages = [
    {
        "role": "system",
        "content": """
You are an autonomous Python code-fixing agent.

Use the available tools to inspect, debug, and modify the project.
Run the tests to identify failures and verify your changes.

Do not assume the cause of a failure without inspecting the code.
Preserve existing functionality.
Do not modify tests merely to make them pass.

Continue until the task is fixed and verified.
        """.strip(),
    },
    {
        "role": "user",
        "content": """
Fix the bug in the training pipeline.

The CustomDataset supports both labelled and unlabelled datasets.
However, train_model currently produces incorrect predictions
for unlabelled datasets.

Investigate the relevant source files, fix the implementation,
and run the complete test suite.

Do not change the expected behavior of labelled datasets.
        """.strip(),
    },
]

print("Starting baseline debugging run...\n")

baseline_result = run_agent_v4(
    llm=gemini,
    messages=baseline_messages,
    registry=project_registry,
    tool_definitions=project_tool_definitions,
    usage_tracker=baseline_tracker,
    max_steps=12,
    verbose=True,
)

print("\n" + "=" * 55)
print("BASELINE RESULTS")
print("=" * 55)

print("Status:", baseline_result.status)
print("Agent steps:", baseline_result.steps)
print("Pending tool calls:", len(baseline_result.pending_tool_calls))

baseline_tracker.display()

# Independently verify the final project.
print("\nFINAL TEST RESULTS")
print(run_tests(test_path="tests"))


Starting baseline debugging run...

Completed agent step 1
Completed agent step 2
Completed agent step 3
Completed agent step 4
Completed agent step 5
Completed agent step 6
Completed agent step 7
Completed agent step 8
Completed agent step 9
Completed agent step 10
Completed agent step 11
Completed agent step 12

BASELINE RESULTS
Status: completed
Agent steps: 12
Pending tool calls: 0
TOKEN USAGE SUMMARY
requests                 : 13
prompt_tokens            : 20,039
completion_tokens        : 409
total_tokens             : 20,448
latest_prompt_tokens     : 2,208
largest_prompt_tokens    : 2,208

FINAL TEST RESULTS
Pytest exit code: 0

STDOUT:
.....                                                                    [100%]
5 passed in 0.01s


STDERR:




## Experiment B: Debugging with Working Memory

Restore the original buggy project so both experiments start from the same code.

This run enables:
- Structured Working Memory to retain findings and progress.
- Automatic context compaction to remove older conversation details while preserving the original task and relevant tool interactions.
- Token and compaction tracking.

The debugging task and tool access remain the same as the baseline run.


In [138]:

# Restore the original buggy project for a fair comparison.
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)

shutil.copytree(SNAPSHOT_PATH, WORKSPACE)

memory_tracker = UsageTracker()

memory = WorkingMemory(
    task="Fix the bug in the training pipeline so that train_model supports both labelled and unlabelled datasets."
)

memory_result = run_agent_v6(
    llm=gemini,
    messages=baseline_messages,
    registry=project_registry,
    tool_definitions=project_tool_definitions,
    usage_tracker=memory_tracker,
    working_memory=memory,
    memory_path="/content/agent_workspace/memory_trial.json",
    context_limit=1500,
    compaction_target=1500,
    max_steps=12,
    verbose=True,
)

print("\n" + "=" * 55)
print("MEMORY + COMPACTION RESULTS")
print("=" * 55)
print("Status:", memory_result.status)
print("Agent steps:", memory_result.steps)
print("Pending tool calls:", len(memory_result.pending_tool_calls))

print("\nTOKEN USAGE")
memory_tracker.display()

print("\nWORKING MEMORY")
memory.display()

print("\nCOMPACTION EVENTS")
print(memory_result.compaction_events)

print("\nFINAL TEST RESULTS")
print(run_tests(test_path="tests"))


Completed agent step 1
Completed agent step 2
Completed agent step 3
Completed agent step 4
Completed agent step 5
Context compacted: 1639 -> 1426 estimated tokens
Completed agent step 6
Context compacted: 1666 -> 1311 estimated tokens
Completed agent step 7
Completed agent step 8
Context compacted: 1772 -> 1357 estimated tokens
Completed agent step 9
Context compacted: 1577 -> 1399 estimated tokens
Completed agent step 10
Context compacted: 1606 -> 1314 estimated tokens
Completed agent step 11
Context compacted: 1519 -> 1450 estimated tokens
Completed agent step 12

MEMORY + COMPACTION RESULTS
Status: max_steps_reached
Agent steps: 12
Pending tool calls: 1

TOKEN USAGE
TOKEN USAGE SUMMARY
requests                 : 13
prompt_tokens            : 18,336
completion_tokens        : 334
total_tokens             : 18,670
latest_prompt_tokens     : 1,940
largest_prompt_tokens    : 1,940

WORKING MEMORY
AGENT WORKING MEMORY

TASK:
  Fix the bug in the training pipeline so that train_model sup


## Resume Experiment B

Continue from the existing memory-enabled session rather than restarting.

Preserve:
- The current active conversation history.
- The Working Memory state.
- The existing token tracker.
- The pending assistant response and tool call.

Allow a few additional steps for the agent to finish.


In [139]:

import inspect

print("run_agent_v6 signature:")
print(inspect.signature(run_agent_v6))

print("\nCurrent status:", memory_result.status)
print("Pending tool calls:", len(memory_result.pending_tool_calls))
print("Working memory modified files:", memory.modified_files)


run_agent_v6 signature:
(llm, messages, registry, tool_definitions, usage_tracker, working_memory, initial_response=None, initial_response_already_counted=False, full_history=None, memory_path=None, context_limit=6000, compaction_target=4500, max_steps=10, verbose=True)

Current status: max_steps_reached
Pending tool calls: 1
Working memory modified files: ['src/trainer.py']



## Experiment B: Working Memory Without Context Compaction

Run the third independent experiment using run_agent_v5.

Restore the original buggy project before execution. Use the same debugging task and tool access as the other two experiments.

This experiment enables Working Memory but does not use automatic context compaction.


In [140]:

import inspect

print("run_agent_v5 signature:")
print(inspect.signature(run_agent_v5))


run_agent_v5 signature:
(llm, messages, registry, tool_definitions, usage_tracker, working_memory, initial_response=None, initial_response_already_counted=False, memory_path=None, max_steps=10, verbose=True)



## Experiment B: Working Memory Without Context Compaction

Restore the original buggy project and run an independent debugging trial.

Configuration:
- Working Memory: enabled
- Automatic context compaction: disabled
- Tool-output management: enabled
- Maximum agent steps: 12

Use the same task, model, tools, and initial project snapshot as the other experiments.


In [141]:

# Restore the original buggy project.
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)

shutil.copytree(SNAPSHOT_PATH, WORKSPACE)

# Fresh tracker and fresh memory.
memory_only_tracker = UsageTracker()

memory_only = WorkingMemory(
    task="Fix the bug in the training pipeline so that train_model supports both labelled and unlabelled datasets."
)

memory_only_result = run_agent_v5(
    llm=gemini,
    messages=baseline_messages,
    registry=project_registry,
    tool_definitions=project_tool_definitions,
    usage_tracker=memory_only_tracker,
    working_memory=memory_only,
    memory_path="/content/agent_workspace/memory_only_trial.json",
    max_steps=12,
    verbose=True,
)

print("\n" + "=" * 55)
print("WORKING MEMORY ONLY RESULTS")
print("=" * 55)
print("Status:", memory_only_result.status)
print("Agent steps:", memory_only_result.steps)
print("Pending tool calls:", len(memory_only_result.pending_tool_calls))

print("\nTOKEN USAGE")
memory_only_tracker.display()

print("\nWORKING MEMORY")
memory_only.display()

print("\nFINAL TEST RESULTS")
print(run_tests(test_path="tests"))


Completed agent step 1
Completed agent step 2
Completed agent step 3
Completed agent step 4
Completed agent step 5
Completed agent step 6
Completed agent step 7
Completed agent step 8
Completed agent step 9
Completed agent step 10
Completed agent step 11

WORKING MEMORY ONLY RESULTS
Status: completed
Agent steps: 11
Pending tool calls: 0

TOKEN USAGE
TOKEN USAGE SUMMARY
requests                 : 12
prompt_tokens            : 18,762
completion_tokens        : 334
total_tokens             : 19,096
latest_prompt_tokens     : 2,600
largest_prompt_tokens    : 2,600

WORKING MEMORY
AGENT WORKING MEMORY

TASK:
  Fix the bug in the training pipeline so that train_model supports both labelled and unlabelled datasets.

PROJECT STRUCTURE:
  - None

INSPECTED FILES:
  - src/trainer.py
  - src/dataset.py
  - tests/test_trainer.py
  - src/model.py
  - tests/test_dataset.py

IDENTIFIED ISSUES:
  - None

MODIFIED FILES:
  - src/trainer.py

DECISIONS:
  - None

TEST STATUS:
  Pytest exit code: 0

STDO


# Final Results: Code-Fixing Agent Comparison

## 1. Experimental configurations

We evaluated three configurations of our Gemini-based code-fixing agent on the same Python project and the same intentionally introduced bug.

1. **Baseline Agent:** No Working Memory and no context compaction.
2. **Memory Agent:** Working Memory enabled, context compaction disabled.
3. **Memory + Compaction Agent:** Working Memory and automatic context compaction enabled.

## 2. Results

| Metric | Baseline | Memory Only | Memory + Compaction |
|---|---:|---:|---:|
| Agent steps | 12 | 11 | 12 |
| Gemini requests | 13 | 12 | 13 |
| Prompt tokens | 20,039 | 18,762 | 18,336 |
| Completion tokens | 409 | 334 | 334 |
| **Total tokens** | **20,448** | **19,096** | **18,670** |
| Tests passed | 5/5 | 5/5 | 5/5 |
| Context compactions | 0 | 0 | 6 |
| Final status | Completed | Completed | Step limit reached |

## 3. Key observations

- All three configurations successfully fixed the introduced training-pipeline bug.
- All three passed the complete test suite: **5 out of 5 tests**.
- Working Memory alone reduced token consumption by approximately **6.6%** compared with the baseline.
- Working Memory with compaction reduced token consumption by approximately **8.7%** compared with the baseline.
- The memory-enabled agent with compaction successfully compacted its context six times, reducing older conversation history while retaining relevant information.
- The memory + compaction run reached its step limit with one pending tool call, although the code modification was already successful and all tests passed.

## 4. Conclusion

We successfully implemented and tested a tool-using code-fixing agent with:

1. Gemini API integration and function calling.
2. A controlled workspace with file exploration and modification tools.
3. Automated test execution.
4. Tool argument validation and error handling.
5. Token usage tracking.
6. Structured Working Memory.
7. Conversation context compaction.
8. Session persistence.

The experiments demonstrate that Working Memory and context compaction can reduce token consumption while preserving successful debugging performance on this test project.

**Overall, the complete agent harness is operational and has passed its first end-to-end debugging experiment.**

*Note: These are results from a small sanity experiment, not a statistically controlled benchmark. Token usage and agent steps can vary between model runs.*
